# NFL world model on Colab: next-play transformer as a fourth-down simulator

**What this is.** A causal transformer is trained on real NFL play-by-play (nflverse, via
`nflreadpy`): training seasons 2010 to 2021, validation 2022 to 2023, test 2024 to 2025. It
reads a game's records so far and predicts the next play's transition. It is then used as a
simulator: from any game state it plays the rest of the game many times.

**What it answers.**
1. Does the model predict the next play better than a frequency baseline (per-field NLL)?
2. Do simulated games look like real games (points, plays, ties, home win share)?
3. Is win probability by simulation calibrated, compared with nflverse's own `home_wp` and
   `vegas_home_wp` on the same states (log loss, Brier, game-bootstrap intervals)?
4. On test-season fourth downs, what does the simulator recommend (go, field goal, punt), and
   how often does that match the real coach? Plus one constructed state: up 4, 4:00 left,
   fourth and 2 at the offense's own 40.

**How long it takes.** About 55 minutes on a T4, L4 or A100 GPU. The run sizes itself to the
GPU it finds and to a fixed time budget; each stage checks the clock and scales down or skips
(with the reason recorded), so it always ends with a results file.

**Choosing a GPU.** Runtime > Change runtime type > Hardware accelerator: GPU (T4 is fine;
L4 or A100 run larger samples). Then Runtime > Run all.

**If you are disconnected.** Reconnect and Run all again. With Google Drive mounted, every
finished stage and the training checkpoint are kept in
`MyDrive/fourth_and_rl_world_model`; finished stages are skipped and training resumes from
its checkpoint. The test seasons are scored only once (preregistered in `prereg.json`).

**What to bring back.** The last cell downloads `world_model_results.zip`. Unzip it into the
repo under `results/world_model/` on the branch `claude/world-model-colab`.


### Stopping and resuming

- Press the stop button at any time. The current stage saves what it has (training keeps a
  checkpoint after every epoch; evaluation and planning save each finished batch of states),
  `results.json` is written with every stage's status, and the cell prints
  "Stopped safely at <stage>. Run all again to continue from here."
- Run all again later, even after a disconnect or with an updated notebook: finished stages
  are skipped and training continues from its checkpoint. Stages are keyed on the
  configuration and the data, not on the code, so an edited notebook still resumes.
- Keep Google Drive mounted, and do not delete the `fourth_and_rl_world_model` folder in
  Drive: that folder is the saved progress.
- To train tonight and evaluate tomorrow, set `STOP_AFTER = "train"` in the next cell, Run
  all, and tomorrow set it back to `None` and Run all again.


In [ ]:
# Parameters.
# STOP_AFTER: None runs everything; "data", "train", "eval", "test" or "plan" stops after
# that stage (the results file is still written).
STOP_AFTER = None
# FORCE: stages to recompute even if done, e.g. ("eval",) after a fix in simulate.py.
FORCE = ()

In [ ]:
# Colab already has torch, numpy, pandas, polars and matplotlib; only nflreadpy is added.
# Nothing is pinned so Colab's own torch build is kept.
!pip install -q nflreadpy pytest

In [ ]:
# Optional: keep checkpoints and outputs on Google Drive so a disconnect loses nothing.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted: outputs go to MyDrive/fourth_and_rl_world_model')
except Exception as e:  # not on Colab, or the mount was declined
    print('Drive not mounted, outputs go to /content/wm_run (lost on disconnect):', e)

In [ ]:
import os
os.makedirs('/content/wm_code', exist_ok=True)

In [ ]:
%%writefile /content/wm_code/schema.py
"""The data contract for the NFL world model. Every other module follows this file.

A game is a sequence of RECORDS, one per snap or kick in regulation (overtime is cut).
Record t holds the pre-snap STATE and the ACTION taken. The model reads records 0..t and
predicts the TRANSITION to record t+1: a few "delta" fields (points scored, seconds
elapsed, timeouts used) plus the next record's categorical fields and its action.
Arithmetic bookkeeping (score, clock, timeouts) is done by `apply_transition`, never by
the model, so those three can never become inconsistent.

All values are small non-negative integers ("codes"). Each field has its own vocabulary
of size `size`; code c of field f is embedded with that field's own embedding table.
Perspective: possession and score are stored from the HOME team's side; yard line is from
the side of the team with the ball (yardline_100, 1 = opponent's 1, 99 = own 1).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

# ---------------------------------------------------------------- categorical values
PHASES = ("PAD", "KICKOFF", "PLAY", "TRY", "END")  # TRY = extra point or two-point try
PHASE = {name: i for i, name in enumerate(PHASES)}

POSS = ("PAD", "HOME", "AWAY")
POSS_CODE = {name: i for i, name in enumerate(POSS)}

ACTIONS = (
    "PAD",
    "RUN",
    "PASS",
    "PUNT",
    "FG",
    "KNEEL",
    "SPIKE",
    "KICKOFF",
    "ONSIDE",
    "XP",
    "TWO_PT",
    "NOPLAY",  # play wiped out by a penalty whose run/pass type is unknown
)
ACTION = {name: i for i, name in enumerate(ACTIONS)}
GO_ACTIONS = (ACTION["RUN"], ACTION["PASS"])  # "go for it" on fourth down = run or pass

# Points change on one transition, HOME minus AWAY. Code 0 is padding.
SCORE_DELTAS = (0, -8, -7, -6, -3, -2, -1, 0, 1, 2, 3, 6, 7, 8)  # index 0 unused (pad)
SCORE_DELTA_CODE = {v: i for i, v in enumerate(SCORE_DELTAS) if i > 0}

# ---------------------------------------------------------------- numeric ranges
MAX_YDSTOGO = 30  # ydstogo is clipped to 1..30; code = ydstogo (0 = not a down)
MAX_ELAPSED = 60  # seconds between records, clipped to 0..60; code = seconds + 1 (0 = pad)
CLOCK_BIN = 5  # clock embedding uses 5-second bins of game seconds remaining
GAME_SECONDS = 3600
HALF_SECONDS = 1800
SCORE_CLIP = 35  # score differential clipped to -35..35; code = diff + 36 (0 = pad)
N_TIMEOUTS = 3
SPREAD_CLIP = 21  # pregame home spread (home favoured by x) clipped, rounded to integers

# ---------------------------------------------------------------- field tables
STATE_FIELDS = (
    # name, vocabulary size
    ("phase", len(PHASES)),
    ("poss", len(POSS)),
    ("down", 5),  # 0 = not a down (kickoff or try), 1..4
    ("ydstogo", MAX_YDSTOGO + 1),  # 0 = not a down
    ("yardline", 100),  # 1..99; 0 = pad
    ("clock", GAME_SECONDS // CLOCK_BIN + 2),  # bin of seconds remaining, +1 offset, 0 = pad
    ("score", 2 * SCORE_CLIP + 2),  # home minus away + 36
    ("to_home", N_TIMEOUTS + 2),  # timeouts + 1, 0 = pad
    ("to_away", N_TIMEOUTS + 2),
    ("action", len(ACTIONS)),
)
STATE_INDEX = {name: i for i, (name, _) in enumerate(STATE_FIELDS)}

# The transition from record t to record t+1, in the order the decoder samples it.
TARGET_FIELDS = (
    ("phase", len(PHASES)),  # END means the game (regulation) is over
    ("poss", len(POSS)),
    ("score_delta", len(SCORE_DELTAS)),
    ("elapsed", MAX_ELAPSED + 2),
    ("to_used_home", 4),  # 0, 1, 2 used (code = used + 1; 0 = pad)
    ("to_used_away", 4),
    ("down", 5),
    ("ydstogo", MAX_YDSTOGO + 1),
    ("yardline", 100),
    ("action", len(ACTIONS)),
)
TARGET_INDEX = {name: i for i, (name, _) in enumerate(TARGET_FIELDS)}

HEADER_FIELDS = (
    ("spread", 2 * SPREAD_CLIP + 2),  # home spread + 22 (0 = pad)
    ("opening_receiver", len(POSS)),  # HOME or AWAY received the opening kickoff
)

# Raw (un-binned) per-record columns kept next to the codes, used for bookkeeping.
RAW_COLUMNS = ("clock_seconds", "score_diff", "timeouts_home", "timeouts_away")


@dataclass
class Split:
    """One split (train, validation or test) in array form.

    states   int16 [N, len(STATE_FIELDS)]   record codes
    raw      int32 [N, len(RAW_COLUMNS)]    unbinned clock seconds, score diff, timeouts
    targets  int16 [N, len(TARGET_FIELDS)]  transition from record i to record i+1; for the
                                            last record of a game phase = END and every
                                            other target field = 0 (masked in the loss)
    offsets  int64 [G + 1]                  records of game g are offsets[g]:offsets[g+1]
    header   int16 [G, len(HEADER_FIELDS)]
    final_diff int16 [G]                    home minus away at the end of regulation
    game_ids list[str]                      nflverse game_id per game
    seasons  int16 [G]
    play_ids int64 [N]                      nflverse play_id per record (for audits)
    """

    states: np.ndarray
    raw: np.ndarray
    targets: np.ndarray
    offsets: np.ndarray
    header: np.ndarray
    final_diff: np.ndarray
    game_ids: list[str]
    seasons: np.ndarray
    play_ids: np.ndarray


# ---------------------------------------------------------------- encoders
def clock_code(seconds: np.ndarray) -> np.ndarray:
    s = np.clip(np.asarray(seconds), 0, GAME_SECONDS)
    return (s // CLOCK_BIN + 1).astype(np.int16)


def score_code(diff: np.ndarray) -> np.ndarray:
    return (np.clip(np.asarray(diff), -SCORE_CLIP, SCORE_CLIP) + SCORE_CLIP + 1).astype(np.int16)


def timeout_code(n: np.ndarray) -> np.ndarray:
    return (np.clip(np.asarray(n), 0, N_TIMEOUTS) + 1).astype(np.int16)


def elapsed_code(seconds: np.ndarray) -> np.ndarray:
    return (np.clip(np.asarray(seconds), 0, MAX_ELAPSED) + 1).astype(np.int16)


def spread_code(spread: np.ndarray) -> np.ndarray:
    s = np.rint(np.clip(np.asarray(spread, dtype=float), -SPREAD_CLIP, SPREAD_CLIP))
    return (s + SPREAD_CLIP + 1).astype(np.int16)


def score_delta_value(code: np.ndarray) -> np.ndarray:
    """Points (home minus away) for a score-delta code; pad code 0 maps to 0."""
    table = np.asarray(SCORE_DELTAS, dtype=np.int32)
    return table[np.asarray(code)]


def apply_transition(raw: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Bookkeeping: the next record's raw columns from the current raw and a transition.

    raw [B, 4] (clock_seconds, score_diff, timeouts_home, timeouts_away); target [B, T].
    Clock falls by the elapsed seconds (never below 0, and never across halftime: a
    transition that crosses 1800 seconds stops at 1800, where both teams get 3 timeouts
    back). Score adds the delta. Timeouts subtract the used counts, floored at 0.
    """
    raw = np.asarray(raw, dtype=np.int32)
    t = np.asarray(target)
    elapsed = np.clip(t[:, TARGET_INDEX["elapsed"]].astype(np.int32) - 1, 0, MAX_ELAPSED)
    clock = raw[:, 0]
    new_clock = np.maximum(clock - elapsed, 0)
    crossed = (clock > HALF_SECONDS) & (new_clock <= HALF_SECONDS)
    new_clock = np.where(crossed, HALF_SECONDS, new_clock)
    score = raw[:, 1] + score_delta_value(t[:, TARGET_INDEX["score_delta"]])
    used_h = np.clip(t[:, TARGET_INDEX["to_used_home"]].astype(np.int32) - 1, 0, 2)
    used_a = np.clip(t[:, TARGET_INDEX["to_used_away"]].astype(np.int32) - 1, 0, 2)
    to_h = np.where(crossed, N_TIMEOUTS, np.maximum(raw[:, 2] - used_h, 0))
    to_a = np.where(crossed, N_TIMEOUTS, np.maximum(raw[:, 3] - used_a, 0))
    return np.stack([new_clock, score, to_h, to_a], axis=1).astype(np.int32)


def next_state_codes(raw_next: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Record codes for record t+1 from its raw columns and the sampled transition."""
    t = np.asarray(target)
    out = np.zeros((len(t), len(STATE_FIELDS)), dtype=np.int16)
    for name in ("phase", "poss", "down", "ydstogo", "yardline", "action"):
        out[:, STATE_INDEX[name]] = t[:, TARGET_INDEX[name]]
    out[:, STATE_INDEX["clock"]] = clock_code(raw_next[:, 0])
    out[:, STATE_INDEX["score"]] = score_code(raw_next[:, 1])
    out[:, STATE_INDEX["to_home"]] = timeout_code(raw_next[:, 2])
    out[:, STATE_INDEX["to_away"]] = timeout_code(raw_next[:, 3])
    return out


def outcome(final_diff: np.ndarray) -> np.ndarray:
    """Home result at the end of regulation: 1 win, 0.5 tie, 0 loss."""
    d = np.asarray(final_diff)
    return np.where(d > 0, 1.0, np.where(d < 0, 0.0, 0.5))


In [ ]:
%%writefile /content/wm_code/records.py
"""Turn nflverse play-by-play into world-model records (see schema.py for the contract).

Standalone: imports only numpy, polars, nflreadpy and the stdlib.

nflverse column facts this module relies on (checked on 2022-2023 data, nflreadpy 0.1.5):
- Kickoffs: `posteam` is the RECEIVING team and `yardline_100` is the kick spot measured
  from the receiver's side (35 for a normal kickoff, 20 for a free kick after a safety).
- `total_home_score` / `total_away_score` are POST-play; `posteam_score` is pre-play. The
  pre-play score of a row is therefore the previous row's totals (0 at game start).
- `home_timeouts_remaining` / `away_timeouts_remaining` are POST-play (the timeout row
  itself, and a play row followed by a timeout, already show the decrement). Pre-play
  timeouts are the previous row's values within the same half (3 at the start of a half).
- Two-point tries: `two_point_attempt == 1` with play_type run or pass and null down.
  Penalised tries are play_type no_play with null down.
- Onside kicks: desc contains "onside" (`own_kickoff_recovery` is the result, not the
  attempt).
- Schedules: `spread_line` > 0 means the HOME team is favoured by that many points.
"""

from __future__ import annotations

import numpy as np
import polars as pl
import schema as S

MIN_SEASON, MAX_SEASON = 2010, 2025  # 2026 is the project's sealed holdout: never load it

RECORD_PLAY_TYPES = (
    "run", "pass", "punt", "field_goal", "kickoff", "extra_point", "qb_kneel", "qb_spike",
    "no_play",
)  # fmt: skip

NEEDED_COLUMNS = (
    "game_id", "play_id", "season", "season_type", "home_team", "away_team", "posteam",
    "order_sequence", "desc", "qtr", "game_half", "down", "ydstogo", "yardline_100",
    "game_seconds_remaining", "total_home_score", "total_away_score",
    "home_timeouts_remaining", "away_timeouts_remaining", "play_type", "play_type_nfl",
    "rush", "pass", "two_point_attempt", "home_score", "away_score",
)  # fmt: skip


def _check_seasons(seasons: list[int]) -> None:
    if not seasons:
        raise ValueError("no seasons given")
    bad = [s for s in seasons if s < MIN_SEASON or s > MAX_SEASON]
    if bad:
        raise ValueError(
            f"seasons {bad} outside {MIN_SEASON}..{MAX_SEASON}; 2026 is a sealed holdout"
        )


def load_pbp(seasons: list[int]) -> pl.DataFrame:
    """Raw nflverse play-by-play for the given seasons (2010..2025 only)."""
    seasons = [int(s) for s in seasons]
    _check_seasons(seasons)
    import nflreadpy as nfl

    return nfl.load_pbp(seasons).filter(pl.col("season").is_in(seasons))


def load_schedules(seasons: list[int]) -> pl.DataFrame:
    """nflverse schedules (game_id, spread_line, ...) for the given seasons."""
    seasons = [int(s) for s in seasons]
    _check_seasons(seasons)
    import nflreadpy as nfl

    # nflverse publishes one schedules file for all seasons; keep only the requested ones
    return nfl.load_schedules(seasons).filter(pl.col("season").is_in(seasons))


# ---------------------------------------------------------------- building
def _prepare(pbp: pl.DataFrame) -> pl.DataFrame:
    """Cast the needed columns and add pre-play score and timeouts (full frame)."""
    missing = [c for c in NEEDED_COLUMNS if c not in pbp.columns]
    if missing:
        raise ValueError(f"play-by-play is missing columns {missing}")
    f = pl.Float64
    df = (
        pbp.select(NEEDED_COLUMNS)
        .with_columns(
            pl.col("desc").fill_null(""),
            pl.col("play_type_nfl").fill_null(""),
            *[
                pl.col(c).cast(f)
                for c in (
                    "qtr",
                    "down",
                    "ydstogo",
                    "yardline_100",
                    "game_seconds_remaining",
                    "total_home_score",
                    "total_away_score",
                    "home_timeouts_remaining",
                    "away_timeouts_remaining",
                    "rush",
                    "pass",
                    "two_point_attempt",
                    "order_sequence",
                    "home_score",
                    "away_score",
                )
            ],  # fmt: skip
        )
        .with_columns(pl.col("order_sequence").fill_null(pl.col("play_id").cast(f)))
        .sort("game_id", "order_sequence", "play_id")
    )
    g = "game_id"
    df = df.with_columns(
        pl.col("total_home_score").forward_fill().over(g).fill_null(0),
        pl.col("total_away_score").forward_fill().over(g).fill_null(0),
        pl.col("home_timeouts_remaining").forward_fill().over(g, "game_half"),
        pl.col("away_timeouts_remaining").forward_fill().over(g, "game_half"),
    )
    return df.with_columns(
        pl.col("total_home_score").shift(1).over(g).fill_null(0).alias("pre_home"),
        pl.col("total_away_score").shift(1).over(g).fill_null(0).alias("pre_away"),
        pl.col("home_timeouts_remaining").shift(1).over(g, "game_half").fill_null(3)
        .clip(0, 3).alias("pre_to_home"),
        pl.col("away_timeouts_remaining").shift(1).over(g, "game_half").fill_null(3)
        .clip(0, 3).alias("pre_to_away"),
    )  # fmt: skip


def _drop(df: pl.DataFrame, keep: pl.Expr, reason: str, dropped: dict) -> pl.DataFrame:
    out = df.filter(keep.fill_null(False))
    dropped[reason] = dropped.get(reason, 0) + df.height - out.height
    return out


def _classify(df: pl.DataFrame) -> pl.DataFrame:
    """Add phase and action codes."""
    P, A = S.PHASE, S.ACTION
    pt = pl.col("play_type")
    desc = pl.col("desc")
    two_pt = pl.col("two_point_attempt") == 1
    noplay_nodown = (pt == "no_play") & pl.col("down").is_null()
    noplay_kick = noplay_nodown & desc.str.contains("(?i) kicks ")
    noplay_try = noplay_nodown & ~noplay_kick
    onside = desc.str.contains("(?i)onside")
    phase = (
        pl.when(two_pt | (pt == "extra_point") | noplay_try).then(P["TRY"])
        .when((pt == "kickoff") | noplay_kick).then(P["KICKOFF"])
        .otherwise(P["PLAY"])
    )  # fmt: skip
    action = (
        pl.when(two_pt).then(A["TWO_PT"])
        .when(pt == "extra_point").then(A["XP"])
        .when(((pt == "kickoff") | noplay_kick) & onside).then(A["ONSIDE"])
        .when((pt == "kickoff") | noplay_kick).then(A["KICKOFF"])
        .when(noplay_try & desc.str.contains("(?i)two-point")).then(A["TWO_PT"])
        .when(noplay_try & desc.str.contains("(?i)kick formation|extra point")).then(A["XP"])
        .when(noplay_try).then(A["NOPLAY"])
        .when(pt == "punt").then(A["PUNT"])
        .when(pt == "field_goal").then(A["FG"])
        .when(pt == "qb_kneel").then(A["KNEEL"])
        .when(pt == "qb_spike").then(A["SPIKE"])
        # dropbacks (incl. sacks and scrambles) are PASS, designed runs RUN
        .when(pl.col("pass") == 1).then(A["PASS"])
        .when(pl.col("rush") == 1).then(A["RUN"])
        .when(pt == "pass").then(A["PASS"])
        .when(pt == "run").then(A["RUN"])
        # a punt or field goal wiped out by a penalty is still the punt / FG decision
        .when((pt == "no_play") & desc.str.contains(" punts ")).then(A["PUNT"])
        .when((pt == "no_play") & desc.str.contains("(?i)field goal is")).then(A["FG"])
        .otherwise(A["NOPLAY"])
    )  # fmt: skip
    return df.with_columns(
        phase.cast(pl.Int16).alias("phase"),
        action.cast(pl.Int16).alias("action"),
        noplay_try.alias("_noplay_try"),
    )


def _select_records(df: pl.DataFrame, dropped: dict, repairs: dict) -> pl.DataFrame:
    pt = pl.col("play_type")
    desc = pl.col("desc")
    df = _drop(df, pl.col("qtr") <= 4, "overtime_or_null_qtr", dropped)
    df = _drop(df, pt.is_in(RECORD_PLAY_TYPES), "not_a_snap_or_kick", dropped)
    df = _drop(df, pl.col("posteam").is_not_null(), "null_posteam", dropped)
    timeout_only = (pt == "no_play") & (
        (pl.col("play_type_nfl") == "TIMEOUT") | desc.str.contains(r"^(\(\d+:\d+\) )?Timeout #")
    )
    df = _drop(df, ~timeout_only, "timeout_only", dropped)
    admin = desc.str.contains("(?i)two-minute warning|end of quarter|end quarter|end of half")
    df = _drop(df, ~(admin & (pt == "no_play")), "two_minute_or_end_of_quarter", dropped)
    df = _drop(df, pl.col("game_seconds_remaining").is_not_null(), "null_clock", dropped)
    df = _drop(df, pl.col("yardline_100").is_not_null(), "null_yardline", dropped)
    df = _classify(df)
    is_play = pl.col("phase") == S.PHASE["PLAY"]
    df = _drop(df, ~is_play | pl.col("down").is_between(1, 4), "play_without_down", dropped)
    df = _drop(df, ~is_play | pl.col("ydstogo").is_not_null(), "play_without_ydstogo", dropped)
    repairs["noplay_null_down_as_try"] = int(df["_noplay_try"].sum())
    yl = pl.col("yardline_100")
    repairs["yardline_clipped_1_99"] = int(df.select(((yl < 1) | (yl > 99)).sum()).item())
    ytg = pl.col("ydstogo")
    repairs["ydstogo_clipped_1_30"] = int(
        df.filter(is_play).select(((ytg < 1) | (ytg > S.MAX_YDSTOGO)).sum()).item()
    )
    clk = pl.col("game_seconds_remaining")
    repairs["clock_clipped_0_3600"] = int(df.select(((clk < 0) | (clk > 3600)).sum()).item())
    # schema.apply_transition resets timeouts when the clock reaches 1800, so a
    # first-half snap at 0:00 of Q2 (try after a TD at the gun, untimed down) would look
    # like the start of the second half. Store it at 1801 s: same clock code (bin 361).
    h1_at_half = (pl.col("game_half") == "Half1") & (clk <= S.HALF_SECONDS)
    repairs["first_half_clock_1800_set_to_1801"] = int(df.select(h1_at_half.sum()).item())
    df = df.with_columns(
        pl.when(h1_at_half).then(S.HALF_SECONDS + 1).otherwise(clk).alias("game_seconds_remaining")
    )
    poss = (
        pl.when(pl.col("posteam") == pl.col("home_team")).then(S.POSS_CODE["HOME"])
        .when(pl.col("posteam") == pl.col("away_team")).then(S.POSS_CODE["AWAY"])
        .otherwise(None)
    )  # fmt: skip
    df = df.with_columns(poss.cast(pl.Int16).alias("poss"))
    df = _drop(df, pl.col("poss").is_not_null(), "posteam_not_home_or_away", dropped)
    return df.with_columns(
        pl.when(is_play).then(pl.col("down")).otherwise(0).cast(pl.Int16).alias("down_c"),
        pl.when(is_play).then(ytg.clip(1, S.MAX_YDSTOGO)).otherwise(0).cast(pl.Int16)
        .alias("ytg_c"),
        yl.clip(1, 99).cast(pl.Int16).alias("yl_c"),
        clk.clip(0, S.GAME_SECONDS).cast(pl.Int32).alias("clock_s"),
        (pl.col("pre_home") - pl.col("pre_away")).cast(pl.Int32).alias("diff"),
        pl.col("pre_to_home").cast(pl.Int32).alias("to_h"),
        pl.col("pre_to_away").cast(pl.Int32).alias("to_a"),
    )  # fmt: skip


# target fields that are 0 on a game's last record (its END target)
END_ZERO_FIELDS = tuple(
    S.TARGET_INDEX[f] for f, _ in S.TARGET_FIELDS if f not in ("phase", "score_delta", "elapsed")
)


def _nearest_delta(d: np.ndarray) -> np.ndarray:
    legal = np.asarray(S.SCORE_DELTAS[1:])
    return legal[np.abs(d[:, None] - legal[None, :]).argmin(axis=1)]


def build_split(
    pbp: pl.DataFrame,
    schedules: pl.DataFrame,
    seasons: list[int],
    season_types: tuple[str, ...] = ("REG", "POST"),
    illegal_delta: str = "drop_game",
) -> tuple[S.Split, dict]:
    """Records for the given seasons as a schema.Split, plus a report of counts.

    illegal_delta: what to do when a transition's points change is not in
    schema.SCORE_DELTAS: "drop_game" (default) drops the whole game, "nearest" maps it to
    the nearest legal value (the bookkeeping check then fails on that pair).
    """
    seasons = [int(s) for s in seasons]
    _check_seasons(seasons)
    dropped: dict[str, int] = {}
    repairs: dict[str, int] = {}
    soft: dict[str, int] = {}
    raw = pbp.filter(
        pl.col("season").is_in(seasons) & pl.col("season_type").is_in(list(season_types))
    )
    df = _prepare(raw)
    rows_in = df.height

    # end-of-regulation score: post-play totals of the last regulation row
    reg_end = (
        df.filter(pl.col("qtr") <= 4)
        .group_by("game_id", maintain_order=True)
        .agg(
            (pl.col("total_home_score").last() - pl.col("total_away_score").last())
            .cast(pl.Int32).alias("final_diff"),
            (pl.col("home_score").first() - pl.col("away_score").first()).alias("box_diff"),
            (pl.col("qtr").max() <= 4).alias("no_ot_rows"),
        )
    )  # fmt: skip
    ot_games = set(df.filter(pl.col("qtr") >= 5)["game_id"].unique().to_list())
    reg_end = reg_end.with_columns(pl.col("game_id").is_in(list(ot_games)).alias("ot"))

    rec = _select_records(df, dropped, repairs)

    # ---- transitions (numpy over the sorted record table)
    gid = rec["game_id"].to_numpy()
    n = len(gid)
    starts = np.flatnonzero(np.r_[True, gid[1:] != gid[:-1]]) if n else np.zeros(0, int)
    last = np.zeros(n, bool)
    if n:
        last[np.r_[starts[1:] - 1, n - 1]] = True
    nxt = np.minimum(np.arange(n) + 1, max(n - 1, 0))

    phase = rec["phase"].to_numpy().astype(np.int16)
    poss = rec["poss"].to_numpy().astype(np.int16)
    down = rec["down_c"].to_numpy().astype(np.int16)
    ytg = rec["ytg_c"].to_numpy().astype(np.int16)
    yl = rec["yl_c"].to_numpy().astype(np.int16)
    action = rec["action"].to_numpy().astype(np.int16)
    clock = rec["clock_s"].to_numpy().astype(np.int32)
    diff = rec["diff"].to_numpy().astype(np.int32)
    to_h = rec["to_h"].to_numpy().astype(np.int32)
    to_a = rec["to_a"].to_numpy().astype(np.int32)
    half = rec["game_half"].to_numpy()
    within = ~last

    # points on a game's final record (FG as time expires, TD or try at 0:00) go in its
    # END target: final regulation diff minus the last record's pre-play diff
    fd_map = dict(reg_end.select("game_id", "final_diff").iter_rows())
    end_fd = np.array([fd_map[g] for g in gid[last]], dtype=np.int32)
    delta = diff[nxt] - diff
    delta[last] = end_fd - diff[last]
    soft["games_scoring_on_last_record"] = int((delta[last] != 0).sum())
    legal = np.isin(delta, S.SCORE_DELTAS[1:])
    bad_games = set(gid[~legal].tolist())
    soft["score_delta_illegal_pairs"] = int((~legal).sum())
    if illegal_delta == "nearest":
        soft["score_delta_mapped"] = int((~legal).sum())
        delta = np.where(legal, delta, _nearest_delta(delta))
        bad_games = set()
    elapsed = clock - clock[nxt]
    elapsed[last] = clock[last]  # regulation ends at 0 seconds
    soft["elapsed_negative"] = int((within & (elapsed < 0)).sum())
    soft["elapsed_over_60"] = int((within & (elapsed > S.MAX_ELAPSED)).sum())
    soft["elapsed_over_45"] = int((within & (elapsed > 45)).sum())
    crossing = within & (half == "Half1") & (half[nxt] == "Half2")
    soft["halftime_crossings"] = int(crossing.sum())
    used_h = np.where(crossing, 0, np.maximum(0, to_h - to_h[nxt]))
    used_a = np.where(crossing, 0, np.maximum(0, to_a - to_a[nxt]))
    soft["timeout_increase_mid_half"] = int(
        (within & ~crossing & ((to_h[nxt] > to_h) | (to_a[nxt] > to_a))).sum()
    )
    soft["timeouts_used_over_2"] = int((within & ((used_h > 2) | (used_a > 2))).sum())

    T = S.TARGET_INDEX
    targets = np.zeros((n, len(S.TARGET_FIELDS)), np.int16)
    targets[:, T["phase"]] = np.where(last, S.PHASE["END"], phase[nxt])
    targets[:, T["poss"]] = poss[nxt]
    targets[:, T["score_delta"]] = [S.SCORE_DELTA_CODE.get(int(d), 0) for d in delta]
    targets[:, T["elapsed"]] = S.elapsed_code(elapsed)
    targets[:, T["to_used_home"]] = np.clip(used_h, 0, 2) + 1
    targets[:, T["to_used_away"]] = np.clip(used_a, 0, 2) + 1
    targets[:, T["down"]] = down[nxt]
    targets[:, T["ydstogo"]] = ytg[nxt]
    targets[:, T["yardline"]] = yl[nxt]
    targets[:, T["action"]] = action[nxt]
    # END target: phase, score_delta and elapsed are real; every other field is 0 (masked)
    targets[np.ix_(last, list(END_ZERO_FIELDS))] = 0

    SI = S.STATE_INDEX
    states = np.zeros((n, len(S.STATE_FIELDS)), np.int16)
    states[:, SI["phase"]] = phase
    states[:, SI["poss"]] = poss
    states[:, SI["down"]] = down
    states[:, SI["ydstogo"]] = ytg
    states[:, SI["yardline"]] = yl
    states[:, SI["clock"]] = S.clock_code(clock)
    states[:, SI["score"]] = S.score_code(diff)
    states[:, SI["to_home"]] = S.timeout_code(to_h)
    states[:, SI["to_away"]] = S.timeout_code(to_a)
    states[:, SI["action"]] = action
    raw_cols = np.stack([clock, diff, to_h, to_a], axis=1).astype(np.int32)
    play_ids = rec["play_id"].cast(pl.Float64).to_numpy().astype(np.int64)
    rec_season = rec["season"].to_numpy()

    # ---- drop games with illegal score deltas
    keep = ~np.isin(gid, list(bad_games)) if bad_games else np.ones(n, bool)
    dropped["games_with_illegal_score_delta"] = len(bad_games)
    dropped["records_in_dropped_games"] = int((~keep).sum())
    states, raw_cols, targets = states[keep], raw_cols[keep], targets[keep]
    play_ids, gid, rec_season = play_ids[keep], gid[keep], rec_season[keep]
    m = len(gid)
    starts = np.flatnonzero(np.r_[True, gid[1:] != gid[:-1]]) if m else np.zeros(0, int)
    offsets = np.r_[starts, m].astype(np.int64)
    game_ids = [str(x) for x in gid[starts]]
    seasons_arr = rec_season[starts].astype(np.int16)

    # ---- header
    games = pl.DataFrame({"game_id": game_ids, "_g": np.arange(len(game_ids))})
    sched = schedules.select("game_id", pl.col("spread_line").cast(pl.Float64))
    games = games.join(sched, on="game_id", how="left").sort("_g")
    spread = games["spread_line"].to_numpy()
    soft["games_missing_spread"] = int(np.isnan(spread).sum()) if len(spread) else 0
    spread = np.nan_to_num(spread, nan=0.0)
    header = np.zeros((len(game_ids), len(S.HEADER_FIELDS)), np.int16)
    header[:, 0] = S.spread_code(spread)
    opening = np.zeros(len(game_ids), np.int16)
    soft["games_first_record_not_kickoff"] = 0
    for g in range(len(game_ids)):
        s, e = offsets[g], offsets[g + 1]
        ko = np.flatnonzero(states[s:e, SI["phase"]] == S.PHASE["KICKOFF"])
        if len(ko) == 0 or ko[0] != 0:
            soft["games_first_record_not_kickoff"] += 1
        opening[g] = states[s + ko[0], SI["poss"]] if len(ko) else states[s, SI["poss"]]
    header[:, 1] = opening

    # ---- final regulation score
    fin = games.join(reg_end, on="game_id", how="left").sort("_g")
    final_diff = fin["final_diff"].to_numpy().astype(np.int16)
    no_ot = ~fin["ot"].to_numpy().astype(bool)
    box = fin["box_diff"].to_numpy()
    soft["final_diff_vs_box_mismatch_non_ot"] = int((no_ot & (final_diff != box)).sum())

    split = S.Split(
        states=states,
        raw=raw_cols,
        targets=targets,
        offsets=offsets,
        header=header,
        final_diff=final_diff,
        game_ids=game_ids,
        seasons=seasons_arr,
        play_ids=play_ids,
    )
    sizes = np.diff(offsets)
    report = {
        "seasons": seasons,
        "rows_in": rows_in,
        "rows_kept": int(m),
        "dropped": dropped,
        "repairs": repairs,
        "soft": soft,
        "games": len(game_ids),
        "overtime_games_cut": int(np.sum(~no_ot)),
        "records_per_game": {
            "min": int(sizes.min()) if len(sizes) else 0,
            "mean": round(float(sizes.mean()), 1) if len(sizes) else 0.0,
            "max": int(sizes.max()) if len(sizes) else 0,
        },
    }
    return split, report


# ---------------------------------------------------------------- validation
def _pair_mask(split: S.Split) -> np.ndarray:
    """True for record i when i+1 is in the same game."""
    n = len(split.states)
    m = np.ones(n, bool)
    if n:
        m[split.offsets[1:] - 1] = False
    return m


def transition_mismatches(split: S.Split) -> dict:
    """Within-game pairs where apply_transition does not reproduce raw[i+1], by reason."""
    pair = _pair_mask(split)
    idx = np.flatnonzero(pair)
    pred = S.apply_transition(split.raw[idx], split.targets[idx])
    true = split.raw[idx + 1]
    bad = pred != true
    out = {"pairs": int(len(idx)), "mismatched_pairs": int(bad.any(axis=1).sum())}
    for j, name in enumerate(S.RAW_COLUMNS):
        out[f"mismatch_{name}"] = int(bad[:, j].sum())
    any_bad = bad.any(axis=1)
    el = split.raw[idx, 0] - split.raw[idx + 1, 0]
    half_at_1800 = (
        (split.raw[idx, 0] == S.HALF_SECONDS)
        & (split.states[idx + 1, S.STATE_INDEX["phase"]] == S.PHASE["KICKOFF"])
        & (split.raw[idx + 1, 0] == S.HALF_SECONDS)
    )
    reasons = {
        "reason_elapsed_over_60": any_bad & (el > S.MAX_ELAPSED),
        "reason_elapsed_negative": any_bad & (el < 0),
        "reason_untimed_down_at_1800_before_halftime": any_bad & half_at_1800,
    }
    explained = np.zeros(len(idx), bool)
    for k, v in reasons.items():
        out[k] = int((v & ~explained).sum())
        explained |= v
    out["reason_other"] = int((any_bad & ~explained).sum())
    return out


def validate_split(split: S.Split, min_transition_match: float = 0.99) -> dict:
    """Invariant checks. Returns check -> violations; raises AssertionError on hard ones."""
    st, tg, off = split.states, split.targets, split.offsets
    n, G = len(st), len(split.game_ids)
    hard: dict[str, int] = {}
    for j, (name, size) in enumerate(S.STATE_FIELDS):
        hard[f"state_{name}_out_of_vocab"] = int(((st[:, j] < 0) | (st[:, j] >= size)).sum())
    for j, (name, size) in enumerate(S.TARGET_FIELDS):
        hard[f"target_{name}_out_of_vocab"] = int(((tg[:, j] < 0) | (tg[:, j] >= size)).sum())
    for j, (name, size) in enumerate(S.HEADER_FIELDS):
        col = split.header[:, j]
        hard[f"header_{name}_out_of_vocab"] = int(((col <= 0) | (col >= size)).sum())
    SI, T = S.STATE_INDEX, S.TARGET_INDEX
    never_zero = ("phase", "poss", "yardline", "clock", "score", "to_home", "to_away", "action")
    hard["state_pad_codes"] = int(sum((st[:, SI[f]] == 0).sum() for f in never_zero))
    hard["state_phase_end_or_pad"] = int(np.isin(st[:, SI["phase"]], [0, S.PHASE["END"]]).sum())
    is_play = st[:, SI["phase"]] == S.PHASE["PLAY"]
    hard["down_zero_iff_not_play"] = int(((st[:, SI["down"]] == 0) == is_play).sum())
    hard["ydstogo_zero_iff_not_play"] = int(((st[:, SI["ydstogo"]] == 0) == is_play).sum())
    hard["offsets_bad"] = int(
        len(off) != G + 1 or (G >= 0 and (off[0] != 0 or off[-1] != n))
        or bool(np.any(np.diff(off) <= 0))
    )  # fmt: skip
    if hard["offsets_bad"]:
        raise AssertionError("validate_split failed: offsets do not cover the records")
    pair = _pair_mask(split)
    is_end = tg[:, T["phase"]] == S.PHASE["END"]
    hard["end_not_exactly_last"] = int((is_end != ~pair).sum())
    end_tg = tg[~pair]
    hard["last_target_nonzero_fields"] = int((end_tg[:, list(END_ZERO_FIELDS)] != 0).sum())
    hard["end_target_pad_score_or_elapsed"] = int(
        ((end_tg[:, T["score_delta"]] == 0) | (end_tg[:, T["elapsed"]] == 0)).sum()
    )
    # the last record's score plus its END score delta is the final regulation score
    end_score = split.raw[~pair, 1] + S.score_delta_value(end_tg[:, T["score_delta"]])
    fd_bad = end_score != split.final_diff.astype(np.int32)
    hard["final_diff_ne_last_score_plus_end_delta"] = int(fd_bad.sum())
    pad_tg = tg[pair][
        :,
        [
            T[f]
            for f in (
                "phase",
                "poss",
                "score_delta",
                "elapsed",
                "to_used_home",
                "to_used_away",
                "yardline",
                "action",
            )
        ],
    ]
    hard["target_pad_codes"] = int((pad_tg == 0).sum())
    # target categorical fields equal the next record's fields (exact by construction)
    idx = np.flatnonzero(pair)
    nxt = S.next_state_codes(split.raw[idx + 1], tg[idx])
    hard["target_ne_next_state"] = int((nxt != st[idx + 1]).any(axis=1).sum())
    hard["duplicate_game_ids"] = G - len(set(split.game_ids))
    tm = transition_mismatches(split)
    frac = 1 - tm["mismatched_pairs"] / max(tm["pairs"], 1)
    hard["apply_transition_below_threshold"] = int(frac < min_transition_match)
    report = {**hard, **{f"info_{k}": v for k, v in tm.items()}}
    report["info_apply_transition_match_fraction"] = round(frac, 5)
    report["info_final_diff_mismatch_by_season"] = {
        int(y): int(fd_bad[split.seasons == y].sum()) for y in np.unique(split.seasons)
    }
    report["info_games_scoring_on_last_record"] = int(
        (S.score_delta_value(end_tg[:, T["score_delta"]]) != 0).sum()
    )
    sd = tg[pair, T["score_delta"]]
    report["soft_elapsed_over_45"] = int((tg[pair, T["elapsed"]] - 1 > 45).sum())
    report["soft_nonzero_score_deltas"] = int((S.score_delta_value(sd) != 0).sum())
    failed = {k: v for k, v in hard.items() if v}
    if failed:
        raise AssertionError(f"validate_split failed: {failed}")
    return report


def validate_disjoint(splits: dict[str, S.Split]) -> dict:
    """No game id in two splits. Raises AssertionError if any is shared."""
    seen: dict[str, str] = {}
    shared = 0
    for name, sp in splits.items():
        for g in sp.game_ids:
            if g in seen and seen[g] != name:
                shared += 1
            seen[g] = name
    if shared:
        raise AssertionError(f"{shared} game ids appear in more than one split")
    return {"shared_game_ids": 0}


# ---------------------------------------------------------------- inspection
def _game_of(split: S.Split, i: int) -> int:
    return int(np.searchsorted(split.offsets, i, side="right") - 1)


def decode_record(split: S.Split, i: int) -> dict:
    """Human-readable state, action and transition target of record i."""
    st, tg, rw = split.states[i], split.targets[i], split.raw[i]
    SI, T = S.STATE_INDEX, S.TARGET_INDEX
    g = _game_of(split, i)
    is_end = int(tg[T["phase"]]) == S.PHASE["END"]
    target = {
        "phase": S.PHASES[tg[T["phase"]]],
        "score_delta": int(S.SCORE_DELTAS[tg[T["score_delta"]]]),
        "elapsed": int(tg[T["elapsed"]]) - 1,
    }
    if not is_end:
        target.update(
            poss=S.POSS[tg[T["poss"]]],
            score_delta=int(S.SCORE_DELTAS[tg[T["score_delta"]]]),
            elapsed=int(tg[T["elapsed"]]) - 1,
            to_used_home=int(tg[T["to_used_home"]]) - 1,
            to_used_away=int(tg[T["to_used_away"]]) - 1,
            down=int(tg[T["down"]]),
            ydstogo=int(tg[T["ydstogo"]]),
            yardline=int(tg[T["yardline"]]),
            action=S.ACTIONS[tg[T["action"]]],
        )
    return {
        "game_id": split.game_ids[g],
        "play_id": int(split.play_ids[i]),
        "record_in_game": int(i - split.offsets[g]),
        "phase": S.PHASES[st[SI["phase"]]],
        "poss": S.POSS[st[SI["poss"]]],
        "down": int(st[SI["down"]]),
        "ydstogo": int(st[SI["ydstogo"]]),
        "yardline_100": int(st[SI["yardline"]]),
        "clock_seconds": int(rw[0]),
        "score_diff_home": int(rw[1]),
        "timeouts_home": int(rw[2]),
        "timeouts_away": int(rw[3]),
        "action": S.ACTIONS[st[SI["action"]]],
        "target": target,
    }


def fourth_down_index(split: S.Split) -> np.ndarray:
    """Records that are real fourth-down decisions: PLAY, down 4, RUN/PASS/PUNT/FG."""
    st = split.states
    SI, A = S.STATE_INDEX, S.ACTION
    ok = (
        (st[:, SI["phase"]] == S.PHASE["PLAY"])
        & (st[:, SI["down"]] == 4)
        & np.isin(st[:, SI["action"]], [A["RUN"], A["PASS"], A["PUNT"], A["FG"]])
    )
    return np.flatnonzero(ok)


In [ ]:
%%writefile /content/wm_code/model.py
"""A causal transformer "world model" over game records, with an autoregressive field decoder.

Sequence per game: position 0 is the header token (sum of header field embeddings), then one
token per record (sum of that record's STATE_FIELDS embeddings). Every token also gets a
learned positional embedding. A pre-LayerNorm causal transformer turns record t into h_t.

The FIELD DECODER predicts the transition t -> t+1 one field at a time, in TARGET_FIELDS order:
    logits_j = MLP_j([h_t ; e_0(y_0) + ... + e_{j-1}(y_{j-1})])
where e_k is an embedding table for target field k and y_k is the true code (training, teacher
forcing) or the sampled code (generation). Each head outputs only its own field's vocabulary.

Standalone: imports only torch, numpy and the sibling `schema` module.
"""

from __future__ import annotations

import numpy as np
import schema as S
import torch
import torch.nn.functional as F
from torch import nn

DEFAULT_CFG: dict = {
    # model
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "d_ff": 1024,
    "dec_hidden": 256,
    "dropout": 0.1,
    "max_len": 256,  # records of context; longer games keep their LAST max_len records
    # training
    "seed": 0,
    "epochs": 30,
    "batch_games": 32,
    "eval_batch_games": 64,
    "lr": 3e-4,
    "min_lr_frac": 0.1,
    "weight_decay": 0.1,
    "betas": (0.9, 0.95),
    "warmup_steps": 300,
    "grad_clip": 1.0,
    "patience": 4,
    "stop_after_epoch": None,  # stop cleanly after this many epochs (schedule unchanged)
    # simulation
    "temperature": 1.0,
    "max_steps": 400,  # transitions per rollout before it is counted as truncated
    "sim_batch": 1024,  # rollouts per transformer forward pass
    "sim_max_live": 8192,  # rollouts held at once (queries are processed in groups)
    "mask_pad_codes": True,  # never sample code 0 for fields where 0 means padding
    "score_on_end": True,  # add the sampled score delta of the END transition (final play)
    "sort_by_length": True,  # simulator: encode rows sorted by context length (less padding)
}

# Target fields where code 0 is a real value ("not a down"), not padding.
ZERO_IS_VALUE = frozenset({"down", "ydstogo"})
TARGET_NAMES = tuple(name for name, _ in S.TARGET_FIELDS)
PHASE_T = S.TARGET_INDEX["phase"]
# Fields that carry a real value on a game's LAST record (target phase END): the final play's
# points and seconds. Every other field is padding there: it never counts in the loss, and its
# decoder input is forced to code 0 both in training and in sampling, so the score_delta and
# elapsed heads see the same prefix (END, poss=0) in both.
END_FIELDS = frozenset({"phase", "score_delta", "elapsed"})
NOT_END_T = tuple(j for j, n in enumerate(TARGET_NAMES) if n not in END_FIELDS)


def merge_cfg(cfg: dict | None) -> dict:
    out = dict(DEFAULT_CFG)
    out.update(cfg or {})
    return out


class Block(nn.Module):
    """Pre-LayerNorm transformer block with causal scaled-dot-product attention."""

    def __init__(self, d: int, n_heads: int, d_ff: int, p: float) -> None:
        super().__init__()
        assert d % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.p = p
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))
        self.drop = nn.Dropout(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, d = x.shape
        qkv = self.qkv(self.ln1(x)).view(b, n, 3, self.n_heads, d // self.n_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # each [b, heads, n, d_head]
        a = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.p if self.training else 0.0, is_causal=True
        )
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(b, n, d)))
        return x + self.drop(self.ff(self.ln2(x)))


class WorldModel(nn.Module):
    def __init__(self, cfg: dict | None = None) -> None:
        super().__init__()
        self.cfg = merge_cfg(cfg)
        c = self.cfg
        d = c["d_model"]
        self.max_len = int(c["max_len"])
        self.state_emb = nn.ModuleList(nn.Embedding(n, d) for _, n in S.STATE_FIELDS)
        self.header_emb = nn.ModuleList(nn.Embedding(n, d) for _, n in S.HEADER_FIELDS)
        self.pos_emb = nn.Embedding(self.max_len + 1, d)  # 0 = header, 1..max_len = records
        self.drop = nn.Dropout(c["dropout"])
        self.blocks = nn.ModuleList(
            Block(d, c["n_heads"], c["d_ff"], c["dropout"]) for _ in range(c["n_layers"])
        )
        self.ln_f = nn.LayerNorm(d)
        # embeddings of target fields 0..T-2 (the last field is never a decoder input)
        self.tgt_emb = nn.ModuleList(nn.Embedding(n, d) for _, n in S.TARGET_FIELDS[:-1])
        h = c["dec_hidden"]
        self.heads = nn.ModuleList(
            nn.Sequential(nn.Linear(2 * d, h), nn.GELU(), nn.Linear(h, n))
            for _, n in S.TARGET_FIELDS
        )
        self.apply(self._init)

    @staticmethod
    def _init(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters())

    # ------------------------------------------------------------------ encoder
    def encode(self, header: torch.Tensor, states: torch.Tensor) -> torch.Tensor:
        """header [B, H], states [B, L, S] (L <= max_len, right-padded) -> h [B, L, d]."""
        b, n, _ = states.shape
        if n > self.max_len:
            raise ValueError(f"{n} records exceed max_len={self.max_len}; cut to the last ones")
        x0 = sum(emb(header[:, k]) for k, emb in enumerate(self.header_emb)).unsqueeze(1)
        xr = sum(emb(states[:, :, k]) for k, emb in enumerate(self.state_emb))
        x = torch.cat([x0, xr], dim=1) + self.pos_emb.weight[: n + 1].unsqueeze(0)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        return self.ln_f(x)[:, 1:]

    # ------------------------------------------------------------------ decoder
    def _prefix_sums(self, targets: torch.Tensor) -> torch.Tensor:
        """targets [..., T] -> s [..., T, d], s_j = sum of embeddings of fields 0..j-1."""
        e = torch.stack([emb(targets[..., k]) for k, emb in enumerate(self.tgt_emb)], dim=-2)
        c = torch.cumsum(e, dim=-2)
        return torch.cat([torch.zeros_like(c[..., :1, :]), c], dim=-2)

    def decode_logits(self, h: torch.Tensor, targets: torch.Tensor) -> list[torch.Tensor]:
        """Teacher-forced logits for every target field. h [..., d], targets [..., T]."""
        s = self._prefix_sums(targets)
        return [head(torch.cat([h, s[..., j, :]], dim=-1)) for j, head in enumerate(self.heads)]

    def field_logits(self, h: torch.Tensor, prefix: torch.Tensor, j: int) -> torch.Tensor:
        """Logits of field j given h [R, d] and the codes of fields 0..j-1 in prefix [R, >=j]."""
        s = torch.zeros_like(h)
        for k in range(j):
            s = s + self.tgt_emb[k](prefix[:, k])
        return self.heads[j](torch.cat([h, s], dim=-1))

    @torch.no_grad()
    def sample_transition(
        self,
        h: torch.Tensor,
        temperature: float = 1.0,
        generator: torch.Generator | None = None,
        mask_pad: bool = True,
    ) -> torch.Tensor:
        """Sample one transition per row of h [R, d] -> codes [R, T] (long).

        The heads run in float32 even inside an autocast region: bf16 logits are rounded to
        about 1/64 of their size, which would bias the sampled probabilities by around 1%.
        On rows whose sampled phase is END, fields outside END_FIELDS are set to code 0
        (the value they have on every END row in training).
        """
        r = h.shape[0]
        out = torch.zeros(r, len(S.TARGET_FIELDS), dtype=torch.long, device=h.device)
        with torch.autocast(h.device.type, enabled=False):
            h = h.float()
            s = torch.zeros_like(h)
            is_end = None
            for j, head in enumerate(self.heads):
                logits = head(torch.cat([h, s], dim=-1)).float()
                if mask_pad and TARGET_NAMES[j] not in ZERO_IS_VALUE:
                    logits[:, 0] = -float("inf")
                if temperature <= 0:
                    code = logits.argmax(-1)
                else:
                    u = torch.rand(logits.shape, generator=generator, device=h.device)
                    gumbel = -torch.log(-torch.log(u.clamp_(1e-10, 1.0 - 1e-7)))
                    code = (logits / temperature + gumbel).argmax(-1)
                if j == PHASE_T:
                    is_end = code == S.PHASE["END"]
                elif TARGET_NAMES[j] not in END_FIELDS:
                    code = torch.where(is_end, torch.zeros_like(code), code)
                out[:, j] = code
                if j < len(self.tgt_emb):
                    s = s + self.tgt_emb[j](code)
        return out

    # ------------------------------------------------------------------ loss
    @staticmethod
    def end_prefix(targets: torch.Tensor, last: torch.Tensor) -> torch.Tensor:
        """Targets with every field outside END_FIELDS set to 0 on last records."""
        tg = targets.clone()
        tg[..., list(NOT_END_T)] = tg[..., list(NOT_END_T)] * (~last).unsqueeze(-1).to(tg.dtype)
        return tg

    @staticmethod
    def target_masks(targets: torch.Tensor, mask: torch.Tensor, last: torch.Tensor) -> list:
        """Per field, which targets count in the loss ([B, L] bool each).

        A target counts when its record is real and its code is not padding. Code 0 is
        padding for every field except down and ydstogo ("not a down"). On a game's last
        record only END_FIELDS count: phase (END) always, score_delta and elapsed when
        records.py filled them (code > 0).
        """
        out = []
        for j, name in enumerate(TARGET_NAMES):
            y = targets[..., j]
            if j == PHASE_T:
                m = mask
            elif name in ZERO_IS_VALUE:
                m = mask & ~last
            else:
                m = mask & (y != 0)
                if name not in END_FIELDS:
                    m = m & ~last
            out.append(m)
        return out

    def loss(self, batch: dict, denom: torch.Tensor | None = None) -> tuple[torch.Tensor, dict]:
        """Sum over fields of the mean cross-entropy per target (see target_masks).

        batch: header [B,H], states [B,L,S], targets [B,L,T], mask [B,L] (real record),
        last [B,L] (last record of its game). denom [T], if given, replaces the per-field
        target counts in the means (used to split one batch into exact micro-batches).
        Returns (total, {"fields": {name: mean NLL tensor}, "sums": [T], "counts": [T]}).
        """
        h = self.encode(batch["header"], batch["states"])
        mask, last = batch["mask"], batch["last"]
        tg = self.end_prefix(batch["targets"], last)
        logits = self.decode_logits(h, tg)
        masks = self.target_masks(tg, mask, last)
        total = h.new_zeros((), dtype=torch.float32)
        fields, sums, counts = {}, [], []
        for j, (name, _) in enumerate(S.TARGET_FIELDS):
            y = tg[..., j]
            m = masks[j]
            ce = F.cross_entropy(logits[j].float().flatten(0, 1), y.flatten(), reduction="none")
            ce = ce.view_as(y) * m
            n = m.sum()
            sm = ce.sum()
            d = n.clamp(min=1) if denom is None else denom[j].clamp(min=1)
            mean = sm / d
            fields[name] = mean
            sums.append(sm.detach())
            counts.append(n.detach())
            total = total + mean
        return total, {"fields": fields, "sums": torch.stack(sums), "counts": torch.stack(counts)}


def window_start(game_start: int, i: int, max_len: int) -> int:
    """First record of the context window that ends at record i (inclusive)."""
    return max(game_start, i + 1 - max_len)


def set_seed(seed: int) -> None:
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def device_of(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def param_summary(model: WorldModel) -> str:
    n = model.count_params()
    return f"{n:,} parameters ({n / 1e6:.2f} M), {4 * n / 2**20:.1f} MiB in float32"


__all__ = [
    "DEFAULT_CFG",
    "ZERO_IS_VALUE",
    "END_FIELDS",
    "WorldModel",
    "merge_cfg",
    "window_start",
    "set_seed",
    "device_of",
    "param_summary",
]


In [ ]:
%%writefile /content/wm_code/train.py
"""Batching, training loop (AdamW, warmup + cosine, AMP, resume, early stopping) and NLL."""

from __future__ import annotations

import copy
import math
import os
import time
from collections.abc import Callable, Iterator

import numpy as np
import schema as S
import torch
from model import TARGET_NAMES, WorldModel, device_of, merge_cfg


# ---------------------------------------------------------------------- batching
def collate(split: S.Split, games: np.ndarray, max_len: int) -> dict:
    """Pad the given games into one batch, one game per row, each cut to its last max_len."""
    starts = split.offsets[games]
    ends = split.offsets[games + 1]
    starts = np.maximum(starts, ends - max_len)
    lens = (ends - starts).astype(np.int64)
    b, n = len(games), int(lens.max())
    states = np.zeros((b, n, len(S.STATE_FIELDS)), dtype=np.int64)
    targets = np.zeros((b, n, len(S.TARGET_FIELDS)), dtype=np.int64)
    mask = np.zeros((b, n), dtype=bool)
    last = np.zeros((b, n), dtype=bool)
    for r, (a, e, k) in enumerate(zip(starts, ends, lens, strict=True)):
        states[r, :k] = split.states[a:e]
        targets[r, :k] = split.targets[a:e]
        mask[r, :k] = True
        last[r, k - 1] = True
    return {
        "header": torch.from_numpy(split.header[games].astype(np.int64)),
        "states": torch.from_numpy(states),
        "targets": torch.from_numpy(targets),
        "mask": torch.from_numpy(mask),
        "last": torch.from_numpy(last),
        "games": torch.from_numpy(np.asarray(games, dtype=np.int64)),
    }


def make_batches(
    split: S.Split,
    batch_games: int,
    max_len: int,
    rng: np.random.Generator | None,
) -> Iterator[dict]:
    """Yield padded batches of whole games. rng=None keeps the split's order (evaluation)."""
    n_games = len(split.offsets) - 1
    order = np.arange(n_games) if rng is None else rng.permutation(n_games)
    for b in range(0, n_games, batch_games):
        yield collate(split, order[b : b + batch_games], max_len)


def to_device(batch: dict, device: torch.device) -> dict:
    nb = device.type == "cuda"
    return {k: v.to(device, non_blocking=nb) for k, v in batch.items()}


# ---------------------------------------------------------------------- precision
def amp_settings(device: torch.device) -> tuple[bool, torch.dtype, bool]:
    """(enabled, dtype, use_grad_scaler). bf16 only on GPUs with native support (sm80+)."""
    if device.type != "cuda":
        return False, torch.float32, False
    major, _ = torch.cuda.get_device_capability(device)
    if major >= 8:
        return True, torch.bfloat16, False
    return True, torch.float16, True


# ---------------------------------------------------------------------- evaluation
@torch.no_grad()
def evaluate_nll(model: WorldModel, split: S.Split, cfg: dict | None = None) -> dict:
    """Per-field mean NLL (nats per target) and total NLL per record over a split."""
    c = merge_cfg({**model.cfg, **(cfg or {})})
    device = device_of(model)
    enabled, dtype, _ = amp_settings(device)
    was_training = model.training
    model.eval()
    sums = torch.zeros(len(S.TARGET_FIELDS), dtype=torch.float64)
    counts = torch.zeros(len(S.TARGET_FIELDS), dtype=torch.float64)
    n_records = 0
    for batch in make_batches(split, c["eval_batch_games"], model.max_len, None):
        batch = to_device(batch, device)
        with torch.autocast(device.type, dtype=dtype, enabled=enabled):
            _, info = model.loss(batch)
        sums += info["sums"].double().cpu()
        counts += info["counts"].double().cpu()
        n_records += int(batch["mask"].sum())
    model.train(was_training)
    fields = {n: float(sums[j] / max(counts[j], 1.0)) for j, n in enumerate(TARGET_NAMES)}
    return {
        "total": float(sums.sum() / max(n_records, 1)),  # nats per record, all fields
        "sum_of_field_means": float(sum(fields.values())),
        "fields": fields,
        "n_records": n_records,
    }


# ---------------------------------------------------------------------- training
def _lr_factor(step: int, progress: float, cfg: dict) -> float:
    """Linear warmup over warmup_steps, then cosine from 1 to min_lr_frac over all epochs."""
    warm = min(1.0, (step + 1) / max(1, cfg["warmup_steps"]))
    p = min(1.0, max(0.0, progress / max(1, cfg["epochs"])))
    floor = cfg["min_lr_frac"]
    return warm * (floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * p)))


def _param_groups(model: torch.nn.Module, weight_decay: float) -> list[dict]:
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        is_matrix = p.dim() >= 2 and "emb" not in name
        (decay if is_matrix else no_decay).append(p)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def _save(path: str, payload: dict) -> None:
    tmp = f"{path}.tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)


def _is_oom(err: BaseException) -> bool:
    oom = getattr(torch, "OutOfMemoryError", None) or getattr(torch.cuda, "OutOfMemoryError", None)
    return (oom is not None and isinstance(err, oom)) or "out of memory" in str(err).lower()


def load_checkpoint(path: str, cfg_override: dict | None = None, device: str | None = None):
    """Rebuild the BEST model stored at path. Returns (model, checkpoint dict)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model = WorldModel({**ck["cfg"], **(cfg_override or {})})
    model.load_state_dict(ck["model"])
    if device is not None:
        model.to(device)
    model.eval()
    return model, ck


def _micro_slices(batch: dict, k: int) -> list[dict]:
    """Split a batch into k row groups, each trimmed to its own longest game (the causal
    encoder makes right padding irrelevant, so the results do not change)."""
    b = batch["mask"].shape[0]
    out = []
    for rows in np.array_split(np.arange(b), min(k, b)):
        if len(rows) == 0:
            continue
        sl = slice(int(rows[0]), int(rows[-1]) + 1)
        n = max(1, int(batch["mask"][sl].sum(1).max()))
        part = {}
        for key, v in batch.items():
            v = v[sl]
            if key in ("states", "targets", "mask", "last"):
                v = v[:, :n]
            part[key] = v
        out.append(part)
    return out


def _full_counts(model: WorldModel, batch: dict) -> torch.Tensor:
    """Per-field target counts of a whole batch (the denominators of its loss)."""
    tg = model.end_prefix(batch["targets"], batch["last"])
    masks = model.target_masks(tg, batch["mask"], batch["last"])
    return torch.stack([m.sum() for m in masks]).float()


def _fresh_scaler(scaler: torch.amp.GradScaler, enabled: bool) -> torch.amp.GradScaler:
    """A scaler with the same scale and growth tracker but no per-step state (an OOM between
    unscale_ and update would otherwise make the next unscale_ raise)."""
    new = torch.amp.GradScaler("cuda", enabled=enabled)
    if enabled:
        new.load_state_dict(scaler.state_dict())
    return new


def train_model(
    model: WorldModel,
    train_split: S.Split,
    val_split: S.Split,
    cfg: dict,
    ckpt_path: str,
    deadline_ts: float,
    log: Callable[[str], None] = print,
) -> dict:
    """Train with AdamW, warmup + cosine, gradient clipping and AMP on CUDA.

    One checkpoint file at ckpt_path holds: "model" = best weights so far (by validation NLL),
    "current_model" + optimizer + scaler + RNG states + epoch + step + position inside the
    epoch (to resume exactly), cfg, history. If ckpt_path exists, training resumes from it.
    Validation NLL is evaluated after every epoch; training stops on early stopping
    (patience), at cfg["epochs"], at cfg["stop_after_epoch"], or cleanly at the first batch
    after time.time() > deadline_ts (a mid-epoch stop saves the position and resumes there).
    On out-of-memory the SAME step is retried with the batch split into twice as many
    micro-batches whose gradients are accumulated with the full batch's denominators, so the
    data order, batch composition and schedule do not change. At return the model holds the
    best weights (or, if no epoch finished, the current ones).
    """
    c = merge_cfg({**model.cfg, **cfg})
    device = device_of(model)
    enabled, amp_dtype, use_scaler = amp_settings(device)
    torch.manual_seed(c["seed"])
    opt = torch.optim.AdamW(
        _param_groups(model, c["weight_decay"]), lr=c["lr"], betas=tuple(c["betas"])
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    state = {
        "epoch": 0,
        "step": 0,
        "best_val": float("inf"),
        "best_epoch": -1,
        "bad_epochs": 0,
        "history": [],
        "batch_games": int(c["batch_games"]),
        "micro": 1,  # micro-batches per step (grows on out-of-memory)
        "batch_in_epoch": 0,  # batches of the current epoch already done (mid-epoch resume)
        "epoch_loss_sum": 0.0,
        "epoch_loss_n": 0,
        "finished": False,
    }
    best_weights = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})

    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["current_model"])
        opt.load_state_dict(ck["optimizer"])
        if ck.get("scaler") and use_scaler:
            scaler.load_state_dict(ck["scaler"])
        best_weights = ck["model"]
        for k in state:
            if k in ck:
                state[k] = ck[k]
        torch.set_rng_state(ck["torch_rng"])
        if device.type == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(ck["cuda_rng"])
        log(
            f"resumed from {ckpt_path}: epoch {state['epoch']} batch {state['batch_in_epoch']}, "
            f"step {state['step']}, best val {state['best_val']:.4f}"
        )

    def checkpoint() -> None:
        if not ckpt_path:
            return
        _save(
            ckpt_path,
            {
                "model": best_weights,
                "current_model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "optimizer": opt.state_dict(),
                "scaler": scaler.state_dict() if use_scaler else None,
                "cfg": model.cfg,
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
                **state,
            },
        )

    def train_step(batch: dict) -> float:
        """One optimizer step on batch, split into state["micro"] micro-batches."""
        nonlocal scaler
        while True:
            k = int(state["micro"])
            try:
                opt.zero_grad(set_to_none=True)
                if k == 1:
                    with torch.autocast(device.type, dtype=amp_dtype, enabled=enabled):
                        total, _ = model.loss(batch)
                    scaler.scale(total).backward()
                    value = float(total.detach())
                    del total
                else:
                    denom = _full_counts(model, batch)
                    value = 0.0
                    for part in _micro_slices(batch, k):
                        with torch.autocast(device.type, dtype=amp_dtype, enabled=enabled):
                            part_total, _ = model.loss(part, denom=denom)
                        scaler.scale(part_total).backward()
                        value += float(part_total.detach())
                        del part_total
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), c["grad_clip"])
                scaler.step(opt)
                scaler.update()
                return value
            except RuntimeError as err:  # torch.OutOfMemoryError subclasses RuntimeError
                rows = int(batch["mask"].shape[0])
                if not _is_oom(err) or k >= rows:
                    raise
            # outside the except block, so the traceback (and its activations) is released
            opt.zero_grad(set_to_none=True)
            scaler = _fresh_scaler(scaler, use_scaler)
            if device.type == "cuda":
                torch.cuda.empty_cache()
            state["micro"] = 2 * k
            log(f"out of memory with {k} micro-batch(es) per step; retrying with {2 * k}")

    stopped = "finished" if state["finished"] else "epochs"
    n_games = len(train_split.offsets) - 1
    records_seen, train_time = 0, 0.0
    while not state["finished"] and state["epoch"] < c["epochs"]:
        epoch = state["epoch"]
        if c["stop_after_epoch"] is not None and epoch >= c["stop_after_epoch"]:
            stopped = "stop_after_epoch"
            break
        if time.time() > deadline_ts:
            stopped = "deadline"
            break
        model.train()
        bg = state["batch_games"]
        n_batches = math.ceil(n_games / bg)
        rng = np.random.default_rng([c["seed"], epoch])
        t0 = time.time()
        skip = int(state["batch_in_epoch"])
        hit_deadline = False
        for bi, batch in enumerate(make_batches(train_split, bg, model.max_len, rng)):
            if bi < skip:
                continue  # done before a mid-epoch stop; same permutation, so same batches
            if time.time() > deadline_ts:
                hit_deadline = True
                break
            factor = _lr_factor(state["step"], epoch + bi / n_batches, c)
            for g in opt.param_groups:
                g["lr"] = c["lr"] * factor
            batch = to_device(batch, device)
            value = train_step(batch)
            state["step"] += 1
            state["batch_in_epoch"] = bi + 1
            state["epoch_loss_sum"] += value
            state["epoch_loss_n"] += 1
            records_seen += int(batch["mask"].sum())
        train_time += time.time() - t0
        if hit_deadline:
            stopped = "deadline"
            if state["best_epoch"] < 0:  # no validated epoch yet: keep what was learned
                best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            checkpoint()
            log(f"deadline reached during epoch {epoch}; saved current state")
            break
        val = evaluate_nll(model, val_split, c)
        train_loss = state["epoch_loss_sum"] / max(state["epoch_loss_n"], 1)
        improved = val["total"] < state["best_val"]
        if improved:
            state["best_val"] = val["total"]
            state["best_epoch"] = epoch
            state["bad_epochs"] = 0
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            state["bad_epochs"] += 1
        state["history"].append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_total": val["total"],
                "val_fields": val["fields"],
                "lr": opt.param_groups[0]["lr"],
                "seconds": time.time() - t0,
                "batch_games": bg,
                "micro": int(state["micro"]),
            }
        )
        state["epoch"] = epoch + 1
        state["batch_in_epoch"] = 0
        state["epoch_loss_sum"], state["epoch_loss_n"] = 0.0, 0
        if state["bad_epochs"] >= c["patience"]:
            state["finished"] = True
            stopped = "early_stopping"
        elif state["epoch"] >= c["epochs"]:
            state["finished"] = True
        checkpoint()
        log(
            f"epoch {epoch}: train {train_loss:.4f}  val {val['total']:.4f} nats/record"
            f"{'  (best)' if improved else ''}  {time.time() - t0:.0f}s"
        )
    model.load_state_dict(best_weights)
    return {
        "stopped": stopped,
        "epochs_done": state["epoch"],
        "best_epoch": state["best_epoch"],
        "best_val_nll": state["best_val"],
        "batch_games": state["batch_games"],
        "micro_batches": int(state["micro"]),
        "history": state["history"],
        "train_records_per_s": records_seen / train_time if train_time > 0 else float("nan"),
    }


In [ ]:
%%writefile /content/wm_code/simulate.py
"""Use the trained world model as a game simulator.

A rollout starts from a real record (its game's header and records up to and including it),
optionally with that record's action forced, and samples transitions until the game ends.
Bookkeeping (clock, score, timeouts) is done by `schema.apply_transition`; the model only
samples the transition fields. The model's own learned action distribution plays both teams
for every record after the start, so the result is "what happens if this team makes this call
and then everyone behaves like an average NFL team in the training data".
"""

from __future__ import annotations

import time

import numpy as np
import pandas as pd
import schema as S
import torch
from model import TARGET_NAMES, WorldModel, device_of, merge_cfg, window_start
from train import amp_settings

T = len(S.TARGET_FIELDS)
TI = S.TARGET_INDEX
SI = S.STATE_INDEX
PH = S.PHASE
A = S.ACTION
GO = "GO"

# Which actions may follow which phase; anything else is counted as inconsistent.
_ALLOWED = np.zeros((len(S.PHASES), len(S.ACTIONS)), dtype=bool)
for _a in ("RUN", "PASS", "PUNT", "FG", "KNEEL", "SPIKE", "NOPLAY"):
    _ALLOWED[PH["PLAY"], A[_a]] = True
for _a in ("KICKOFF", "ONSIDE", "NOPLAY"):
    _ALLOWED[PH["KICKOFF"], A[_a]] = True
for _a in ("XP", "TWO_PT", "NOPLAY"):
    _ALLOWED[PH["TRY"], A[_a]] = True

COUNTER_KEYS = (
    "rollouts",
    "transitions",
    "ended_by_end",
    "ended_by_clock",
    "truncated_max_steps",
    "inconsistent",  # transitions with at least one problem below
    "bad_phase_pad",
    "bad_poss_pad",
    "bad_down",  # down not 1..4 on PLAY, or not 0 off PLAY
    "bad_ydstogo",
    "bad_yardline_zero",
    "bad_action_for_phase",
)


def check_transitions(tg: np.ndarray) -> dict:
    """Count inconsistent sampled transitions (rows whose phase is not END)."""
    tg = np.asarray(tg)
    ph = tg[:, TI["phase"]]
    live = ph != PH["END"]
    play = ph == PH["PLAY"]
    down, togo = tg[:, TI["down"]], tg[:, TI["ydstogo"]]
    bad = {
        "bad_phase_pad": live & (ph == PH["PAD"]),
        "bad_poss_pad": live & (tg[:, TI["poss"]] == S.POSS_CODE["PAD"]),
        "bad_down": live & np.where(play, (down < 1) | (down > 4), down != 0),
        "bad_ydstogo": live & np.where(play, togo == 0, togo != 0),
        "bad_yardline_zero": live & (tg[:, TI["yardline"]] == 0),
        "bad_action_for_phase": live & ~_ALLOWED[ph, tg[:, TI["action"]]],
    }
    out = {k: int(v.sum()) for k, v in bad.items()}
    out["inconsistent"] = int(np.logical_or.reduce(list(bad.values())).sum())
    return out


def _torch_generator(rng: np.random.Generator, device: torch.device) -> torch.Generator:
    g = torch.Generator(device=device)
    g.manual_seed(int(rng.integers(0, 2**62)))
    return g


def _game_of(split: S.Split, idx: np.ndarray) -> np.ndarray:
    return np.searchsorted(split.offsets, idx, side="right") - 1


def _window_buffer(split: S.Split, idx: np.ndarray, max_len: int) -> tuple[np.ndarray, np.ndarray]:
    """Context windows ending at each record in idx -> (buf [Q, max_len, S], lens [Q])."""
    games = _game_of(split, idx)
    buf = np.zeros((len(idx), max_len, len(S.STATE_FIELDS)), dtype=np.int64)
    lens = np.zeros(len(idx), dtype=np.int64)
    for q, (g, i) in enumerate(zip(games, idx, strict=True)):
        a = window_start(int(split.offsets[g]), int(i), max_len)
        k = int(i) - a + 1
        buf[q, :k] = split.states[a : i + 1]
        lens[q] = k
    return buf, lens


@torch.no_grad()
def _last_hidden(
    model: WorldModel, header: torch.Tensor, buf: torch.Tensor, lens: np.ndarray, chunk: int
) -> torch.Tensor:
    device = buf.device
    enabled, dtype, _ = amp_settings(device)
    outs = []
    for a in range(0, len(lens), chunk):
        sl = slice(a, a + chunk)
        n = int(lens[sl].max())
        with torch.autocast(device.type, dtype=dtype, enabled=enabled):
            h = model.encode(header[sl], buf[sl, :n])
        ln = torch.as_tensor(lens[sl] - 1, device=device)
        outs.append(h[torch.arange(len(ln), device=device), ln].float())
    return torch.cat(outs)


@torch.no_grad()
def _go_probs(model: WorldModel, split: S.Split, idx: np.ndarray, cfg: dict) -> np.ndarray:
    """P(RUN), P(PASS) for record i, renormalised over the two, from the model's prediction
    of record i's action at record i-1 (given the true other fields of record i)."""
    device = device_of(model)
    out = np.full((len(idx), 2), 0.5)
    games = _game_of(split, idx)
    has_prev = idx > split.offsets[games]
    if not has_prev.any():
        return out
    prev = idx[has_prev] - 1
    buf, lens = _window_buffer(split, prev, model.max_len)
    header = torch.as_tensor(split.header[games[has_prev]].astype(np.int64), device=device)
    h = _last_hidden(model, header, torch.as_tensor(buf, device=device), lens, cfg["sim_batch"])
    prefix = torch.as_tensor(split.targets[prev].astype(np.int64), device=device)
    logits = model.field_logits(h, prefix, TI["action"]).float()
    p = torch.softmax(logits, -1)[:, list(S.GO_ACTIONS)].cpu().numpy().astype(np.float64)
    tot = p.sum(1, keepdims=True)
    ok = tot[:, 0] > 0
    out_prev = np.full_like(p, 0.5)
    out_prev[ok] = p[ok] / tot[ok]
    out[has_prev] = out_prev
    return out


@torch.no_grad()
def _run(
    model: WorldModel,
    header: np.ndarray,
    buf: np.ndarray,
    lens: np.ndarray,
    raw: np.ndarray,
    cfg: dict,
    deadline_ts: float,
    gen: torch.Generator,
    track: bool = False,
) -> dict | None:
    """Roll every row to the end of its game. Returns None if the deadline passes first."""
    device = device_of(model)
    max_len, max_steps, chunk = model.max_len, int(cfg["max_steps"]), int(cfg["sim_batch"])
    temp, mask_pad = float(cfg["temperature"]), bool(cfg["mask_pad_codes"])
    sort_len = bool(cfg.get("sort_by_length", True))
    r = len(lens)
    buf_t = torch.as_tensor(buf, device=device)
    head_t = torch.as_tensor(header.astype(np.int64), device=device)
    lens = lens.astype(np.int64).copy()
    raw = np.asarray(raw, dtype=np.int32).copy()
    active = np.ones(r, dtype=bool)
    steps = np.zeros(r, dtype=np.int64)
    final = raw[:, 1].astype(np.int64).copy()
    appended = np.zeros(r, dtype=np.int64)
    points = np.zeros(r, dtype=np.int64)
    kickoffs = np.zeros(r, dtype=np.int64)
    cnt = dict.fromkeys(COUNTER_KEYS, 0)
    cnt["rollouts"] = r
    enabled, dtype, _ = amp_settings(device)
    while active.any():
        if time.time() > deadline_ts:
            return None
        act = np.flatnonzero(active)
        # Encode in chunks; with sort_by_length the chunks hold rows of similar context
        # length, so less padding is computed. The causal encoder ignores right padding, and
        # sampling below uses the original row order and chunks, so the draws are unchanged.
        enc_order = np.argsort(lens[act], kind="stable") if sort_len else np.arange(len(act))
        hl_all = torch.empty(len(act), model.cfg["d_model"], device=device)
        for a in range(0, len(act), chunk):
            pos_rows = enc_order[a : a + chunk]
            rows = act[pos_rows]
            rows_t = torch.as_tensor(rows, device=device)
            n = int(lens[rows].max())
            with torch.autocast(device.type, dtype=dtype, enabled=enabled):
                h = model.encode(head_t[rows_t], buf_t[rows_t, :n])
            ln = torch.as_tensor(lens[rows] - 1, device=device)
            hl_all[torch.as_tensor(pos_rows, device=device)] = h[
                torch.arange(len(rows), device=device), ln
            ].float()
            del h
        tg_parts = [
            model.sample_transition(hl_all[a : a + chunk], temp, gen, mask_pad)
            for a in range(0, len(act), chunk)
        ]
        tg = torch.cat(tg_parts).cpu().numpy()
        steps[act] += 1
        cnt["transitions"] += len(act)
        for k, v in check_transitions(tg).items():
            cnt[k] += v
        phase = tg[:, TI["phase"]]
        is_end = phase == PH["END"]
        raw_next = S.apply_transition(raw[act], tg)
        delta = S.score_delta_value(tg[:, TI["score_delta"]]).astype(np.int64)
        if not cfg["score_on_end"]:
            delta = np.where(is_end, 0, delta)
            raw_next[is_end, 1] = raw[act[is_end], 1]
        points[act] += np.abs(delta)
        clock0 = ~is_end & (raw_next[:, 0] <= 0) & (phase != PH["TRY"])
        trunc = ~is_end & ~clock0 & (steps[act] >= max_steps)
        cont = ~(is_end | clock0 | trunc)
        cnt["ended_by_end"] += int(is_end.sum())
        cnt["ended_by_clock"] += int(clock0.sum())
        cnt["truncated_max_steps"] += int(trunc.sum())
        final[act] = raw_next[:, 1]
        raw[act] = raw_next
        stop = act[~cont]
        active[stop] = False
        rows = act[cont]
        if len(rows):
            codes = S.next_state_codes(raw_next[cont], tg[cont]).astype(np.int64)
            appended[rows] += 1
            kickoffs[rows] += codes[:, SI["phase"]] == PH["KICKOFF"]
            full = rows[lens[rows] >= max_len]
            if len(full):
                ft = torch.as_tensor(full, device=device)
                buf_t[ft, :-1] = buf_t[ft, 1:].clone()
            pos = np.minimum(lens[rows], max_len - 1)
            buf_t[torch.as_tensor(rows, device=device), torch.as_tensor(pos, device=device)] = (
                torch.as_tensor(codes, device=device)
            )
            lens[rows] = np.minimum(lens[rows] + 1, max_len)
    out = {"final_diff": final, "steps": steps, "counters": cnt}
    if track:
        out.update({"appended": appended, "points": points, "kickoffs": kickoffs})
    return out


def _resolve_action(forced_action) -> int | str | None:
    if forced_action is None or forced_action == GO:
        return forced_action
    if isinstance(forced_action, str):
        return A[forced_action]
    code = int(forced_action)
    if not 0 < code < len(S.ACTIONS):
        raise ValueError(f"action code {code} is outside the action vocabulary")
    return code


def rollout(
    model: WorldModel,
    split: S.Split,
    query_idx,
    forced_action,
    n_rollouts: int,
    cfg: dict | None,
    deadline_ts: float,
    rng: np.random.Generator,
) -> dict:
    """Home win probability by simulation from each query record.

    forced_action: None (keep the real action), an action code or name, or "GO" (record i's
    action is RUN or PASS, sampled from the model's own probabilities renormalised over the two).
    Queries are processed in groups of up to cfg["sim_max_live"] rollouts; if the deadline passes,
    the unfinished queries come back as NaN with done=False.
    """
    c = merge_cfg({**model.cfg, **(cfg or {})})
    model.eval()
    device = device_of(model)
    idx = np.atleast_1d(np.asarray(query_idx, dtype=np.int64))
    nq, n = len(idx), int(n_rollouts)
    fa = _resolve_action(forced_action)
    res = {
        "query_idx": idx,
        "home_wp": np.full(nq, np.nan),
        "home_wp_se": np.full(nq, np.nan),
        "mean_steps": np.full(nq, np.nan),
        "done": np.zeros(nq, dtype=bool),
        "forced_action": forced_action,
        "counters": dict.fromkeys(COUNTER_KEYS, 0),
        "start_actions": {},  # action of the query record, over all rollouts
    }
    gen = _torch_generator(rng, device)
    games = _game_of(split, idx)
    per_group = max(1, int(c["sim_max_live"]) // max(1, n))
    t0 = time.time()
    for a in range(0, nq, per_group):
        if time.time() > deadline_ts:
            break
        q = np.arange(a, min(nq, a + per_group))
        buf, lens = _window_buffer(split, idx[q], model.max_len)
        buf = np.repeat(buf, n, axis=0)
        lens = np.repeat(lens, n)
        if fa == GO:
            p = _go_probs(model, split, idx[q], c)
            u = rng.random(len(q) * n)
            pick = np.where(u < np.repeat(p[:, 0], n), S.GO_ACTIONS[0], S.GO_ACTIONS[1])
            buf[np.arange(len(lens)), lens - 1, SI["action"]] = pick
        elif fa is not None:
            buf[np.arange(len(lens)), lens - 1, SI["action"]] = fa
        starts = buf[np.arange(len(lens)), lens - 1, SI["action"]]
        for code, k in zip(*np.unique(starts, return_counts=True), strict=True):
            name = _action_name(code)
            res["start_actions"][name] = res["start_actions"].get(name, 0) + int(k)
        header = np.repeat(split.header[games[q]], n, axis=0)
        raw = np.repeat(split.raw[idx[q]], n, axis=0)
        out = _run(model, header, buf, lens, raw, c, deadline_ts, gen)
        if out is None:
            break
        o = S.outcome(out["final_diff"]).reshape(len(q), n)
        res["home_wp"][q] = o.mean(1)
        res["home_wp_se"][q] = o.std(1, ddof=1) / np.sqrt(n) if n > 1 else np.nan
        res["mean_steps"][q] = out["steps"].reshape(len(q), n).mean(1)
        res["done"][q] = True
        for k, v in out["counters"].items():
            res["counters"][k] += v
    res["seconds"] = time.time() - t0
    return res


def _action_name(code: int) -> str:
    return S.ACTIONS[int(code)]


def _choice_of(code: int) -> str:
    if int(code) in S.GO_ACTIONS:
        return GO
    return _action_name(code)


def plan_fourth_downs(
    model: WorldModel,
    split: S.Split,
    idx,
    n_rollouts: int,
    cfg: dict | None,
    deadline_ts: float,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """For every fourth-down PLAY record in idx: win probability of GO, FG and PUNT for the
    team with the ball, standard errors, best action, margin and a "too close" flag (margin
    under 2 standard errors of the difference between the best and second-best action)."""
    idx = np.atleast_1d(np.asarray(idx, dtype=np.int64))
    st = split.states[idx]
    keep = (st[:, SI["down"]] == 4) & (st[:, SI["phase"]] == PH["PLAY"])
    idx = idx[keep]
    poss = split.states[idx, SI["poss"]]
    home_has_ball = poss == S.POSS_CODE["HOME"]
    choices = (GO, "FG", "PUNT")
    games = _game_of(split, idx)
    df = pd.DataFrame(
        {
            "idx": idx,
            "game_id": [split.game_ids[g] for g in games],
            "season": split.seasons[games],
            "poss": np.where(home_has_ball, "HOME", "AWAY"),
            "ydstogo": split.states[idx, SI["ydstogo"]],
            "yardline_100": split.states[idx, SI["yardline"]],
            "clock_seconds": split.raw[idx, 0],
            "score_diff_poss": np.where(home_has_ball, 1, -1) * split.raw[idx, 1],
        }
    )
    # The three actions are simulated chunk by chunk of states (one rollout group each), so
    # a deadline leaves whole states finished rather than GO for all and PUNT for none.
    c = merge_cfg({**model.cfg, **(cfg or {})})
    per = max(1, int(c["sim_max_live"]) // max(1, int(n_rollouts)))
    counters = {ch: dict.fromkeys(COUNTER_KEYS, 0) for ch in choices}
    for ch in choices:
        key = ch.lower()
        df[f"wp_{key}"] = np.nan
        df[f"se_{key}"] = np.nan
        df[f"done_{key}"] = False
    for a in range(0, len(idx), per):
        sl = slice(a, min(len(idx), a + per))
        for ch in choices:
            r = rollout(model, split, idx[sl], ch, n_rollouts, cfg, deadline_ts, rng)
            key = ch.lower()
            hb = home_has_ball[sl]
            df.loc[df.index[sl], f"wp_{key}"] = np.where(hb, r["home_wp"], 1.0 - r["home_wp"])
            df.loc[df.index[sl], f"se_{key}"] = r["home_wp_se"]
            df.loc[df.index[sl], f"done_{key}"] = r["done"]
            for k, v in r["counters"].items():
                counters[ch][k] += v
    wps = df[[f"wp_{c.lower()}" for c in choices]].to_numpy()
    ses = df[[f"se_{c.lower()}" for c in choices]].to_numpy()
    done = df[[f"done_{c.lower()}" for c in choices]].to_numpy().all(1)
    order = np.argsort(-np.nan_to_num(wps, nan=-1.0), axis=1)
    rows = np.arange(len(df))
    b, s2 = order[:, 0], order[:, 1] if len(choices) > 1 else order[:, 0]
    margin = wps[rows, b] - wps[rows, s2]
    margin_se = np.sqrt(ses[rows, b] ** 2 + ses[rows, s2] ** 2)
    df["best"] = np.where(done, np.array(choices, dtype=object)[b], None)
    df["margin"] = np.where(done, margin, np.nan)
    df["margin_se"] = np.where(done, margin_se, np.nan)
    df["too_close"] = done & (margin < 2 * margin_se)
    real = split.states[idx, SI["action"]]
    df["real_action"] = [_action_name(a) for a in real]
    df["real_choice"] = [_choice_of(a) for a in real]
    df["agrees"] = done & (df["best"] == df["real_choice"])
    df["done"] = done
    df.attrs["counters"] = counters
    return df


def wp_by_simulation(
    model: WorldModel,
    split: S.Split,
    idx,
    n_rollouts: int,
    cfg: dict | None,
    deadline_ts: float,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """Simulated home win probability from each record with its real action, plus the actual
    home result (1/0.5/0) for calibration."""
    idx = np.atleast_1d(np.asarray(idx, dtype=np.int64))
    r = rollout(model, split, idx, None, n_rollouts, cfg, deadline_ts, rng)
    games = _game_of(split, idx)
    df = pd.DataFrame(
        {
            "idx": idx,
            "game_id": [split.game_ids[g] for g in games],
            "season": split.seasons[games],
            "phase": [S.PHASES[p] for p in split.states[idx, SI["phase"]]],
            "poss": [S.POSS[p] for p in split.states[idx, SI["poss"]]],
            "clock_seconds": split.raw[idx, 0],
            "score_diff": split.raw[idx, 1],
            "home_wp_sim": r["home_wp"],
            "home_wp_se": r["home_wp_se"],
            "mean_steps": r["mean_steps"],
            "home_result": S.outcome(split.final_diff[games]),
            "done": r["done"],
        }
    )
    df.attrs["counters"] = r["counters"]
    return df


def real_game_stats(split: S.Split, games: np.ndarray | None = None) -> dict:
    """League statistics of real games, defined the same way as for simulated games."""
    g_all = np.arange(len(split.offsets) - 1) if games is None else np.asarray(games)
    n_rec = np.diff(split.offsets)[g_all]
    pts_rec = np.abs(S.score_delta_value(split.targets[:, TI["score_delta"]]))
    kick_rec = split.states[:, SI["phase"]] == PH["KICKOFF"]
    seg = split.offsets[:-1]
    pts = np.add.reduceat(pts_rec, seg)[g_all] if len(pts_rec) else np.zeros(len(g_all))
    kicks = np.add.reduceat(kick_rec.astype(np.int64), seg)[g_all]
    fd = split.final_diff[g_all].astype(np.int64)
    last = split.offsets[1:][g_all] - 1
    chain_diff = split.raw[last, 1].astype(np.int64)  # score before the final transition
    return {
        "records_per_game": float(n_rec.mean()),
        "points_per_game": float(pts.mean()),  # all transitions that carry a score delta
        "final_margin_sd": float(fd.std()),
        "tie_share": float((fd == 0).mean()),
        "kickoffs_per_game": float(kicks.mean()),
        "home_win_share": float(S.outcome(fd).mean()),
        "final_play_points_share": float((fd != chain_diff).mean()),
    }


def simulate_games(
    model: WorldModel,
    split: S.Split,
    n_games: int,
    cfg: dict | None,
    deadline_ts: float,
    rng: np.random.Generator,
) -> dict:
    """Simulate whole games from the header and opening record of sampled real games and
    compare league statistics with the same real games."""
    c = merge_cfg({**model.cfg, **(cfg or {})})
    model.eval()
    device = device_of(model)
    n_real = len(split.offsets) - 1
    games = rng.choice(n_real, size=n_games, replace=n_games > n_real)
    starts = split.offsets[games]
    gen = _torch_generator(rng, device)
    per = int(c["sim_max_live"])
    parts = []
    for a in range(0, n_games, per):
        g = games[a : a + per]
        s = starts[a : a + per]
        buf, lens = _window_buffer(split, s, model.max_len)
        out = _run(model, split.header[g], buf, lens, split.raw[s], c, deadline_ts, gen, track=True)
        if out is None:
            break
        parts.append(out)
    if not parts:
        return {"n_games": 0, "sim": {}, "real": real_game_stats(split, games), "counters": {}}
    cat = {k: np.concatenate([p[k] for p in parts]) for k in ("final_diff", "appended", "points")}
    cat["kickoffs"] = np.concatenate([p["kickoffs"] for p in parts])
    done = len(cat["final_diff"])
    first_kick = (split.states[starts[:done], SI["phase"]] == PH["KICKOFF"]).astype(np.int64)
    fd = cat["final_diff"]
    sim = {
        "records_per_game": float((cat["appended"] + 1).mean()),
        "points_per_game": float(cat["points"].mean()),
        "final_margin_sd": float(fd.std()),
        "tie_share": float((fd == 0).mean()),
        "kickoffs_per_game": float((cat["kickoffs"] + first_kick).mean()),
        "home_win_share": float(S.outcome(fd).mean()),
    }
    counters = dict.fromkeys(COUNTER_KEYS, 0)
    for p in parts:
        for k, v in p["counters"].items():
            counters[k] += v
    real = real_game_stats(split, games[:done])
    table = pd.DataFrame({"real": pd.Series(real), "simulated": pd.Series(sim)}, index=list(real))
    return {"n_games": done, "sim": sim, "real": real, "table": table, "counters": counters}


__all__ = [
    "COUNTER_KEYS",
    "GO",
    "TARGET_NAMES",
    "check_transitions",
    "plan_fourth_downs",
    "real_game_stats",
    "rollout",
    "simulate_games",
    "wp_by_simulation",
]


In [ ]:
%%writefile /content/wm_code/toy.py
"""A tiny synthetic `Split` for tests: short, rule-based games whose codes and transitions are
valid under schema.py and consistent with `schema.apply_transition`. Not football-realistic.
Conventions follow the real records: a kickoff record's poss is the RECEIVING team, and the
last record's target is END plus the final play's score_delta and elapsed."""

from __future__ import annotations

import numpy as np
import schema as S

PH, A, P = S.PHASE, S.ACTION, S.POSS_CODE


def _policy(phase: int, down: int, togo: int, yl: int, rng: np.random.Generator) -> int:
    if phase == PH["KICKOFF"]:
        return A["KICKOFF"]
    if phase == PH["TRY"]:
        return A["XP"] if rng.random() < 0.9 else A["TWO_PT"]
    if down == 4:
        if rng.random() < 0.2 or (togo <= 2 and yl < 60):
            return A["RUN"] if rng.random() < 0.5 else A["PASS"]
        return A["FG"] if yl <= 37 else A["PUNT"]
    return A["RUN"] if rng.random() < 0.45 else A["PASS"]


def _step(phase, poss, down, togo, yl, action, rng):
    """Next (phase, poss, down, togo, yardline), points for the team with the ball, elapsed."""
    other = 3 - poss
    if phase == PH["KICKOFF"]:  # poss of a kickoff record = receiving team
        return (PH["PLAY"], poss, 1, 10, 75), 0, int(rng.integers(4, 9))
    if phase == PH["TRY"]:
        pts = (
            (1 if rng.random() < 0.94 else 0)
            if action == A["XP"]
            else (2 if rng.random() < 0.48 else 0)
        )
        return (PH["KICKOFF"], other, 0, 0, 65), pts, 0
    el = int(rng.integers(5, 41))
    if action == A["FG"]:
        if rng.random() < max(0.2, 1.0 - yl / 50):
            return (PH["KICKOFF"], other, 0, 0, 65), 3, 5
        return (PH["PLAY"], other, 1, 10, int(np.clip(100 - (yl + 7), 1, 99))), 0, 5
    if action == A["PUNT"]:
        land = yl - 40
        yl2 = 80 if land <= 0 else int(np.clip(100 - land, 1, 99))
        return (PH["PLAY"], other, 1, 10, yl2), 0, 8
    gain = int(rng.integers(-3, 16))
    if gain >= yl:
        return (PH["TRY"], poss, 0, 0, 15), 6, el
    yl2 = int(np.clip(yl - gain, 1, 99))
    if gain >= togo:
        return (PH["PLAY"], poss, 1, min(10, yl2), yl2), 0, el
    if down == 4:
        return (PH["PLAY"], other, 1, 10, 100 - yl2), 0, el
    return (PH["PLAY"], poss, down + 1, int(np.clip(togo - gain, 1, S.MAX_YDSTOGO)), yl2), 0, el


def make_toy_split(
    n_games: int, rng: np.random.Generator, start_clock: int = 600, first_game_id: int = 0
) -> S.Split:
    """n_games short games (regulation clock starts at start_clock seconds)."""
    states, raws, targets, offsets, header, final = [], [], [], [0], [], []
    for _ in range(n_games):
        receiver = int(rng.integers(1, 3))
        header.append([S.spread_code(rng.integers(-7, 8)), receiver])
        cat = (PH["KICKOFF"], receiver, 0, 0, 65)
        raw = np.array([start_clock, 0, 3, 3], dtype=np.int32)
        action = A["KICKOFF"]
        while True:
            phase, poss, down, togo, yl = cat
            nxt, pts, el = _step(phase, poss, down, togo, yl, action, rng)
            sign = 1 if poss == P["HOME"] else -1
            used_h = int(rng.random() < 0.03 and raw[2] > 0)
            used_a = int(rng.random() < 0.03 and raw[3] > 0)
            n_action = _policy(nxt[0], nxt[2], nxt[3], nxt[4], rng)
            tg = np.zeros(len(S.TARGET_FIELDS), dtype=np.int16)
            for name, v in zip(("phase", "poss", "down", "ydstogo", "yardline"), nxt, strict=True):
                tg[S.TARGET_INDEX[name]] = v
            tg[S.TARGET_INDEX["score_delta"]] = S.SCORE_DELTA_CODE[sign * pts]
            tg[S.TARGET_INDEX["elapsed"]] = S.elapsed_code(el)
            tg[S.TARGET_INDEX["to_used_home"]] = used_h + 1
            tg[S.TARGET_INDEX["to_used_away"]] = used_a + 1
            tg[S.TARGET_INDEX["action"]] = n_action
            raw_next = S.apply_transition(raw[None], tg[None])[0]
            end = raw_next[0] == 0 and nxt[0] != PH["TRY"]
            rec = np.zeros(len(S.STATE_FIELDS), dtype=np.int16)
            for name, v in zip(("phase", "poss", "down", "ydstogo", "yardline"), cat, strict=True):
                rec[S.STATE_INDEX[name]] = v
            rec[S.STATE_INDEX["clock"]] = S.clock_code(raw[0])
            rec[S.STATE_INDEX["score"]] = S.score_code(raw[1])
            rec[S.STATE_INDEX["to_home"]] = S.timeout_code(raw[2])
            rec[S.STATE_INDEX["to_away"]] = S.timeout_code(raw[3])
            rec[S.STATE_INDEX["action"]] = action
            states.append(rec)
            raws.append(raw.copy())
            if end:
                last_tg = np.zeros_like(tg)
                last_tg[S.TARGET_INDEX["phase"]] = PH["END"]
                for name in ("score_delta", "elapsed"):
                    last_tg[S.TARGET_INDEX[name]] = tg[S.TARGET_INDEX[name]]
                targets.append(last_tg)
                final.append(int(raw_next[1]))  # includes the final play's points
                break
            targets.append(tg)
            cat, raw, action = nxt, raw_next, n_action
        offsets.append(len(states))
    n = len(states)
    return S.Split(
        states=np.asarray(states, dtype=np.int16),
        raw=np.asarray(raws, dtype=np.int32),
        targets=np.asarray(targets, dtype=np.int16),
        offsets=np.asarray(offsets, dtype=np.int64),
        header=np.asarray(header, dtype=np.int16),
        final_diff=np.asarray(final, dtype=np.int16),
        game_ids=[f"toy_{first_game_id + g}" for g in range(n_games)],
        seasons=np.full(n_games, 2000, dtype=np.int16),
        play_ids=np.arange(n, dtype=np.int64),
    )


In [ ]:
%%writefile /content/wm_code/run.py
"""Runner for the NFL world-model experiment, built to finish inside one Colab GPU session.

Six stages, each idempotent. A finished stage writes a done-marker holding its config hash,
so re-running everything skips finished work and training resumes from its checkpoint:

  1 data     download play-by-play, build and validate train/validation/test splits
  2 train    fit the world model (checkpointed, deadline-bound)
  3 eval     validation only: per-field NLL vs a frequency baseline, simulated league
             stats vs real, win probability by simulation vs nflverse on the same states
  4 test     write prereg.json (config hash and the exact plan) FIRST, then score the
             test seasons once; a second scoring is refused unless allow_rescore is set
  5 plan     fourth-down recommendations on sampled test fourth downs, one constructed
             hypothetical state, and agreement with the real coaches
  6 package  results.json (every number), figures, a zip to download

A global deadline guard gives every stage a deadline; stages scale down (fewer states or
rollouts) or are skipped with a recorded reason, and packaging always runs, so a run
always ends with a results file. Numbers from nflverse (`home_wp`, `vegas_home_wp`) are
kept in their own labelled fields and never blended with ours.
"""

from __future__ import annotations

import copy
import hashlib
import importlib
import json
import os
import platform
import sys
import time
import traceback
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import schema as S

STEPS_PER_ROLLOUT = 90  # mean transitions from a random state to the end of regulation
MAX_SEASON = 2025  # never download a season after this one (2026 is in progress)
SEED = 20261118
STAGES = ("data", "train", "eval", "test", "plan", "package")
UPSTREAM = {
    "data": (),
    "train": ("data",),
    "eval": ("train",),
    "test": ("train",),
    "plan": ("train",),
    "package": (),
}
# Module sources recorded per stage (metadata and warnings only; hashes never use them,
# except that a changed records/schema makes the data stage rebuild in memory and compare).
STAGE_SOURCES = {
    "data": ["schema", "records"],
    "train": ["model", "train"],
    "eval": ["simulate"],
    "test": ["simulate"],
    "plan": ["simulate"],
    "package": [],
}
HERE = Path(__file__).resolve().parent


# ---------------------------------------------------------------- presets
_FULL_SEASONS = {
    "train": list(range(2010, 2022)),
    "val": [2022, 2023],
    "test": [2024, 2025],
}


def _gpu_preset(name: str, model: dict, train: dict, eval_: dict, plan: dict, assumed: dict):
    return {
        "name": name,
        "device": "cuda",
        "seasons": copy.deepcopy(_FULL_SEASONS),
        "model": model,
        "train": train,
        "eval": eval_,
        "test": dict(eval_),
        "plan": plan,
        "budget_min": 55.0,
        # Per-stage ceilings in minutes. They sum to more than the budget on purpose: a
        # stage that finishes early hands its time on; the global deadline is the hard cap.
        "stage_min": {"data": 8, "train": 20, "eval": 9, "test": 7, "plan": 8, "package": 2},
        # Minimum seconds a stage needs to be worth starting; below it, it is skipped.
        "min_start_s": {"data": 60, "train": 60, "eval": 45, "test": 45, "plan": 45,
                        "package": 0},
        "reserve_package_s": 90,
        "assumed": assumed,
    }


PRESETS: dict[str, dict] = {
    "smoke": {
        "name": "smoke",
        "device": "cpu",
        "seasons": {"train": [2021], "val": [2022], "test": [2023]},
        "model": {"d_model": 64, "n_layers": 2, "n_heads": 4, "d_ff": 256, "dec_hidden": 64,
                  "max_len": 256, "dropout": 0.1},
        "train": {"batch_games": 16, "lr": 1e-3, "weight_decay": 0.01, "epochs": 4,
                  "patience": 2, "warmup_steps": 50, "max_train_games": None,
                  "sim_batch": 256, "sim_max_live": 1024, "score_on_end": True},
        "eval": {"wp_states": 16, "wp_rollouts": 16, "sim_games": 4, "bootstrap": 200,
                 "nll_max_games": 60},
        "test": {"wp_states": 16, "wp_rollouts": 16, "sim_games": 0, "bootstrap": 200,
                 "nll_max_games": 60},
        "plan": {"n_fourth": 8, "n_rollouts": 16, "hypo_rollouts": 64},

        "budget_min": 12.0,
        "stage_min": {"data": 5, "train": 2.5, "eval": 2, "test": 1.5, "plan": 1.5,
                      "package": 1},
        "min_start_s": {"data": 20, "train": 10, "eval": 10, "test": 10, "plan": 10,
                        "package": 0},
        "reserve_package_s": 20,
        "assumed": {"train_rec_per_s": 3_000, "rollout_units_per_s": 2_000},
    },
    "t4": _gpu_preset(
        "t4",
        model={"d_model": 256, "n_layers": 6, "n_heads": 8, "d_ff": 1024, "dec_hidden": 256,
               "max_len": 256, "dropout": 0.1},
        train={"batch_games": 32, "lr": 6e-4, "weight_decay": 0.1, "epochs": 40,
               "patience": 4, "warmup_steps": 300, "max_train_games": None,
               "sim_batch": 1024, "sim_max_live": 8192, "score_on_end": True},
        eval_={"wp_states": 600, "wp_rollouts": 64, "sim_games": 128, "bootstrap": 1000,
               "nll_max_games": None},
        plan={"n_fourth": 150, "n_rollouts": 128, "hypo_rollouts": 1024},
        assumed={"train_rec_per_s": 60_000, "rollout_units_per_s": 40_000},
    ),
    "l4": _gpu_preset(
        "l4",
        model={"d_model": 320, "n_layers": 6, "n_heads": 8, "d_ff": 1280, "dec_hidden": 320,
               "max_len": 256, "dropout": 0.1},
        train={"batch_games": 48, "lr": 6e-4, "weight_decay": 0.1, "epochs": 50,
               "patience": 4, "warmup_steps": 300, "max_train_games": None,
               "sim_batch": 2048, "sim_max_live": 16384, "score_on_end": True},
        eval_={"wp_states": 1000, "wp_rollouts": 96, "sim_games": 256, "bootstrap": 1000,
               "nll_max_games": None},
        plan={"n_fourth": 250, "n_rollouts": 192, "hypo_rollouts": 2048},
        assumed={"train_rec_per_s": 120_000, "rollout_units_per_s": 80_000},
    ),
    "a100": _gpu_preset(
        "a100",
        model={"d_model": 384, "n_layers": 8, "n_heads": 8, "d_ff": 1536, "dec_hidden": 384,
               "max_len": 256, "dropout": 0.1},
        train={"batch_games": 64, "lr": 5e-4, "weight_decay": 0.1, "epochs": 60,
               "patience": 5, "warmup_steps": 300, "max_train_games": None,
               "sim_batch": 4096, "sim_max_live": 32768, "score_on_end": True},

        eval_={"wp_states": 1500, "wp_rollouts": 128, "sim_games": 512, "bootstrap": 1000,
               "nll_max_games": None},
        plan={"n_fourth": 400, "n_rollouts": 256, "hypo_rollouts": 4096},
        assumed={"train_rec_per_s": 300_000, "rollout_units_per_s": 200_000},
    ),
}
DOWNGRADE = {"a100": "l4", "l4": "t4", "t4": "t4", "smoke": "smoke"}


def pick_preset(device_name: str | None = None, mem_gb: float | None = None) -> str:
    """Preset key from the CUDA device name and memory; "smoke" when there is no GPU."""
    if device_name is None:
        try:
            import torch

            if not torch.cuda.is_available():
                return "smoke"
            props = torch.cuda.get_device_properties(0)
            device_name, mem_gb = props.name, props.total_memory / 2**30
        except Exception:
            return "smoke"
    name = device_name.upper()
    mem = mem_gb or 0.0
    if "A100" in name or "H100" in name or "H200" in name or "B200" in name:
        return "a100"
    if "L4" in name or "A10" in name or "L40" in name or "A6000" in name:
        return "l4"
    if "T4" in name or "P100" in name or "V100" in name:
        return "t4"
    if mem >= 38:
        return "a100"
    if mem >= 20:
        return "l4"
    return "t4"


def get_preset(name: str, **overrides: Any) -> dict:
    p = copy.deepcopy(PRESETS[name])
    for k, v in overrides.items():
        if isinstance(v, dict) and isinstance(p.get(k), dict):
            p[k].update(v)
        else:
            p[k] = v
    return p


def check_seasons(seasons) -> list[int]:
    seasons = [int(s) for s in seasons]
    assert seasons, "no seasons given"
    assert max(seasons) <= MAX_SEASON, f"refusing to download a season after {MAX_SEASON}"
    assert min(seasons) >= 1999, "nflverse play-by-play starts in 1999"
    return seasons


# ---------------------------------------------------------------- storage
def default_workdir() -> Path:
    if os.environ.get("WM_WORKDIR"):
        return Path(os.environ["WM_WORKDIR"])
    drive = Path("/content/drive/MyDrive")
    if drive.is_dir():
        return drive / "fourth_and_rl_world_model"
    if Path("/content").is_dir():
        return Path("/content/wm_run")
    return HERE.parent.parent / "results" / "world_model_local"


def _jsonable(x: Any) -> Any:
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return _jsonable(x.tolist())
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        v = float(x)
        return v if np.isfinite(v) else None
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if hasattr(x, "to_dict") and callable(x.to_dict):  # pandas objects
        try:
            return _jsonable(x.to_dict(orient="list"))
        except TypeError:
            return _jsonable(x.to_dict())
    if isinstance(x, Path):
        return str(x)
    if x is None or isinstance(x, (str, int, bool)):
        return x
    return str(x)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(_jsonable(obj), indent=2))
    tmp.replace(path)


def read_json(path: Path) -> Any:
    return json.loads(Path(path).read_text())


def config_hash(obj: Any) -> str:
    return hashlib.sha256(json.dumps(_jsonable(obj), sort_keys=True).encode()).hexdigest()[:16]


def _source_hash(module_names) -> str:
    h = hashlib.sha256()
    for m in module_names:
        p = HERE / f"{m}.py"
        mod = sys.modules.get(m)
        if mod is not None and getattr(mod, "__file__", None):
            p = Path(mod.__file__)
        h.update(p.read_bytes() if p.exists() else m.encode())
    return h.hexdigest()[:16]


def _mod(name: str):
    """Import a sibling module lazily, so tests can put stubs in sys.modules."""
    return sys.modules.get(name) or importlib.import_module(name)


# ---------------------------------------------------------------- split utilities
_SPLIT_ARRAYS = ("states", "raw", "targets", "offsets", "header", "final_diff", "seasons",
                 "play_ids")


def save_split(path: Path, split: S.Split, **extra: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {k: np.asarray(getattr(split, k)) for k in _SPLIT_ARRAYS}
    arrays["game_ids"] = np.asarray(list(split.game_ids), dtype=str)
    arrays.update({f"extra_{k}": np.asarray(v) for k, v in extra.items()})
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    tmp.replace(path)


def load_split(path: Path) -> tuple[S.Split, dict[str, np.ndarray]]:
    with np.load(path, allow_pickle=False) as z:
        kw = {k: z[k] for k in _SPLIT_ARRAYS}
        kw["game_ids"] = [str(g) for g in z["game_ids"]]
        extra = {k[6:]: z[k] for k in z.files if k.startswith("extra_")}
    return S.Split(**kw), extra


def concat_splits(splits: list[S.Split]) -> S.Split:
    offs, base = [np.zeros(1, dtype=np.int64)], 0
    for s in splits:
        offs.append(np.asarray(s.offsets[1:], dtype=np.int64) + base)
        base += int(s.offsets[-1])
    return S.Split(
        states=np.concatenate([s.states for s in splits]),
        raw=np.concatenate([s.raw for s in splits]),
        targets=np.concatenate([s.targets for s in splits]),
        offsets=np.concatenate(offs),
        header=np.concatenate([s.header for s in splits]),
        final_diff=np.concatenate([s.final_diff for s in splits]),
        game_ids=[g for s in splits for g in s.game_ids],
        seasons=np.concatenate([s.seasons for s in splits]),
        play_ids=np.concatenate([s.play_ids for s in splits]),
    )


def subset_split(split: S.Split, games) -> S.Split:
    """A split holding only the listed games (in the given order)."""
    games = np.asarray(games, dtype=np.int64)
    rec = [np.arange(split.offsets[g], split.offsets[g + 1]) for g in games]
    lens = np.array([len(r) for r in rec], dtype=np.int64)
    idx = np.concatenate(rec) if rec else np.zeros(0, dtype=np.int64)
    return S.Split(
        states=split.states[idx],
        raw=split.raw[idx],
        targets=split.targets[idx],
        offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
        header=split.header[games],
        final_diff=split.final_diff[games],
        game_ids=[split.game_ids[g] for g in games],
        seasons=split.seasons[games],
        play_ids=split.play_ids[idx],
    )


def record_game(split: S.Split) -> np.ndarray:
    """Game index of every record."""
    return np.repeat(np.arange(len(split.game_ids)), np.diff(split.offsets))


def n_records(split: S.Split) -> int:
    return int(split.offsets[-1])


def synthetic_split(n_games: int = 8, seed: int = 0) -> S.Split:
    """A schema-valid toy split (drives in which the clock runs down), for tests and the
    micro-benchmark only. It is not football data."""
    rng = np.random.default_rng(seed)
    states, raws, targets, offsets, final = [], [], [], [0], []
    for _g in range(n_games):
        clock, diff, poss = S.GAME_SECONDS, 0, 1
        rows, raw = [], []
        down, togo, yl = 0, 0, 65
        phase = S.PHASE["KICKOFF"]
        while clock > 0:
            act = S.ACTION["KICKOFF"] if phase == S.PHASE["KICKOFF"] else int(
                rng.choice([S.ACTION["RUN"], S.ACTION["PASS"]]))
            if phase == S.PHASE["PLAY"] and down == 4:
                act = S.ACTION["PUNT"]
            rows.append([phase, poss, down, togo, yl, int(S.clock_code(clock)),
                         int(S.score_code(diff)), 4, 4, act])
            raw.append([clock, diff, 3, 3])
            clock = max(clock - int(rng.integers(5, 45)), 0)
            if phase == S.PHASE["KICKOFF"] or act == S.ACTION["PUNT"]:
                poss = 3 - poss
                phase, down, togo, yl = S.PHASE["PLAY"], 1, 10, int(rng.integers(50, 80))
            else:
                yl = max(1, yl - int(rng.integers(0, 8)))
                down = down + 1 if down < 4 else 1
                togo = 10
                if yl <= 2:
                    diff += 7 if poss == 1 else -7
                    phase, down, togo, yl = S.PHASE["KICKOFF"], 0, 0, 65
                    poss = 3 - poss
        states.extend(rows)
        raws.extend(raw)
        tg = np.zeros((len(rows), len(S.TARGET_FIELDS)), dtype=np.int16)
        r = np.asarray(rows)
        for name in ("phase", "poss", "down", "ydstogo", "yardline", "action"):
            tg[:-1, S.TARGET_INDEX[name]] = r[1:, S.STATE_INDEX[name]]
        tg[:-1, S.TARGET_INDEX["score_delta"]] = [
            S.SCORE_DELTA_CODE.get(int(b[1] - a[1]), S.SCORE_DELTA_CODE[0])
            for a, b in zip(raw[:-1], raw[1:], strict=True)]
        tg[:-1, S.TARGET_INDEX["elapsed"]] = S.elapsed_code(
            np.asarray(raw)[:-1, 0] - np.asarray(raw)[1:, 0])
        tg[:-1, S.TARGET_INDEX["to_used_home"]] = 1
        tg[:-1, S.TARGET_INDEX["to_used_away"]] = 1
        tg[-1, S.TARGET_INDEX["phase"]] = S.PHASE["END"]
        targets.append(tg)
        offsets.append(offsets[-1] + len(rows))
        final.append(diff)
    n = offsets[-1]
    return S.Split(
        states=np.asarray(states, dtype=np.int16),
        raw=np.asarray(raws, dtype=np.int32),
        targets=np.concatenate(targets),
        offsets=np.asarray(offsets, dtype=np.int64),
        header=np.tile(np.array([[S.SPREAD_CLIP + 1, 1]], dtype=np.int16), (n_games, 1)),
        final_diff=np.asarray(final, dtype=np.int16),
        game_ids=[f"SYN_{g:04d}" for g in range(n_games)],
        seasons=np.full(n_games, 2000, dtype=np.int16),
        play_ids=np.arange(n, dtype=np.int64),
    )


# ---------------------------------------------------------------- run context
class Run:
    """Everything a stage needs: preset, folders, the deadline guard and the log.

    Stage hashes are keyed on CONFIG and DATA CONTENT, never on source code: the train
    checkpoint and done-marker depend on the model and train config plus a content hash of
    the saved train and validation arrays, so an edited notebook resumes training as long as
    the data and config are the same. Module source hashes are recorded in each done-marker
    as metadata and produce a warning when they change.
    """

    def __init__(self, preset: dict, workdir: Path | None = None,
                 budget_min: float | None = None, allow_rescore: bool = False,
                 seed: int = SEED, quiet: bool = False, force: tuple = ()) -> None:
        self.preset = preset
        self.workdir = Path(workdir or default_workdir())
        self.workdir.mkdir(parents=True, exist_ok=True)
        for sub in ("data", "stages", "outputs", "figures", "ckpt", "partial"):
            (self.workdir / sub).mkdir(exist_ok=True)
        self.allow_rescore = allow_rescore
        self.force = set(force)
        self.seed = seed
        self.quiet = quiet
        self.session_start = time.time()
        self.ledger_path = self.workdir / "ledger.json"
        ledger = read_json(self.ledger_path) if self.ledger_path.exists() else {}
        self.consumed_before = float(ledger.get("consumed_s", 0.0))
        total = 60.0 * float(budget_min if budget_min is not None else preset["budget_min"])
        self.total_s = total
        left = total - self.consumed_before
        # Never less than the packaging reserve: packaging must always be able to run.
        self.deadline = self.session_start + max(left, preset["reserve_package_s"] + 5)
        self._models: dict[str, Any] = {}

    # time ------------------------------------------------------------
    def remaining(self) -> float:
        return self.deadline - time.time()

    def stage_deadline(self, stage: str) -> float:
        ceiling = time.time() + 60.0 * self.preset["stage_min"][stage]
        reserve = 0.0 if stage == "package" else float(self.preset["reserve_package_s"])
        return min(ceiling, self.deadline - reserve)

    def add_consumed(self, seconds: float, stage: str) -> None:
        ledger = read_json(self.ledger_path) if self.ledger_path.exists() else {}
        ledger["consumed_s"] = float(ledger.get("consumed_s", 0.0)) + float(seconds)
        ledger.setdefault("stages", []).append(
            {"stage": stage, "seconds": round(float(seconds), 1), "at": time.time()})
        write_json(self.ledger_path, ledger)

    def log(self, *msg: Any) -> None:
        line = " ".join(str(m) for m in msg)
        with open(self.workdir / "run.log", "a") as f:
            f.write(f"{time.strftime('%H:%M:%S')} {line}\n")
        if not self.quiet:
            print(line, flush=True)

    # config ----------------------------------------------------------
    def model_cfg(self) -> dict:
        """The cfg handed to model, train and simulate."""
        p = self.preset
        cfg = {**p["model"], **p["train"], "device": self.device(), "seed": self.seed}
        cfg.pop("max_train_games", None)
        return cfg

    def device(self) -> str:
        if self.preset["device"] == "cuda":
            try:
                import torch

                if torch.cuda.is_available():
                    return "cuda"
            except Exception:
                pass
        return "cpu"

    def content_hash(self, name: str) -> str:
        """Content hash of data/<name>.npz, cached by file size and mtime."""
        path = self.workdir / "data" / f"{name}.npz"
        if not path.exists():
            return "missing"
        cache_path = self.workdir / "data" / "content_hashes.json"
        cache = read_json(cache_path) if cache_path.exists() else {}
        st = path.stat()
        key = f"{st.st_size}:{st.st_mtime_ns}"
        if cache.get(name, {}).get("key") == key:
            return cache[name]["hash"]
        sp, extra = load_split(path)
        h = split_arrays_hash(sp, extra)
        cache[name] = {"key": key, "hash": h}
        write_json(cache_path, cache)
        return h

    def stage_src(self, stage: str) -> str:
        return _source_hash(STAGE_SOURCES[stage])

    def stage_hash(self, stage: str) -> str:
        p = self.preset
        parts: dict[str, Any] = {"stage": stage, "seed": self.seed}
        if stage == "data":
            parts["seasons"] = p["seasons"]
        elif stage == "train":
            parts.update(model=p["model"], train=p["train"],
                         data_train=self.content_hash("train"),
                         data_val=self.content_hash("val"))
        elif stage in ("eval", "test", "plan"):
            parts[stage] = p[stage]
            parts["up_train"] = self.stage_hash("train")
            if stage in ("test", "plan"):
                parts["data_test"] = self.content_hash("test")
        return config_hash(parts)

    def full_config_hash(self) -> str:
        return config_hash({k: self.stage_hash(k) for k in STAGES if k != "package"})

    # files -------------------------------------------------------------
    def marker(self, stage: str) -> Path:
        return self.workdir / "stages" / f"{stage}.done"

    def output_path(self, stage: str) -> Path:
        return self.workdir / "outputs" / f"{stage}.json"

    def partial_dir(self, stage: str) -> Path:
        d = self.workdir / "partial" / f"{stage}_{self.stage_hash(stage)}"
        d.mkdir(parents=True, exist_ok=True)
        return d

    def is_done(self, stage: str) -> bool:
        if stage in self.force:
            return False
        m = self.marker(stage)
        if not m.exists():
            return False
        mk = read_json(m)
        if mk.get("hash") != self.stage_hash(stage):
            return False
        # Data is the one stage whose code is re-checked: a changed records.py is rebuilt
        # in memory and compared (see stage_data).
        return not (stage == "data" and mk.get("src") != self.stage_src("data"))

    def status(self, stage: str) -> str:
        p = self.output_path(stage)
        return read_json(p).get("status", "missing") if p.exists() else "missing"

    def split(self, name: str) -> tuple[S.Split, dict]:
        return load_split(self.workdir / "data" / f"{name}.npz")

    def ckpt_path(self) -> Path:
        return self.workdir / "ckpt" / f"model_{self.stage_hash('train')}.pt"

    def load_model(self):
        key = self.stage_hash("train")
        if key in self._models:
            return self._models[key]
        model, _ck = _mod("train").load_checkpoint(str(self.ckpt_path()), device=self.device())
        model.eval()
        self._models[key] = model
        return model


def run_stage(run: Run, stage: str, fn=None) -> dict:
    """Run one stage under the deadline guard. Finished stages are skipped by hash.

    A stage may return status "partial" (it ran out of time but saved its progress); it gets
    no done-marker, so the next run continues it. A KeyboardInterrupt is recorded as
    "interrupted" and re-raised after the stage's output file is written.
    """
    fn = fn or STAGE_FUNCS[stage]
    h = run.stage_hash(stage)
    if stage != "package" and run.is_done(stage):
        out = read_json(run.output_path(stage))
        mk = read_json(run.marker(stage))
        run.log(f"[{stage}] already done (hash {h}); skipping")
        if mk.get("src") and mk["src"] != run.stage_src(stage):
            run.log(f"[{stage}] note: {', '.join(STAGE_SOURCES[stage])} changed since this "
                    "stage finished; its results are kept (stages are keyed on config and "
                    f"data). Pass force=('{stage}',) to recompute.")
        return out
    for up in UPSTREAM[stage]:
        if not run.is_done(up):
            out = {"status": "skipped", "reason": f"upstream stage '{up}' has not finished",
                   "hash": h}
            write_json(run.output_path(stage), out)
            run.log(f"[{stage}] skipped: {out['reason']}")
            return out
    available = run.stage_deadline(stage) - time.time()
    if available < run.preset["min_start_s"][stage]:
        out = {"status": "skipped", "hash": h,
               "reason": f"time budget: {available:.0f}s left for this stage, needs "
                         f"{run.preset['min_start_s'][stage]}s"}
        write_json(run.output_path(stage), out)
        run.log(f"[{stage}] skipped: {out['reason']}")
        return out
    run.log(f"[{stage}] starting ({available / 60:.1f} min available, "
            f"{run.remaining() / 60:.1f} min left in the run)")
    t0 = time.time()
    interrupted = False
    try:
        out = fn(run, run.stage_deadline(stage))
        out.setdefault("status", "done")
    except KeyboardInterrupt:
        interrupted = True
        out = {"status": "interrupted",
               "reason": "stopped by the user; saved progress is resumed on the next run"}
    except Exception as e:  # recorded, not swallowed silently: packaging reports it
        out = {"status": "error", "error": f"{type(e).__name__}: {e}",
               "traceback": traceback.format_exc()}
        run.log(f"[{stage}] ERROR {out['error']}\n{out['traceback']}")
    out["seconds"] = round(time.time() - t0, 1)
    out["hash"] = h
    out["src"] = run.stage_src(stage)
    write_json(run.output_path(stage), out)
    if stage != "package":
        run.add_consumed(out["seconds"], stage)
    if out["status"] == "done" and stage != "package":
        # Re-hash: the data stage changes the content hashes the other stages depend on.
        write_json(run.marker(stage), {"hash": run.stage_hash(stage), "src": out["src"],
                                       "finished": time.time()})
    run.log(f"[{stage}] {out['status']} in {out['seconds']:.0f}s")
    if interrupted:
        raise KeyboardInterrupt
    return out


# ---------------------------------------------------------------- stage 1: data
def _pbp_wp_columns(pbp) -> Any:
    """game_id, play_id, home_wp, vegas_home_wp from a polars or pandas frame, as pandas."""
    cols = [c for c in ("game_id", "play_id", "home_wp", "vegas_home_wp") if c in pbp.columns]
    if hasattr(pbp, "to_pandas"):
        df = pbp.select(cols).to_pandas()
    else:
        df = pbp[cols].copy()
    for c in ("home_wp", "vegas_home_wp"):
        if c not in df.columns:
            df[c] = np.nan
    return df


def align_nflverse_wp(split: S.Split, wp_df) -> dict[str, np.ndarray]:
    """nflverse home_wp and vegas_home_wp aligned to every record (NaN when missing)."""
    import pandas as pd

    rg = record_game(split)
    keys = pd.DataFrame({
        "game_id": np.asarray(split.game_ids, dtype=object)[rg],
        "play_id": np.asarray(split.play_ids, dtype=np.float64),
    })
    wp = wp_df.copy()
    wp["play_id"] = wp["play_id"].astype(np.float64)
    wp = wp.drop_duplicates(["game_id", "play_id"])
    merged = keys.merge(wp, on=["game_id", "play_id"], how="left")
    return {
        "nflverse_home_wp": merged["home_wp"].to_numpy(dtype=np.float64),
        "nflverse_vegas_home_wp": merged["vegas_home_wp"].to_numpy(dtype=np.float64),
    }


def stage_data(run: Run, deadline: float) -> dict:
    """Per season: reuse the saved arrays when they exist. If records.py or schema.py
    changed since they were built, rebuild the season in memory and keep the saved file
    when the arrays are identical (a cosmetic change); replace it when they differ, which
    changes the data content hash and so restarts training."""
    rec = _mod("records")
    p = run.preset
    ddir = run.workdir / "data"
    src_now = run.stage_src("data")
    out: dict[str, Any] = {"splits": {}, "source_hash": src_now, "seasons_rebuilt": {}}
    built: dict[str, S.Split] = {}
    for split_name in ("train", "val", "test"):
        seasons = check_seasons(p["seasons"][split_name])
        parts, reports, wp_parts = [], {}, []
        for season in seasons:
            cache = ddir / f"season_{season}.npz"
            rep_path = ddir / f"season_{season}_report.json"
            meta_path = ddir / f"season_{season}_meta.json"
            meta = read_json(meta_path) if meta_path.exists() else {}
            if time.time() > deadline and not cache.exists():
                raise TimeoutError(f"data stage ran out of time before season {season}")
            if cache.exists() and rep_path.exists():
                sp, extra = load_split(cache)
                report = read_json(rep_path)
                if meta.get("src") != src_now:
                    run.log(f"[data] records/schema code changed: rebuilding {season} in "
                            "memory to compare")
                    new_sp, new_report, new_extra = _build_season(rec, season)
                    same = split_arrays_hash(new_sp, new_extra) == split_arrays_hash(sp, extra)
                    out["seasons_rebuilt"][season] = "identical, kept" if same else "replaced"
                    if not same:
                        sp, report, extra = new_sp, new_report, new_extra
                        save_split(cache, sp, **extra)
                        write_json(rep_path, report)
                    write_json(meta_path, {"src": src_now})
            else:
                run.log(f"[data] downloading and building season {season}")
                sp, report, extra = _build_season(rec, season)
                save_split(cache, sp, **extra)
                write_json(rep_path, report)
                write_json(meta_path, {"src": src_now})
            reports[season] = _jsonable(report)
            parts.append(sp)
            wp_parts.append(extra)
        split = concat_splits(parts)
        extra = {k: np.concatenate([w[k] for w in wp_parts]) for k in wp_parts[0]}
        validation = rec.validate_split(split)
        built[split_name] = split
        target = ddir / f"{split_name}.npz"
        new_hash = split_arrays_hash(split, extra)
        if not target.exists() or run.content_hash(split_name) != new_hash:
            save_split(target, split, **extra)  # unchanged arrays keep their file (and mtime)
        write_json(ddir / f"{split_name}_validation.json", validation)
        write_json(ddir / f"{split_name}_build_report.json", reports)
        out["splits"][split_name] = {
            "seasons": seasons,
            "games": len(split.game_ids),
            "records": n_records(split),
            "content_hash": run.content_hash(split_name),
            "nflverse_wp_coverage": float(np.mean(np.isfinite(extra["nflverse_home_wp"]))),
            "validation": _jsonable(validation),
            "build_report": reports,
        }
        run.log(f"[data] {split_name}: {len(split.game_ids)} games, "
                f"{n_records(split)} records, seasons {seasons}")
    if hasattr(rec, "validate_disjoint"):
        out["disjoint"] = _jsonable(rec.validate_disjoint(built))
    return out



# ---------------------------------------------------------------- frequency baseline
def target_mask(split: S.Split) -> np.ndarray:
    """[N, T] bool: which target fields are scored, the same rule as model.WorldModel.loss.
    Phase is always scored. Down and ydstogo (where 0 means "not a down") are scored on
    every record except a game's last; every other field is scored where its code is not
    padding (0)."""
    t = np.asarray(split.targets)
    last = np.zeros(len(t), dtype=bool)
    last[np.asarray(split.offsets[1:]) - 1] = True
    m = t != 0
    for name in ("down", "ydstogo"):
        m[:, S.TARGET_INDEX[name]] = ~last
    m[:, S.TARGET_INDEX["phase"]] = True
    return m



def fit_frequency_baseline(split: S.Split, alpha: float = 0.5) -> list[np.ndarray]:
    """Per target field, P(field | current phase, current down) with add-alpha smoothing.
    Pad code 0 gets no mass except for fields where 0 is a real value (down, ydstogo)."""
    st, tg = split.states, split.targets
    ph = st[:, S.STATE_INDEX["phase"]].astype(np.int64)
    dn = st[:, S.STATE_INDEX["down"]].astype(np.int64)
    mask = target_mask(split)
    tables = []
    for f, (name, size) in enumerate(S.TARGET_FIELDS):
        counts = np.zeros((len(S.PHASES), 5, size))
        m = mask[:, f]
        np.add.at(counts, (ph[m], dn[m], tg[m, f].astype(np.int64)), 1.0)
        prior = np.full(size, alpha)
        if name not in ("down", "ydstogo"):
            prior[0] = 0.0
        counts += prior
        tables.append(counts / counts.sum(axis=2, keepdims=True))
    return tables


def baseline_nll(tables: list[np.ndarray], split: S.Split) -> dict:
    st, tg = split.states, split.targets
    ph = st[:, S.STATE_INDEX["phase"]].astype(np.int64)
    dn = st[:, S.STATE_INDEX["down"]].astype(np.int64)
    mask = target_mask(split)
    per_field, total, n_rows = {}, 0.0, int(mask[:, 0].sum())
    for f, (name, _size) in enumerate(S.TARGET_FIELDS):
        m = mask[:, f]
        p = tables[f][ph[m], dn[m], tg[m, f].astype(np.int64)]
        nll = -np.log(np.maximum(p, 1e-12))
        per_field[name] = float(nll.mean()) if m.any() else None
        total += float(nll.sum())
    return {"per_field": per_field, "total_per_transition": total / max(n_rows, 1),
            "n_transitions": n_rows}


def _per_field(d: dict) -> dict:
    src = d.get("fields", d.get("per_field", d)) if isinstance(d, dict) else {}
    return {name: float(src[name]) for name, _ in S.TARGET_FIELDS
            if name in src and src[name] is not None}


def nll_comparison(run: Run, split: S.Split, cfg_key: str) -> dict:
    """Model per-field NLL (via train.evaluate_nll) next to the frequency baseline."""
    train_mod = _mod("train")
    tr, _ = run.split("train")
    tables = fit_frequency_baseline(tr)
    max_games = run.preset[cfg_key].get("nll_max_games")
    if max_games and len(split.game_ids) > max_games:
        split = subset_split(split, np.arange(max_games))
    base = baseline_nll(tables, split)
    model_d = train_mod.evaluate_nll(run.load_model(), split, run.model_cfg())
    model_pf = _per_field(model_d)
    return {
        "games": len(split.game_ids),
        "model": _jsonable(model_d),
        "baseline_freq_phase_down": base,
        "per_field": {name: {"model": model_pf.get(name),
                             "baseline": base["per_field"].get(name)}
                      for name, _ in S.TARGET_FIELDS},
        "note": "Model per-field NLL is conditional on the fields sampled before it in the "
                "same transition; the baseline is the marginal given phase and down. The "
                "totals per transition are the comparable numbers.",
    }


# ---------------------------------------------------------------- league stats
def real_league_stats(split: S.Split) -> dict:
    st, tg = split.states, split.targets
    g = len(split.game_ids)
    phase = st[:, S.STATE_INDEX["phase"]]
    down = st[:, S.STATE_INDEX["down"]]
    act = st[:, S.STATE_INDEX["action"]]
    pts = np.abs(S.score_delta_value(tg[:, S.TARGET_INDEX["score_delta"]]))
    fourth = (phase == S.PHASE["PLAY"]) & (down == 4)
    go = np.isin(act, S.GO_ACTIONS)
    fd = fourth & (go | (act == S.ACTION["PUNT"]) | (act == S.ACTION["FG"]))
    final = np.asarray(split.final_diff, dtype=float)
    return {
        "games": g,
        "plays_per_game": float((phase == S.PHASE["PLAY"]).sum() / g),
        "points_per_game_both_teams": float(pts.sum() / g),
        "touchdowns_per_game": float(np.isin(pts, (6, 7, 8)).sum() / g),
        "field_goals_per_game": float((pts == 3).sum() / g),
        "punts_per_game": float((act == S.ACTION["PUNT"]).sum() / g),
        "fourth_down_go_rate": float((fd & go).sum() / max(fd.sum(), 1)),
        "fourth_down_punt_rate": float((fd & (act == S.ACTION["PUNT"])).sum() / max(fd.sum(), 1)),
        "fourth_down_fg_rate": float((fd & (act == S.ACTION["FG"])).sum() / max(fd.sum(), 1)),
        "pass_rate": float((act == S.ACTION["PASS"]).sum()
                           / max(np.isin(act, S.GO_ACTIONS).sum(), 1)),
        "home_win_rate_regulation": float(np.mean(S.outcome(final))),
        "regulation_tie_rate": float(np.mean(final == 0)),
        "mean_abs_final_margin": float(np.mean(np.abs(final))),
    }


# ---------------------------------------------------------------- WP calibration
def log_loss(p: np.ndarray, y: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def brier(p: np.ndarray, y: np.ndarray) -> np.ndarray:
    return (np.asarray(p, dtype=float) - np.asarray(y, dtype=float)) ** 2


def game_bootstrap(values: np.ndarray, groups: np.ndarray, n_boot: int,
                   rng: np.random.Generator) -> dict:
    """Mean of per-state values with a 95% interval from resampling whole games."""
    values = np.asarray(values, dtype=float)
    uniq, inv = np.unique(groups, return_inverse=True)
    sums = np.bincount(inv, weights=values, minlength=len(uniq))
    cnts = np.bincount(inv, minlength=len(uniq)).astype(float)
    draws = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    boot = sums[draws].sum(axis=1) / np.maximum(cnts[draws].sum(axis=1), 1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"mean": float(values.mean()), "lo95": float(lo), "hi95": float(hi),
            "n_states": int(len(values)), "n_games": int(len(uniq))}


def reliability(p: np.ndarray, y: np.ndarray, n_bins: int = 10) -> dict:
    p, y = np.asarray(p, dtype=float), np.asarray(y, dtype=float)
    b = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    out = {"bin_mean_pred": [], "bin_mean_outcome": [], "bin_count": []}
    for k in range(n_bins):
        m = b == k
        out["bin_mean_pred"].append(float(p[m].mean()) if m.any() else None)
        out["bin_mean_outcome"].append(float(y[m].mean()) if m.any() else None)
        out["bin_count"].append(int(m.sum()))
    return out


def sample_wp_states(split: S.Split, extra: dict, n: int, rng: np.random.Generator) -> np.ndarray:
    """Random PLAY records that also have both nflverse probabilities, so all three
    estimates are scored on exactly the same states."""
    phase = split.states[:, S.STATE_INDEX["phase"]]
    play = phase == S.PHASE["PLAY"]
    ok = play.copy()
    for k in ("nflverse_home_wp", "nflverse_vegas_home_wp"):
        if k in extra:
            ok &= np.isfinite(extra[k])
    cand = np.flatnonzero(ok)
    if len(cand) == 0:  # nflverse missing everywhere: score ours alone, flagged later
        cand = np.flatnonzero(play)
    n = min(n, len(cand))
    return np.sort(rng.choice(cand, size=n, replace=False)) if n else cand[:0]


def wp_calibration(run: Run, split: S.Split, extra: dict, cfg_key: str, deadline: float,
                   seed: int, pdir: Path) -> dict:
    """Win probability by simulation on a fixed random sample of PLAY records, scored next
    to nflverse's home_wp and vegas_home_wp on the same states (theirs, labelled)."""
    sim = _mod("simulate")
    c = run.preset[cfg_key]
    n_states, n_roll = int(c["wp_states"]), int(c["wp_rollouts"])
    idx = sample_wp_states(split, extra, n_states, np.random.default_rng(seed))
    model, cfg = run.load_model(), run.model_cfg()

    def fn(part, rng):
        return sim.wp_by_simulation(model, split, part, n_roll, cfg, deadline, rng)

    df, info = chunked(run, pdir, fn, idx, _chunk_size(run, n_roll, cfg_key), deadline, seed)
    rng = np.random.default_rng(seed + 1)
    used = idx[: len(df)]
    col = "home_wp_sim" if "home_wp_sim" in df.columns else "sim_wp"
    if len(df) and "idx" in df.columns:
        assert np.array_equal(df["idx"].to_numpy(), used), "simulate returned rows out of order"
    ok = np.isfinite(df[col].to_numpy(dtype=float)) if len(df) else np.zeros(0, dtype=bool)
    if "done" in df.columns:
        ok &= df["done"].to_numpy(dtype=bool)
    n_unfinished = int((~ok).sum())
    df, used = df[ok], used[ok]
    base = {"n_states": int(len(used)), "n_unfinished_dropped": n_unfinished,
            "n_rollouts_per_state": n_roll, "progress": info,
            "complete": info["complete"]}
    if len(used) == 0:
        return {**base, "metrics": {}, "paired_differences": {},
                "note": "no finished rollouts inside the time budget"}
    y = S.outcome(np.asarray(split.final_diff)[record_game(split)[used]])
    games = record_game(split)[used]
    preds = {"ours_simulation": df[col].to_numpy(dtype=float)}
    if "nflverse_home_wp" in extra:
        theirs = {"nflverse_home_wp (theirs)": extra["nflverse_home_wp"][used],
                  "nflverse_vegas_home_wp (theirs)": extra["nflverse_vegas_home_wp"][used]}
        if all(np.isfinite(v).all() for v in theirs.values()):
            preds.update(theirs)
    n_boot = int(c["bootstrap"])
    metrics, diffs = {}, {}
    for k, p in preds.items():
        metrics[k] = {
            "log_loss": game_bootstrap(log_loss(p, y), games, n_boot, rng),
            "brier": game_bootstrap(brier(p, y), games, n_boot, rng),
            "reliability": reliability(p, y),
        }
    for k, p in preds.items():
        if k == "ours_simulation":
            continue
        diffs[f"ours minus {k}"] = {
            "log_loss": game_bootstrap(log_loss(preds["ours_simulation"], y) - log_loss(p, y),
                                       games, n_boot, rng),
            "brier": game_bootstrap(brier(preds["ours_simulation"], y) - brier(p, y),
                                    games, n_boot, rng),
        }
    label_check = None
    lab = "home_result" if "home_result" in df.columns else "actual"
    if lab in df.columns:
        label_check = float(np.mean(np.isclose(df[lab].to_numpy(dtype=float), y)))
    return {
        **base,
        "mean_steps_per_rollout": float(df["mean_steps"].mean())
        if "mean_steps" in df.columns and len(df) else None,
        "outcome_label": "home result at end of regulation: 1 win, 0.5 tie, 0 loss",
        "simulate_actual_matches_label": label_check,
        "metrics": metrics,
        "paired_differences": diffs,
        "source_note": "nflverse home_wp and vegas_home_wp are nflverse's published "
                       "estimates on the same plays, not ours.",
    }


# ---------------------------------------------------------------- stage 2: train
def stage_train(run: Run, deadline: float) -> dict:
    import torch

    model_mod, train_mod = _mod("model"), _mod("train")
    torch.manual_seed(run.seed)
    np.random.seed(run.seed % 2**32)
    tr, _ = run.split("train")
    va, _ = run.split("val")
    mg = run.preset["train"].get("max_train_games")
    if mg and len(tr.game_ids) > mg:
        tr = subset_split(tr, np.arange(mg))
        va = subset_split(va, np.arange(min(len(va.game_ids), max(mg // 4, 8))))
    cfg = run.model_cfg()
    model = model_mod.WorldModel(cfg)
    ck_path = run.ckpt_path()
    fresh_reason = _check_checkpoint(ck_path, model)
    if fresh_reason:
        stale = ck_path.with_name(ck_path.name + f".stale_{int(time.time())}")
        ck_path.replace(stale)
        run.log(f"[train] checkpoint does not fit the current model ({fresh_reason}); "
                f"moved it to {stale.name} and starting fresh")
    progress = run.workdir / "outputs" / "train_progress.json"
    prev = read_json(progress) if progress.exists() else {}
    if prev.get("ckpt") != str(ck_path) or fresh_reason:
        prev = {"seconds": 0.0, "lines": [], "ckpt": str(ck_path)}
    budget_s = 60.0 * run.preset["stage_min"]["train"] - float(prev.get("seconds", 0.0))
    deadline = min(deadline, time.time() + max(budget_s, run.preset["min_start_s"]["train"]))
    t0, base = time.time(), float(prev.get("seconds", 0.0))

    def log(*msg: Any) -> None:
        # Keep the training time used current, so a disconnect mid-training does not hand
        # the resumed run a fresh training budget.
        prev["seconds"] = base + time.time() - t0
        prev["lines"] = (prev.get("lines", []) + [" ".join(str(m) for m in msg)])[-200:]
        write_json(progress, prev)
        run.log("[train]", *msg)

    run.log(f"[train] {model.count_params():,} parameters; {len(tr.game_ids)} train games, "
            f"{n_records(tr):,} records; device {cfg['device']}; checkpoint {ck_path}")
    history = train_mod.train_model(model, tr, va, cfg, str(ck_path), deadline, log)
    if not ck_path.exists():
        raise RuntimeError("train_model returned without writing a checkpoint")
    return {
        "params": int(model.count_params()),
        "train_games": len(tr.game_ids),
        "train_records": n_records(tr),
        "val_games_for_early_stopping": len(va.game_ids),
        "history": _jsonable(history),
        "train_seconds_total": round(base + time.time() - t0, 1),
        "checkpoint": str(ck_path),
        "fresh_start_reason": fresh_reason,
    }


# ---------------------------------------------------------------- stage 3: eval (val)
def stage_eval(run: Run, deadline: float) -> dict:
    sim = _mod("simulate")
    pdir = run.partial_dir("eval")
    va, extra = run.split("val")
    out: dict[str, Any] = {"split": "validation", "seasons": run.preset["seasons"]["val"]}
    out["nll"] = _cached_json(pdir / "nll.json", lambda: nll_comparison(run, va, "eval"))
    run.log(f"[eval] NLL per transition: model "
            f"{_total_nll(out['nll']['model'])}, baseline "
            f"{out['nll']['baseline_freq_phase_down']['total_per_transition']:.3f}")
    out["league_real"] = real_league_stats(va)
    n_games = int(run.preset["eval"]["sim_games"])
    # Simulated games get at most a third of the stage; the rest goes to WP calibration.
    sim_deadline = time.time() + (deadline - time.time()) / 3
    league_path = pdir / "league_sim.json"
    if n_games <= 0:
        out["league_sim"] = {"skipped": "sim_games is 0 in this preset"}
    elif league_path.exists():
        out["league_sim"] = read_json(league_path)
    else:
        res = sim.simulate_games(run.load_model(), va, n_games, run.model_cfg(), sim_deadline,
                                 np.random.default_rng(run.seed + 30))
        res.pop("table", None)  # the same numbers as "sim" and "real"
        out["league_sim"] = _jsonable(res)
        if res.get("n_games", 0) >= n_games:
            write_json(league_path, out["league_sim"])
        run.log(f"[eval] league stats over {res.get('n_games')} simulated games: "
                f"sim {res.get('sim')} real {res.get('real')}")
    out["wp"] = wp_calibration(run, va, extra, "eval", deadline, run.seed + 3, pdir / "wp")
    _log_wp(run, "eval", out["wp"])
    if not out["wp"]["complete"]:
        out["status"] = "partial"
        out["reason"] = (f"time budget: {out['wp']['n_states']} of "
                         f"{out['wp']['progress']['planned_states']} states; the next run "
                         "continues from the saved chunks")
    return out


def _total_nll(d: dict) -> str:
    for k in ("total", "total_per_transition", "nll", "mean"):
        if isinstance(d, dict) and isinstance(d.get(k), (int, float)):
            return f"{d[k]:.3f}"
    return "?"


def _log_wp(run: Run, stage: str, wp: dict) -> None:
    for k, m in wp.get("metrics", {}).items():
        ll, br = m["log_loss"], m["brier"]
        run.log(f"[{stage}] {k}: log loss {ll['mean']:.4f} [{ll['lo95']:.4f}, "
                f"{ll['hi95']:.4f}], Brier {br['mean']:.4f} on {ll['n_states']} states")


# ---------------------------------------------------------------- stage 4: prereg + test
def test_plan(run: Run) -> dict:
    p = run.preset
    return {
        "config_hash": run.full_config_hash(),
        "train_hash": run.stage_hash("train"),
        "test_hash": run.stage_hash("test"),
        "test_seasons": p["seasons"]["test"],
        "seed": run.seed + 4,
        "metrics": [
            "per-field and total NLL per transition, model vs frequency baseline fitted on "
            "the training split (P(field | phase, down), add-0.5 smoothing)",
            f"win probability by simulation on {p['test']['wp_states']} random PLAY records "
            f"with both nflverse probabilities present, {p['test']['wp_rollouts']} rollouts "
            "each; log loss and Brier with a 95% game-bootstrap interval "
            f"({p['test']['bootstrap']} resamples) for ours, nflverse home_wp and nflverse "
            "vegas_home_wp on the same states, and paired differences",
            f"fourth-down plan on {p['plan']['n_fourth']} sampled test fourth downs, "
            f"{p['plan']['n_rollouts']} rollouts per action; agreement of the recommended "
            "action with the real coach on decisions not flagged too_close",
        ],
        "outcome_label": "home result at end of regulation: 1 win, 0.5 tie, 0 loss",
        "rule": "test seasons are scored once; a second scoring needs allow_rescore=True "
                "and is recorded as such",
    }


def stage_test(run: Run, deadline: float) -> dict:
    prereg_path = run.workdir / "prereg.json"
    scored_path = run.workdir / "outputs" / "test_scored.json"
    plan = test_plan(run)
    prereg = read_json(prereg_path) if prereg_path.exists() else None
    rescore = False
    if scored_path.exists():
        scored = read_json(scored_path)
        rescore = scored.get("test_hash") != plan["test_hash"]
        if rescore and not run.allow_rescore:
            raise PermissionError(
                f"test seasons were already scored (test hash {scored.get('test_hash')}); "
                "refusing a second scoring with a different configuration. Set "
                "allow_rescore=True to override; the rescore is recorded in prereg.json.")
    if prereg is None or prereg.get("test_hash") != plan["test_hash"]:
        prereg = {**plan, "written_at": time.time(), "attempts": [], "previous": prereg}
    prereg["attempts"].append({"started_at": time.time(), "rescore": rescore})
    write_json(prereg_path, prereg)  # written BEFORE any test data is loaded
    run.log(f"[test] preregistered plan written to {prereg_path}")

    pdir = run.partial_dir("test")
    te, extra = run.split("test")
    out: dict[str, Any] = {"split": "test", "seasons": run.preset["seasons"]["test"],
                           "prereg": str(prereg_path), "config_hash": plan["config_hash"],
                           "test_hash": plan["test_hash"]}
    out["nll"] = _cached_json(pdir / "nll.json", lambda: nll_comparison(run, te, "test"))
    out["league_real"] = real_league_stats(te)
    out["wp"] = wp_calibration(run, te, extra, "test", deadline, run.seed + 4, pdir / "wp")
    _log_wp(run, "test", out["wp"])
    if out["wp"]["complete"]:
        write_json(scored_path, {"config_hash": plan["config_hash"],
                                 "test_hash": plan["test_hash"], "at": time.time()})
    else:
        out["status"] = "partial"
        out["reason"] = "time budget; the next run continues the same preregistered plan"
    return out


# ---------------------------------------------------------------- stage 5: plan
HYPOTHETICAL = {
    "description": "offense up 4, 4:00 left in the 4th quarter, 4th and 2 at its own 40",
    "lead_for_offense": 4,
    "clock_seconds": 240,
    "down": 4,
    "ydstogo": 2,
    "yardline_100": 60,
}


def build_hypothetical(split: S.Split) -> tuple[S.Split, dict]:
    """Edit the codes of a real late-game record into the hypothetical state.

    The base record is the first PLAY record, in game order of the split, with 3:20 to
    5:00 left in the game and a real down. Its game is cut after that record; score,
    clock, down, distance and yard line are overwritten (score from the offense's side),
    while possession, timeouts, the pregame spread and the earlier records stay as they
    were in the real game. The record's own action field is left as recorded; the planner
    replaces it with each candidate action.
    """
    st, raw = split.states, split.raw
    phase = st[:, S.STATE_INDEX["phase"]]
    down = st[:, S.STATE_INDEX["down"]]
    poss = st[:, S.STATE_INDEX["poss"]]
    clock = raw[:, 0]
    cand = np.flatnonzero((phase == S.PHASE["PLAY"]) & (down >= 1) & (clock >= 200)
                          & (clock <= 300) & (poss > 0))
    if len(cand) == 0:
        raise ValueError("no late-game PLAY record to edit")
    i = int(cand[0])
    g = int(record_game(split)[i])
    sub = subset_split(split, [g])
    j = i - int(split.offsets[g])
    sub = S.Split(
        states=sub.states[: j + 1].copy(), raw=sub.raw[: j + 1].copy(),
        targets=sub.targets[: j + 1].copy(),
        offsets=np.array([0, j + 1], dtype=np.int64), header=sub.header.copy(),
        final_diff=sub.final_diff.copy(), game_ids=[sub.game_ids[0] + "_HYPOTHETICAL"],
        seasons=sub.seasons.copy(), play_ids=sub.play_ids[: j + 1].copy())
    h = HYPOTHETICAL
    home_has_ball = int(sub.states[j, S.STATE_INDEX["poss"]]) == S.POSS_CODE["HOME"]
    diff = h["lead_for_offense"] if home_has_ball else -h["lead_for_offense"]
    before = {name: int(sub.states[j, S.STATE_INDEX[name]]) for name, _ in S.STATE_FIELDS}
    raw_before = [int(v) for v in sub.raw[j]]
    sub.raw[j, 0] = h["clock_seconds"]
    sub.raw[j, 1] = diff
    sub.states[j, S.STATE_INDEX["clock"]] = S.clock_code(h["clock_seconds"])
    sub.states[j, S.STATE_INDEX["score"]] = S.score_code(diff)
    sub.states[j, S.STATE_INDEX["down"]] = h["down"]
    sub.states[j, S.STATE_INDEX["ydstogo"]] = h["ydstogo"]
    sub.states[j, S.STATE_INDEX["yardline"]] = h["yardline_100"]
    sub.targets[j] = 0
    sub.targets[j, S.TARGET_INDEX["phase"]] = S.PHASE["END"]
    doc = {
        **h,
        "base_game_id": split.game_ids[g],
        "base_play_id": int(split.play_ids[i]),
        "base_record_index": i,
        "base_codes_before_edit": before,
        "base_raw_before_edit": dict(zip(S.RAW_COLUMNS, raw_before, strict=True)),
        "codes_after_edit": {name: int(sub.states[j, S.STATE_INDEX[name]])
                             for name, _ in S.STATE_FIELDS},
        "raw_after_edit": dict(zip(S.RAW_COLUMNS, [int(v) for v in sub.raw[j]], strict=True)),
        "offense": "HOME" if home_has_ball else "AWAY",
        "kept_from_real_game": "possession, timeouts, pregame spread, opening receiver, "
                               "all earlier records",
        "limitation": "earlier records are the real game's, so the history does not lead "
                      "exactly to the edited score and clock",
    }
    return sub, doc


def _agreement(df) -> dict:
    if df is None or len(df) == 0 or "best" not in df.columns:
        return {"n": 0}
    n_unfinished = 0
    if "done" in df.columns:
        n_unfinished = int((~df["done"].astype(bool)).sum())
        df = df[df["done"].astype(bool)]
    if len(df) == 0:
        return {"n": 0, "n_unfinished_dropped": n_unfinished}
    col = "real_choice" if "real_choice" in df.columns else "real_action"
    real = df[col].astype(str).str.upper()
    best = df["best"].astype(str).str.upper()
    close = df["too_close"].astype(bool) if "too_close" in df.columns else best.isna()
    agree = real == best
    confusion: dict[str, dict[str, int]] = {}
    for b, r in zip(best, real, strict=True):
        confusion.setdefault(b, {}).setdefault(r, 0)
        confusion[b][r] += 1
    return {
        "n": int(len(df)),
        "n_unfinished_dropped": n_unfinished,
        "real_column": col,
        "n_clear": int((~close).sum()),
        "agreement_all": float(agree.mean()),
        "agreement_clear": float(agree[~close].mean()) if (~close).any() else None,
        "recommended_counts": best.value_counts().to_dict(),
        "real_counts": real.value_counts().to_dict(),
        "confusion_recommended_by_real": confusion,
    }


def stage_plan(run: Run, deadline: float) -> dict:
    rec, sim = _mod("records"), _mod("simulate")
    pdir = run.partial_dir("plan")
    te, _ = run.split("test")
    p = run.preset["plan"]
    model, cfg = run.load_model(), run.model_cfg()
    out: dict[str, Any] = {"split": "test", "seasons": run.preset["seasons"]["test"]}
    # The hypothetical first (one state, many rollouts), with up to a quarter of the time.
    hyp_split, doc = build_hypothetical(te)
    hyp_deadline = time.time() + (deadline - time.time()) / 4
    hyp_idx = np.array([n_records(hyp_split) - 1])

    def hyp_fn(part, rng):
        return sim.plan_fourth_downs(model, hyp_split, part, int(p["hypo_rollouts"]), cfg,
                                     hyp_deadline, rng)

    hdf, hinfo = chunked(run, pdir / "hypothetical", hyp_fn, hyp_idx, 1, hyp_deadline,
                         run.seed + 50)
    out["hypothetical"] = {"state": doc, "result": _jsonable(hdf), "progress": hinfo,
                           "n_rollouts_per_action": int(p["hypo_rollouts"])}
    if len(hdf):
        run.log(f"[plan] hypothetical: {hdf.to_dict(orient='records')}")
    # Sampled real fourth downs (fixed sample; chunks saved as they finish).
    idx_all = np.asarray(rec.fourth_down_index(te))
    n = min(int(p["n_fourth"]), len(idx_all))
    rng = np.random.default_rng(run.seed + 5)
    idx = np.sort(rng.choice(idx_all, size=n, replace=False)) if n else idx_all[:0]

    def fn(part, rng):
        return sim.plan_fourth_downs(model, te, part, int(p["n_rollouts"]), cfg, deadline, rng)

    chunk = max(1, _chunk_size(run, 3 * int(p["n_rollouts"]), "plan"))
    df, info = chunked(run, pdir / "fourth", fn, idx, chunk, deadline, run.seed + 5)
    out["progress"] = info
    out["n_fourth_downs_available"] = int(len(idx_all))
    out["n_rollouts_per_action"] = int(p["n_rollouts"])
    out["decisions"] = _jsonable(df)
    out["agreement"] = _agreement(df)
    run.log(f"[plan] {out['agreement']}")
    if not (info["complete"] and hinfo["complete"]):
        out["status"] = "partial"
        out["reason"] = (f"time budget: {info['reused_states'] + info['computed_states']} of "
                         f"{n} fourth downs; the next run continues from the saved chunks")
    return out


# ---------------------------------------------------------------- stage 6: package
def versions() -> dict:
    v: dict[str, Any] = {"python": platform.python_version(), "platform": platform.platform()}
    for name in ("torch", "numpy", "polars", "pandas", "nflreadpy", "matplotlib"):
        try:
            mod = importlib.import_module(name)
            v[name] = getattr(mod, "__version__", "unknown")
        except Exception:
            v[name] = None
    try:
        import torch

        v["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            v["gpu"] = props.name
            v["gpu_mem_gb"] = round(props.total_memory / 2**30, 1)
            v["cuda"] = torch.version.cuda
    except Exception:
        pass
    return v


def stage_package(run: Run, deadline: float) -> dict:
    results: dict[str, Any] = {
        "experiment": "NFL world model: next-play transformer used as a simulator",
        "preset": run.preset,
        "seed": run.seed,
        "stage_seeds": {"train": run.seed, "eval": run.seed + 3, "test": run.seed + 4,
                        "plan": run.seed + 5},
        "config_hash": run.full_config_hash(),
        "stage_hashes": {s: run.stage_hash(s) for s in STAGES},
        "versions": versions(),
        "workdir": str(run.workdir),
        "budget_s": run.total_s,
        "consumed_s_before_this_session": run.consumed_before,
        "session_seconds": round(time.time() - run.session_start, 1),
        "stages": {},
        "honesty": "Every number here was produced by this run. Fields labelled 'theirs' or "
                   "'nflverse' are nflverse's published estimates on the same plays.",
    }
    for s in STAGES:
        if s == "package":
            continue
        p = run.output_path(s)
        results["stages"][s] = read_json(p) if p.exists() else {"status": "missing"}
    prereg = run.workdir / "prereg.json"
    if prereg.exists():
        results["prereg"] = read_json(prereg)
    figs: dict[str, Any] = {}
    try:
        figs = make_figures(run, results)
    except Exception as e:
        figs = {"error": f"{type(e).__name__}: {e}"}
    results["figures"] = figs
    write_json(run.workdir / "results.json", results)
    zpath = run.workdir / "world_model_results.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(run.workdir / "results.json", "results.json")
        for sub in ("outputs", "figures"):
            for f in sorted((run.workdir / sub).glob("*")):
                z.write(f, f"{sub}/{f.name}")
        for f in sorted((run.workdir / "data").glob("*.json")):
            z.write(f, f"data_reports/{f.name}")
        for f in ("prereg.json", "run.log", "ledger.json", "preset.json"):
            if (run.workdir / f).exists():
                z.write(run.workdir / f, f)
    run.log(f"[package] results at {run.workdir / 'results.json'}; zip at {zpath}")
    return {"results": str(run.workdir / "results.json"), "zip": str(zpath), "figures": figs}


# ---------------------------------------------------------------- figures
_STYLE = {
    "SURFACE": "#fcfcfb", "INK": "#0b0b0b", "INK_2": "#52514e", "MUTED": "#898781",
    "GRID": "#e1e0d9", "AXIS": "#c3c2b7", "BLUE": "#2a78d6", "ORANGE": "#eb6834",
    "AQUA": "#1baf7a", "VIOLET": "#6250d6", "NEUTRAL": "#c3c2b7",
}
_PRED_COLORS = {"ours_simulation": _STYLE["BLUE"], "nflverse_home_wp (theirs)": _STYLE["ORANGE"],
                "nflverse_vegas_home_wp (theirs)": _STYLE["VIOLET"]}
_ACTION_COLORS = {"GO": _STYLE["BLUE"], "FG": _STYLE["ORANGE"], "PUNT": _STYLE["AQUA"]}


def _apply_style(plt) -> None:
    s = _STYLE
    plt.rcParams.update({
        "figure.facecolor": s["SURFACE"], "axes.facecolor": s["SURFACE"],
        "savefig.facecolor": s["SURFACE"], "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"], "font.size": 10.5,
        "text.color": s["INK"], "axes.edgecolor": s["AXIS"], "axes.labelcolor": s["INK_2"],
        "axes.titlecolor": s["INK"], "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "axes.axisbelow": True, "grid.color": s["GRID"],
        "xtick.color": s["MUTED"], "ytick.color": s["MUTED"], "legend.frameon": False,
        "lines.linewidth": 2.0, "savefig.dpi": 160, "savefig.bbox": "tight",
    })


def make_figures(run: Run, results: dict) -> dict:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    _apply_style(plt)
    fdir = run.workdir / "figures"
    made: dict[str, str] = {}
    st = results["stages"]

    hist = (st.get("train", {}).get("history") or {}).get("history") or []
    series = {
        "train (nats per record)": [h.get("train_loss") for h in hist],
        "validation (nats per record)": [h.get("val_total") for h in hist],
    }
    if hist:
        fig, ax = plt.subplots(figsize=(7, 4))
        colors = (_STYLE["NEUTRAL"], _STYLE["BLUE"])
        for (k, v), c in zip(series.items(), colors, strict=True):
            ep = [h.get("epoch", i) + 1 for i, h in enumerate(hist)]
            ax.plot(ep, v, label=k, color=c, marker="o", ms=3)
        base = (st.get("eval", {}).get("nll") or {}).get("baseline_freq_phase_down", {})
        if base.get("total_per_transition"):
            ax.axhline(base["total_per_transition"], color=_STYLE["ORANGE"], lw=1.2, ls="--",
                       label="frequency baseline, validation")
        ax.set_xlabel("epoch")
        ax.set_ylabel("NLL, nats per record (sum over fields)")
        ax.set_title("Training curves")
        ax.legend()
        made["training_curves"] = _save(fig, fdir / "training_curves.png", plt)

    panels = [(name, st[name]["wp"]) for name in ("eval", "test")
              if st.get(name, {}).get("status") in ("done", "partial") and "wp" in st[name]]
    if panels:
        fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5), squeeze=False)
        for ax, (name, wp) in zip(axes[0], panels, strict=True):
            ax.plot([0, 1], [0, 1], color=_STYLE["AXIS"], lw=1, ls="--")
            for k, m in wp["metrics"].items():
                r = m["reliability"]
                xs = [a for a, b in zip(r["bin_mean_pred"], r["bin_mean_outcome"], strict=True)
                      if a is not None and b is not None]
                ys = [b for a, b in zip(r["bin_mean_pred"], r["bin_mean_outcome"], strict=True)
                      if a is not None and b is not None]
                ll = m["log_loss"]["mean"]
                ax.plot(xs, ys, marker="o", color=_PRED_COLORS.get(k, _STYLE["INK_2"]),
                        label=f"{k}, log loss {ll:.3f}")
            label = "validation" if name == "eval" else "test"
            ax.set_title(f"Home win probability calibration ({label}, {wp['n_states']} states)")
            ax.set_xlabel("predicted")
            ax.set_ylabel("observed (regulation result)")
            ax.legend(fontsize=8.5, loc="upper left")
        made["calibration"] = _save(fig, fdir / "calibration.png", plt)

    nll = st.get("eval", {}).get("nll")
    if nll:
        names = [n for n, v in nll["per_field"].items()
                 if v["model"] is not None and v["baseline"] is not None]
        if names:
            x = np.arange(len(names))
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(x - 0.2, [nll["per_field"][n]["baseline"] for n in names], 0.4,
                   color=_STYLE["NEUTRAL"], label="frequency baseline (phase, down)")
            ax.bar(x + 0.2, [nll["per_field"][n]["model"] for n in names], 0.4,
                   color=_STYLE["BLUE"], label="world model")
            ax.set_xticks(x, names, rotation=30, ha="right")
            ax.set_ylabel("NLL (nats), validation")
            ax.set_title("Per-field NLL: world model vs frequency baseline")
            ax.legend()
            made["nll_vs_baseline"] = _save(fig, fdir / "nll_vs_baseline.png", plt)

    plan = st.get("plan", {})
    if plan.get("status") in ("done", "partial") and plan.get("hypothetical"):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        res = plan["hypothetical"]["result"]
        acts = [("GO", "wp_go", "se_go"), ("FG", "wp_fg", "se_fg"),
                ("PUNT", "wp_punt", "se_punt")]
        vals = [(a, res[w][0], (res.get(s) or [0])[0]) for a, w, s in acts
                if w in res and res[w] and res[w][0] is not None]
        ax = axes[0]
        for k, (a, v, se) in enumerate(vals):
            ax.bar(k, v, yerr=1.96 * (se or 0), color=_ACTION_COLORS[a], capsize=4)
            ax.text(k, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(len(vals)), [a for a, _, _ in vals])
        ax.set_ylabel("simulated win probability, offense")
        ax.set_title("Up 4, 4:00 left, 4th and 2 at own 40", fontsize=11)
        ag = plan.get("agreement", {})
        ax = axes[1]
        conf = ag.get("confusion_recommended_by_real", {})
        reals = sorted({r for v in conf.values() for r in v})
        recs = sorted(conf)
        bottom = np.zeros(len(recs))
        for r in reals:
            h = np.array([conf[b].get(r, 0) for b in recs], dtype=float)
            ax.bar(recs, h, bottom=bottom, label=f"coach: {r}",
                   color=_ACTION_COLORS.get(r, _STYLE["NEUTRAL"]))
            bottom += h
        a_all = ag.get("agreement_all")
        ax.set_title(f"Model recommendation vs coach ({ag.get('n', 0)} test 4th downs"
                     + (f", agree {a_all:.0%})" if a_all is not None else ")"), fontsize=11)
        ax.set_xlabel("model recommends")
        ax.set_ylabel("fourth downs")
        ax.legend()
        made["fourth_down"] = _save(fig, fdir / "fourth_down.png", plt)
    return made


def _save(fig, path: Path, plt) -> str:
    fig.savefig(path)
    plt.close(fig)
    return str(path)


def split_arrays_hash(split: S.Split, extra: dict | None = None) -> str:
    """Content hash of a split's arrays (and any extra aligned arrays)."""
    h = hashlib.sha256()
    for k in _SPLIT_ARRAYS:
        a = np.ascontiguousarray(getattr(split, k))
        h.update(f"{k}{a.dtype.str}{a.shape}".encode())
        h.update(a.tobytes())
    h.update("\x1f".join(split.game_ids).encode())
    for k in sorted(extra or {}):
        a = np.ascontiguousarray(extra[k])
        h.update(f"{k}{a.dtype.str}{a.shape}".encode())
        h.update(a.tobytes())
    return h.hexdigest()[:16]


def _build_season(rec, season: int) -> tuple[S.Split, dict, dict]:
    pbp = rec.load_pbp(check_seasons([season]))
    sched = rec.load_schedules(check_seasons([season]))
    sp, report = rec.build_split(pbp, sched, [season])
    extra = align_nflverse_wp(sp, _pbp_wp_columns(pbp))
    del pbp
    return sp, report, extra


def chunked(run: Run, pdir: Path, fn, idx: np.ndarray, chunk: int, deadline: float,
            seed: int) -> tuple[Any, dict]:
    """Run fn(idx_part, rng) -> DataFrame over fixed chunks of idx, saving each finished
    chunk under pdir and reusing saved chunks on the next run. Each chunk has its own seed,
    so a resumed result is the same as an uninterrupted one. Stops before a chunk that
    would not finish by the deadline."""
    import pandas as pd

    pdir.mkdir(parents=True, exist_ok=True)
    idx = np.asarray(idx, dtype=np.int64)
    n_chunks = int(np.ceil(len(idx) / max(chunk, 1)))
    frames, counters = [], []
    reused = computed = 0
    per_state, stopped = None, None
    for k in range(n_chunks):
        part = idx[k * chunk:(k + 1) * chunk]
        f = pdir / f"chunk_{k:04d}.pkl"
        if f.exists():
            frames.append(pd.read_pickle(f))
            cf = f.with_suffix(".json")
            counters.append(read_json(cf) if cf.exists() else None)
            reused += len(part)
            continue
        left = deadline - time.time()
        if left <= 0 or (per_state is not None and left < 1.1 * per_state * len(part)):
            stopped = "time budget"
            break
        t0 = time.time()
        df = fn(part, np.random.default_rng([seed, k]))
        per_state = (time.time() - t0) / max(len(part), 1)
        complete = "done" not in df.columns or bool(df["done"].astype(bool).all())
        frames.append(df)
        counters.append(_jsonable(df.attrs.get("counters")))
        computed += len(part)
        if complete:
            tmp = f.with_suffix(".tmp")
            df.to_pickle(tmp)
            tmp.replace(f)
            write_json(f.with_suffix(".json"), counters[-1])
        else:
            stopped = "deadline reached inside a chunk"
            break
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    info = {"planned_states": int(len(idx)), "chunk_states": int(chunk),
            "chunks": n_chunks, "reused_states": reused, "computed_states": computed,
            "stopped": stopped, "complete": stopped is None,
            "seconds_per_state": None if per_state is None else round(per_state, 4),
            "rollout_counters": counters}
    return df, info


def _chunk_size(run: Run, n_rollouts: int, key: str) -> int:
    c = run.preset[key]
    if c.get("chunk_states"):
        return int(c["chunk_states"])
    live = int(run.preset["train"].get("sim_max_live", 8192))
    return max(4, live // max(int(n_rollouts), 1))


def _cached_json(path: Path, make):
    if path.exists():
        return read_json(path)
    val = _jsonable(make())
    write_json(path, val)
    return val


def _check_checkpoint(path: Path, model) -> str | None:
    """None if there is no checkpoint or it fits the model; else why it does not."""
    if not path.exists():
        return None
    try:
        import torch

        ck = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(ck, dict):
            return "not a checkpoint dict"
        saved = ck.get("current_model") or ck.get("model")
        if not isinstance(saved, dict):
            return None  # not ours to judge (a stub or another format)
        cur = model.state_dict()
        missing = [k for k in cur if k not in saved]
        extra = [k for k in saved if k not in cur]
        shape = [k for k in cur if k in saved and tuple(saved[k].shape) != tuple(cur[k].shape)]
        if missing or extra or shape:
            return (f"{len(missing)} missing, {len(extra)} unexpected, {len(shape)} "
                    "shape-mismatched tensors")
    except Exception as e:
        return f"unreadable: {type(e).__name__}: {e}"
    return None


def checkpoint_info(run: Run) -> dict:
    path = run.ckpt_path()
    info: dict[str, Any] = {"path": str(path), "exists": path.exists()}
    if path.exists():
        try:
            import torch

            ck = torch.load(path, map_location="cpu", weights_only=False)
            if isinstance(ck, dict):
                info["epochs_done"] = ck.get("epoch")
                info["best_val_nll"] = ck.get("best_val")
                info["finished"] = ck.get("finished")
        except Exception as e:
            info["error"] = f"{type(e).__name__}: {e}"
    return info


def status_table(run: Run) -> str:
    """Which stages are done, the checkpoint, its last epoch and best validation NLL, and
    minutes used so far."""
    lines = [f"Work folder {run.workdir}", f"{'stage':<8} {'state':<12} seconds"]
    for s in STAGES[:-1]:
        if run.is_done(s):
            state = "done"
        else:
            state = run.status(s)
            state = "not started" if state == "missing" else state
        secs = read_json(run.output_path(s)).get("seconds", "") \
            if run.output_path(s).exists() else ""
        lines.append(f"{s:<8} {state:<12} {secs}")
    if run.content_hash("train") != "missing":
        ck = checkpoint_info(run)
        if ck["exists"]:
            best = ck.get("best_val_nll")
            best_s = f"{best:.4f}" if isinstance(best, (int, float)) else "n/a"
            lines.append(f"checkpoint {ck['path']}: {ck.get('epochs_done')} epochs done, best "
                         f"validation NLL {best_s} nats/record"
                         + (", training finished" if ck.get("finished") else ""))
        else:
            lines.append(f"checkpoint {ck['path']}: none yet")
    lines.append(f"minutes used so far {run.consumed_before / 60:.1f} of "
                 f"{run.total_s / 60:.0f}; {run.remaining() / 60:.1f} left this session")
    return "\n".join(lines)


def stopped_message(stage: str) -> str:
    return f"Stopped safely at {stage}. Run all again to continue from here."


def _beyond(stage: str, stop_after: str | None) -> bool:
    if stop_after is None or stage == "package":
        return False
    if stop_after not in STAGES:
        raise ValueError(f"STOP_AFTER must be None or one of {STAGES[:-1]}")
    return STAGES.index(stage) > STAGES.index(stop_after)


def notebook_stage(run: Run, stage: str, stop_after: str | None = None) -> dict:
    """One notebook cell's worth of work. Honors STOP_AFTER; on the stop button, saves what
    exists, writes results.json and re-raises so Colab's Run all stops here."""
    if _beyond(stage, stop_after):
        msg = (f"[{stage}] not run: STOP_AFTER = {stop_after!r}. Set STOP_AFTER = None and "
               "Run all to continue.")
        print(msg)
        return {"status": "not_run", "reason": msg}
    try:
        return run_stage(run, stage)
    except KeyboardInterrupt:
        _safe_package(run)
        print(stopped_message(stage), flush=True)
        raise


def _safe_package(run: Run) -> None:
    try:
        run_stage(run, "package")
    except BaseException as e:  # never let packaging hide the original stop
        run.log(f"[package] could not package after the stop: {e}")



STAGE_FUNCS = {
    "data": stage_data,
    "train": stage_train,
    "eval": stage_eval,
    "test": stage_test,
    "plan": stage_plan,
    "package": stage_package,
}


# ---------------------------------------------------------------- preflight
def system_info() -> dict:
    info = versions()
    try:
        import shutil

        du = shutil.disk_usage(default_workdir().parent if default_workdir().parent.exists()
                               else "/")
        info["disk_free_gb"] = round(du.free / 2**30, 1)
    except Exception:
        pass
    try:
        with open("/proc/meminfo") as f:
            kb = int(next(line for line in f if line.startswith("MemTotal")).split()[1])
        info["ram_gb"] = round(kb / 2**20, 1)
    except Exception:
        pass
    return info


def micro_benchmark(preset: dict, seconds: float = 30.0) -> dict:
    """Time training and rollouts for about `seconds` on a synthetic split, and scale the
    preset down if this GPU is slower than the preset assumes. Returns the measurements and
    the (possibly) adjusted preset. The synthetic split is not football; it only fixes
    tensor shapes."""
    import torch

    model_mod, train_mod, sim = _mod("model"), _mod("train"), _mod("simulate")
    device = "cuda" if preset["device"] == "cuda" and torch.cuda.is_available() else "cpu"
    cfg = {**preset["model"], **preset["train"], "device": device, "seed": SEED, "epochs": 1}
    syn = synthetic_split(n_games=24 if device == "cuda" else 6, seed=1)
    model = model_mod.WorldModel(cfg).to(device)
    tmp = Path(os.environ.get("TMPDIR", "/tmp")) / f"wm_bench_{os.getpid()}.pt"
    t0 = time.time()
    hist = train_mod.train_model(model, syn, subset_split(syn, [0, 1]), cfg, tmp,
                                 time.time() + seconds / 2, lambda *a: None)
    train_s = time.time() - t0
    tmp.unlink(missing_ok=True)
    rec_per_s = float(hist.get("train_records_per_s") or n_records(syn) / max(train_s, 1e-6))
    if not np.isfinite(rec_per_s):
        rec_per_s = n_records(syn) / max(train_s, 1e-6)
    model.eval()
    rng = np.random.default_rng(0)
    idx = np.array([int(syn.offsets[g]) + 5 for g in range(4)])
    n_roll = max(int(cfg.get("sim_batch", 1024)) // len(idx), 4)
    t0 = time.time()
    df = sim.wp_by_simulation(model, syn, idx, n_roll, cfg, time.time() + seconds / 2, rng)
    roll_s = time.time() - t0
    steps = np.nan_to_num(df["mean_steps"].to_numpy(dtype=float)) if "mean_steps" in df else 1
    # Rollout-steps per second at this batch size. Untrained rollouts end early, so the live
    # batch shrinks while timing: this undercounts steady-state speed (conservative).
    units = float(np.sum(steps * n_roll)) / max(roll_s, 1e-6)
    a = preset["assumed"]
    tr_ratio = rec_per_s / a["train_rec_per_s"]
    ro_ratio = units / a["rollout_units_per_s"]
    adjusted = copy.deepcopy(preset)
    changes = []
    if preset["device"] == "cuda" and tr_ratio < 0.5 and DOWNGRADE[preset["name"]] != \
            preset["name"]:
        smaller = PRESETS[DOWNGRADE[preset["name"]]]
        adjusted["model"] = copy.deepcopy(smaller["model"])
        adjusted["train"]["batch_games"] = smaller["train"]["batch_games"]

        changes.append(f"model downgraded to the {smaller['name']} size (training at "
                       f"{tr_ratio:.0%} of assumed speed)")
    # Rollout counts from the measured speed: a rollout from a random state runs about
    # STEPS_PER_ROLLOUT transitions to the end of regulation; each stage may spend 60% of
    # its ceiling on rollouts (the pilot and deadline guard handle the rest).
    est = {}
    for key in ("eval", "test"):
        c = adjusted[key]
        cost = (c["wp_states"] * c["wp_rollouts"] * STEPS_PER_ROLLOUT
                + c["sim_games"] * 2 * STEPS_PER_ROLLOUT) / max(units, 1e-6)
        room = 0.6 * 60 * adjusted["stage_min"][key]
        if cost > room:
            f = room / cost
            c["wp_states"] = max(int(c["wp_states"] * f), min(c["wp_states"], 20))
            c["sim_games"] = int(c["sim_games"] * f)
            changes.append(f"{key}: states and simulated games scaled by {f:.2f}")
        est[key] = round(cost / 60, 1)
    c = adjusted["plan"]
    cost = 3 * (c["n_fourth"] * c["n_rollouts"] + c["hypo_rollouts"]) * STEPS_PER_ROLLOUT \
        / max(units, 1e-6)
    room = 0.6 * 60 * adjusted["stage_min"]["plan"]
    if cost > room:
        f = room / cost
        c["n_fourth"] = max(int(c["n_fourth"] * f), min(c["n_fourth"], 10))
        c["hypo_rollouts"] = max(int(c["hypo_rollouts"] * max(f, 0.25)), 16)
        changes.append(f"plan: fourth downs scaled by {f:.2f}")
    est["plan"] = round(cost / 60, 1)
    return {
        "device": device,
        "train_records_per_s": round(rec_per_s, 1),
        "rollout_steps_per_s_untrained": round(units, 1),
        "estimated_rollout_minutes_before_scaling": est,

        "train_speed_vs_assumed": round(tr_ratio, 2),
        "rollout_speed_vs_assumed": round(ro_ratio, 2),
        "note": "untrained-model rollouts end at random, so rollout speed is a rough guide; "
                "each stage also measures a pilot and scales to its own deadline",
        "changes": changes,
        "adjusted_preset": adjusted,
    }


def estimate_minutes(preset: dict) -> dict:
    sm = preset["stage_min"]
    return {"budget_min": preset["budget_min"], "stage_ceilings_min": sm}


def prepare(preset_name: str | None = None, workdir: Path | None = None, benchmark: bool = True,
            budget_min: float | None = None, force_new_preset: bool = False,
            allow_rescore: bool = False, force: tuple = ()) -> Run:
    """Pick (or reload) the preset, benchmark it once, print the status table, return a Run.

    The adjusted preset is saved in the work folder on the first call and reused on every
    resume, so a resumed run keeps the same config hashes and skips finished stages.
    """
    wd = Path(workdir or default_workdir())
    wd.mkdir(parents=True, exist_ok=True)
    saved = wd / "preset.json"
    if saved.exists() and not force_new_preset:
        preset = read_json(saved)
        print(f"Reusing the saved preset '{preset['name']}' from {saved}")
    else:
        name = preset_name or pick_preset()
        preset = get_preset(name)
        if benchmark:
            try:
                bench = micro_benchmark(preset)
                preset = bench.pop("adjusted_preset")
                preset["benchmark"] = bench
                print("Micro-benchmark:", json.dumps(bench, indent=1))
            except Exception as e:
                preset["benchmark"] = {"error": f"{type(e).__name__}: {e}"}
                print("Micro-benchmark failed, keeping the preset as is:", e)
        write_json(saved, preset)
    if budget_min is not None:
        preset["budget_min"] = budget_min
    run = Run(preset, wd, budget_min=budget_min, allow_rescore=allow_rescore, force=force)
    print(status_table(run), flush=True)
    return run


def run_all(run: Run, stages=STAGES, stop_after: str | None = None) -> dict:
    """Run the stages in order (up to stop_after), then package. A KeyboardInterrupt or an
    exception in a stage still ends with results.json; an interrupt prints where it
    stopped."""
    out: dict[str, Any] = {}
    current = None
    try:
        for s in stages:
            if s == "package" or _beyond(s, stop_after):
                continue
            current = s
            out[s] = run_stage(run, s)
    except KeyboardInterrupt:
        _safe_package(run)
        out["stopped_at"] = current
        msg = stopped_message(current)
        run.log(msg)
        if run.quiet:
            print(msg, flush=True)
        return out
    out["package"] = run_stage(run, "package")
    return out


def summary(run: Run) -> str:
    """Plain-language summary of results.json."""
    path = run.workdir / "results.json"
    if not path.exists():
        return "No results.json yet."
    r = read_json(path)
    st = r["stages"]
    lines = [f"Preset {r['preset']['name']}, config {r['config_hash']}, "
             f"GPU {r['versions'].get('gpu', 'none')}."]
    for s in ("data", "train", "eval", "test", "plan"):
        o = st.get(s, {})
        lines.append(f"- {s}: {o.get('status')}"
                     + (f" ({o.get('reason') or o.get('error')})"
                        if o.get("status") != "done" else f" in {o.get('seconds')}s"))
    d = st.get("data", {})
    if d.get("status") == "done":
        for k, v in d["splits"].items():
            lines.append(f"  {k}: {v['games']} games, {v['records']} records, "
                         f"seasons {v['seasons'][0]}-{v['seasons'][-1]}")
    t = st.get("train", {})
    if t.get("status") == "done":
        lines.append(f"  model: {t['params']:,} parameters, trained {t['train_seconds_total']}s")
    for s, label in (("eval", "validation"), ("test", "test")):
        o = st.get(s, {})
        if o.get("status") not in ("done", "partial") or not o.get("wp", {}).get("metrics"):
            continue
        b = o["nll"]["baseline_freq_phase_down"]["total_per_transition"]
        lines.append(f"  {label} NLL per transition: model {_total_nll(o['nll']['model'])}"
                     f" vs frequency baseline {b:.3f} (lower is better)")
        for k, m in o["wp"]["metrics"].items():
            ll = m["log_loss"]
            lines.append(f"  {label} win probability, {k}: log loss {ll['mean']:.4f} "
                         f"(95% game bootstrap {ll['lo95']:.4f} to {ll['hi95']:.4f}), "
                         f"Brier {m['brier']['mean']:.4f}, {ll['n_states']} states")
    p = st.get("plan", {})
    if p.get("status") in ("done", "partial") and p.get("agreement", {}).get("n"):
        ag = p["agreement"]
        lines.append(f"  fourth downs: model agrees with the coach on "
                     f"{ag.get('agreement_all', 0):.0%} of {ag.get('n')} test decisions"
                     + (f", {ag['agreement_clear']:.0%} of the {ag['n_clear']} that are not "
                        "too close to call" if ag.get("agreement_clear") is not None else ""))
        res = p["hypothetical"]["result"]
        bits = [f"{a} {res[k][0]:.3f}" for a, k in (("go", "wp_go"), ("field goal", "wp_fg"),
                                                     ("punt", "wp_punt"))
                if res.get(k) and res[k][0] is not None]
        lines.append(f"  hypothetical (up 4, 4:00 left, 4th and 2 at own 40): "
                     f"{', '.join(bits)}; best {res.get('best', ['?'])[0]}")
    return "\n".join(lines)


def main(argv: list[str] | None = None) -> None:
    import argparse

    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--preset", choices=sorted(PRESETS), default=None)
    ap.add_argument("--workdir", type=Path, default=None)
    ap.add_argument("--budget-min", type=float, default=None)
    ap.add_argument("--no-benchmark", action="store_true")
    ap.add_argument("--allow-rescore", action="store_true")
    ap.add_argument("--force-new-preset", action="store_true")
    ap.add_argument("--stop-after", choices=[s for s in STAGES if s != "package"],
                    default=None)
    ap.add_argument("--force", nargs="*", default=[], choices=list(STAGES[:-1]),
                    help="stages to recompute even if done")
    a = ap.parse_args(argv)
    run = prepare(a.preset, a.workdir, benchmark=not a.no_benchmark, budget_min=a.budget_min,
                  force_new_preset=a.force_new_preset, allow_rescore=a.allow_rescore,
                  force=tuple(a.force))
    out = run_all(run, stop_after=a.stop_after)
    print(summary(run))
    if out.get("stopped_at"):
        raise SystemExit(130)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/wm_code/test_model.py
"""CPU tests for the world model, training loop and simulator, on synthetic toy games."""

from __future__ import annotations

import time

import numpy as np
import pytest
import schema as S
import simulate
import torch
import train
from model import END_FIELDS, WorldModel
from simulate import check_transitions, plan_fourth_downs, rollout, simulate_games
from toy import make_toy_split
from train import _full_counts, _micro_slices, collate, evaluate_nll, train_model

torch.set_num_threads(1)

TINY = {
    "d_model": 32,
    "n_layers": 2,
    "n_heads": 2,
    "d_ff": 64,
    "dec_hidden": 32,
    "dropout": 0.0,
    "max_len": 48,
    "batch_games": 8,
    "eval_batch_games": 16,
    "lr": 3e-3,
    "warmup_steps": 5,
    "epochs": 3,
    "max_steps": 150,
    "sim_batch": 512,
    "seed": 0,
}
FAR = time.time() + 3600.0


def tiny_model(seed: int = 0, **over) -> WorldModel:
    torch.manual_seed(seed)
    return WorldModel({**TINY, **over})


@pytest.fixture(scope="module")
def split() -> S.Split:
    return make_toy_split(24, np.random.default_rng(0))


@pytest.fixture(scope="module")
def trained(split) -> WorldModel:
    m = tiny_model()
    train_model(m, split, split, {**TINY, "epochs": 4}, "", FAR, log=lambda *_: None)
    return m.eval()


def fourth_downs(split: S.Split, poss: str) -> np.ndarray:
    st = split.states
    return np.flatnonzero(
        (st[:, S.STATE_INDEX["down"]] == 4)
        & (st[:, S.STATE_INDEX["phase"]] == S.PHASE["PLAY"])
        & (st[:, S.STATE_INDEX["poss"]] == S.POSS_CODE[poss])
    )


def test_toy_split_follows_the_contract(split):
    for g in range(len(split.offsets) - 1):
        a, b = split.offsets[g], split.offsets[g + 1]
        raw_next = S.apply_transition(split.raw[a : b - 1], split.targets[a : b - 1])
        assert np.array_equal(raw_next, split.raw[a + 1 : b])
        codes = S.next_state_codes(raw_next, split.targets[a : b - 1])
        assert np.array_equal(codes, split.states[a + 1 : b])
        assert split.targets[b - 1, S.TARGET_INDEX["phase"]] == S.PHASE["END"]
    assert check_transitions(split.targets)["inconsistent"] == 0


def test_loss_decreases_when_overfitting_four_games(split):
    m = tiny_model()
    batch = collate(split, np.arange(4), m.max_len)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    first = None
    for _ in range(150):
        total, _ = m.loss(batch)
        first = float(total.detach()) if first is None else first
        opt.zero_grad()
        total.backward()
        opt.step()
    assert float(total.detach()) < 0.5 * first


def test_causal_mask(split):
    m = tiny_model().eval()
    batch = collate(split, np.array([0]), m.max_len)
    t = 10
    other = {k: v.clone() for k, v in batch.items()}
    other["states"][0, t + 1 :] = torch.randint(1, 3, other["states"][0, t + 1 :].shape)
    with torch.no_grad():
        h1 = m.encode(batch["header"], batch["states"])
        h2 = m.encode(other["header"], other["states"])
        l1 = m.decode_logits(h1, batch["targets"])
        l2 = m.decode_logits(h2, batch["targets"])
    assert torch.allclose(h1[:, : t + 1], h2[:, : t + 1], atol=1e-6)
    assert not torch.allclose(h1[:, t + 1 :], h2[:, t + 1 :])
    for a, b in zip(l1, l2, strict=True):
        assert torch.allclose(a[:, : t + 1], b[:, : t + 1], atol=1e-6)


def test_generation_stays_in_vocabulary():
    m = tiny_model().eval()
    gen = torch.Generator().manual_seed(0)
    h = 20.0 * torch.randn(4096, TINY["d_model"], generator=gen)
    for temp in (1.0, 3.0, 0.0):
        codes = m.sample_transition(h, temp, gen).numpy()
        end = codes[:, S.TARGET_INDEX["phase"]] == S.PHASE["END"]
        for j, (name, size) in enumerate(S.TARGET_FIELDS):
            assert codes[:, j].min() >= 0 and codes[:, j].max() < size, name
            if name not in ("down", "ydstogo"):
                assert codes[~end, j].min() >= 1, name  # padding code is never sampled
            if name in END_FIELDS:
                assert (codes[end, j] >= 1).all(), name
            else:
                assert (codes[end, j] == 0).all(), name  # END rows: same prefix as training


def test_forced_actions_are_respected(split, trained):
    idx = np.concatenate([fourth_downs(split, "HOME")[:2], fourth_downs(split, "AWAY")[:2]])
    rng = np.random.default_rng(3)
    fg = rollout(trained, split, idx, S.ACTION["FG"], 8, TINY, FAR, rng)
    assert fg["start_actions"] == {"FG": 8 * len(idx)}
    go = rollout(trained, split, idx, "GO", 64, TINY, FAR, rng)
    assert set(go["start_actions"]) <= {"RUN", "PASS"}
    assert sum(go["start_actions"].values()) == 64 * len(idx)
    real = rollout(trained, split, idx, None, 4, TINY, FAR, rng)
    expect = {}
    for a in split.states[idx, S.STATE_INDEX["action"]]:
        expect[S.ACTIONS[a]] = expect.get(S.ACTIONS[a], 0) + 4
    assert real["start_actions"] == expect


def test_rollouts_end_and_are_counted(split, trained):
    idx = np.arange(0, len(split.states), 37)
    r = rollout(trained, split, idx, None, 8, TINY, FAR, np.random.default_rng(0))
    c = r["counters"]
    assert r["done"].all()
    assert np.all((r["home_wp"] >= 0) & (r["home_wp"] <= 1))
    assert c["ended_by_end"] + c["ended_by_clock"] + c["truncated_max_steps"] == c["rollouts"]
    short = rollout(
        trained, split, idx[:3], None, 8, {**TINY, "max_steps": 1}, FAR, np.random.default_rng(0)
    )
    assert short["counters"]["truncated_max_steps"] > 0
    assert np.all(short["mean_steps"] <= 1)
    late = rollout(
        trained, split, idx[:3], None, 8, TINY, time.time() - 1, np.random.default_rng(0)
    )
    assert not late["done"].any() and np.isnan(late["home_wp"]).all()


def test_plan_fourth_downs_perspective(split, trained):
    home, away = fourth_downs(split, "HOME")[:1], fourth_downs(split, "AWAY")[:1]
    idx = np.concatenate([home, away])
    df = plan_fourth_downs(trained, split, idx, 32, TINY, FAR, np.random.default_rng(11))
    assert list(df["poss"]) == ["HOME", "AWAY"]
    assert set(df["best"]) <= {"GO", "FG", "PUNT"}
    rng = np.random.default_rng(11)  # same draws in the same order as plan_fourth_downs
    for key, act in (("go", "GO"), ("fg", S.ACTION["FG"]), ("punt", S.ACTION["PUNT"])):
        r = rollout(trained, split, idx, act, 32, TINY, FAR, rng)
        assert df[f"wp_{key}"].iloc[0] == pytest.approx(r["home_wp"][0])
        assert df[f"wp_{key}"].iloc[1] == pytest.approx(1.0 - r["home_wp"][1])
        assert df[f"se_{key}"].tolist() == pytest.approx(r["home_wp_se"].tolist())
    wps = df[["wp_go", "wp_fg", "wp_punt"]].to_numpy()
    assert np.allclose(df["margin"], np.sort(wps, 1)[:, -1] - np.sort(wps, 1)[:, -2])
    assert (df["too_close"] == (df["margin"] < 2 * df["margin_se"])).all()


def test_checkpoint_resume_is_identical(split, tmp_path):
    cfg = {**TINY, "dropout": 0.1, "epochs": 2}
    a = tiny_model()
    ra = train_model(a, split, split, cfg, str(tmp_path / "a.pt"), FAR, log=lambda *_: None)
    b = tiny_model()
    path = str(tmp_path / "b.pt")
    rb1 = train_model(
        b, split, split, {**cfg, "stop_after_epoch": 1}, path, FAR, log=lambda *_: None
    )
    assert rb1["epochs_done"] == 1 and rb1["stopped"] == "stop_after_epoch"
    c = tiny_model(seed=123)  # different init: everything must come from the checkpoint
    rc = train_model(c, split, split, cfg, path, FAR, log=lambda *_: None)
    assert rc["epochs_done"] == 2
    for (k, va), vc in zip(a.state_dict().items(), c.state_dict().values(), strict=True):
        assert torch.equal(va, vc), k
    assert [h["val_total"] for h in ra["history"]] == [h["val_total"] for h in rc["history"]]
    ck = torch.load(str(tmp_path / "a.pt"), weights_only=False)
    assert {"model", "optimizer", "epoch", "cfg", "history"} <= set(ck)


def test_deterministic_with_fixed_seed(split, trained):
    idx = fourth_downs(split, "HOME")[:2]
    r1 = rollout(trained, split, idx, "GO", 16, TINY, FAR, np.random.default_rng(5))
    r2 = rollout(trained, split, idx, "GO", 16, TINY, FAR, np.random.default_rng(5))
    assert np.array_equal(r1["home_wp"], r2["home_wp"]) and r1["counters"] == r2["counters"]
    g1 = simulate_games(trained, split, 6, TINY, FAR, np.random.default_rng(1))
    g2 = simulate_games(trained, split, 6, TINY, FAR, np.random.default_rng(1))
    assert g1["sim"] == g2["sim"] and g1["n_games"] == 6
    m1, m2 = tiny_model(), tiny_model()
    cfg = {**TINY, "epochs": 1, "dropout": 0.1}
    train_model(m1, split, split, cfg, "", FAR, log=lambda *_: None)
    train_model(m2, split, split, cfg, "", FAR, log=lambda *_: None)
    for v1, v2 in zip(m1.state_dict().values(), m2.state_dict().values(), strict=True):
        assert torch.equal(v1, v2)
    assert evaluate_nll(m1, split)["total"] == evaluate_nll(m2, split)["total"]


# ------------------------------------------------------------------ review additions
def test_decoder_uses_only_earlier_true_fields(split):
    m = tiny_model().eval()
    batch = collate(split, np.array([0, 1]), m.max_len)
    h = m.encode(batch["header"], batch["states"])
    base = m.decode_logits(h, batch["targets"])
    for j in range(len(S.TARGET_FIELDS)):
        tg = batch["targets"].clone()
        tg[..., j:] = torch.randint(1, 3, tg[..., j:].shape)  # field j and later changed
        other = m.decode_logits(h, tg)
        for k in range(j + 1):
            assert torch.equal(base[k], other[k]), (j, k)
        if j + 1 < len(S.TARGET_FIELDS):
            assert not torch.allclose(base[j + 1], other[j + 1])  # field j is used by j+1


def test_padding_and_other_games_do_not_leak(split):
    m = tiny_model().eval()
    one = collate(split, np.array([0]), m.max_len)
    two = collate(split, np.array([0, 5]), m.max_len)
    k = int(one["mask"].sum())
    with torch.no_grad():
        _, i1 = m.loss(one)
        h1 = m.encode(one["header"], one["states"])
        h2 = m.encode(two["header"], two["states"])
        padded = {kk: v.clone() for kk, v in two.items()}
        padded["states"][0, k:] = torch.randint(1, 3, padded["states"][0, k:].shape)
        h3 = m.encode(padded["header"], padded["states"])
    assert torch.allclose(h1[0, :k], h2[0, :k], atol=1e-5)
    assert torch.allclose(h1[0, :k], h3[0, :k], atol=1e-5)


def test_max_len_keeps_last_records(split):
    lens = np.diff(split.offsets)
    g = int(np.argmax(lens))
    ml = int(lens[g]) - 5
    b = collate(split, np.array([g]), ml)
    e = split.offsets[g + 1]
    assert torch.equal(b["states"][0], torch.as_tensor(split.states[e - ml : e].astype(np.int64)))
    assert bool(b["last"][0, -1])
    buf, ln = simulate._window_buffer(split, np.array([e - 1]), ml)
    assert ln[0] == ml and np.array_equal(buf[0], split.states[e - ml : e])


def test_last_record_loss_counts_only_end_fields(split):
    m = tiny_model().eval()
    batch = collate(split, np.array([0]), m.max_len)
    k = int(batch["mask"].sum())
    last_tg = batch["targets"][0, k - 1]
    assert last_tg[S.TARGET_INDEX["score_delta"]] > 0 and last_tg[S.TARGET_INDEX["elapsed"]] > 0
    masks = m.target_masks(batch["targets"], batch["mask"], batch["last"])
    for j, (name, _) in enumerate(S.TARGET_FIELDS):
        assert bool(masks[j][0, k - 1]) == (name in END_FIELDS), name
    # a non-END field on the last record never reaches the decoder input
    poisoned = {kk: v.clone() for kk, v in batch.items()}
    poisoned["targets"][0, k - 1, S.TARGET_INDEX["poss"]] = 2
    with torch.no_grad():
        a, _ = m.loss(batch)
        b, _ = m.loss(poisoned)
    assert torch.equal(a, b)


def test_score_on_end_is_applied(split, trained, monkeypatch):
    idx = fourth_downs(split, "HOME")[:2]
    end = np.zeros(len(S.TARGET_FIELDS), dtype=np.int64)
    end[S.TARGET_INDEX["phase"]] = S.PHASE["END"]
    end[S.TARGET_INDEX["score_delta"]] = S.SCORE_DELTA_CODE[3]
    end[S.TARGET_INDEX["elapsed"]] = 6

    def fake(h, *a, **k):
        return torch.as_tensor(np.tile(end, (h.shape[0], 1)))

    monkeypatch.setattr(trained, "sample_transition", fake)
    base = split.raw[idx, 1]
    for flag, add in ((True, 3), (False, 0)):
        cfg = {**TINY, "score_on_end": flag}
        r = rollout(trained, split, idx, None, 4, cfg, FAR, np.random.default_rng(0))
        assert np.allclose(r["home_wp"], S.outcome(base + add)), flag
        assert r["counters"]["ended_by_end"] == 8 and (r["mean_steps"] == 1).all()


def test_sorted_encoding_gives_identical_rollouts(split, trained):
    idx = np.arange(3, len(split.states), 23)
    out = {}
    for flag in (False, True):
        cfg = {**TINY, "sort_by_length": flag, "sim_batch": 16}
        out[flag] = rollout(trained, split, idx, None, 6, cfg, FAR, np.random.default_rng(4))
    assert np.array_equal(out[False]["home_wp"], out[True]["home_wp"])
    assert np.array_equal(out[False]["mean_steps"], out[True]["mean_steps"])
    assert out[False]["counters"] == out[True]["counters"]


def test_plan_is_chunked_by_state(split, trained):
    idx = np.concatenate([fourth_downs(split, "HOME")[:3], fourth_downs(split, "AWAY")[:3]])
    cfg = {**TINY, "sim_max_live": 16}  # 2 states per chunk at 8 rollouts
    df = plan_fourth_downs(trained, split, idx, 8, cfg, FAR, np.random.default_rng(2))
    assert df["done"].all() and len(df) == len(idx)
    rng = np.random.default_rng(2)
    for a in range(0, len(idx), 2):
        for key, act in (("go", "GO"), ("fg", S.ACTION["FG"]), ("punt", S.ACTION["PUNT"])):
            r = rollout(trained, split, idx[a : a + 2], act, 8, cfg, FAR, rng)
            home = df["poss"].iloc[a : a + 2].to_numpy() == "HOME"
            want = np.where(home, r["home_wp"], 1 - r["home_wp"])
            assert np.allclose(df[f"wp_{key}"].iloc[a : a + 2], want)
    assert sum(df.attrs["counters"]["GO"][k] for k in ("rollouts",)) == 8 * len(idx)


def test_micro_batches_give_the_same_gradient(split):
    m = tiny_model()
    batch = collate(split, np.arange(6), m.max_len)
    full, _ = m.loss(batch)
    full.backward()
    g_full = [p.grad.clone() for p in m.parameters()]
    m.zero_grad()
    denom = _full_counts(m, batch)
    tot = 0.0
    for part in _micro_slices(batch, 4):
        t, _ = m.loss(part, denom=denom)
        t.backward()
        tot += float(t.detach())
    assert tot == pytest.approx(float(full.detach()), rel=1e-5)
    for a, b in zip(g_full, (p.grad for p in m.parameters()), strict=True):
        assert torch.allclose(a, b, atol=1e-6, rtol=1e-4)


def test_out_of_memory_retries_the_same_step(split, monkeypatch):
    cfg = {**TINY, "epochs": 2}
    ref = tiny_model()
    train_model(ref, split, split, {**cfg}, "", FAR, log=lambda *_: None)
    real_loss = WorldModel.loss
    calls = {"oom": 0}

    def flaky(self, batch, denom=None):
        if self.training and batch["mask"].shape[0] > 2:  # evaluation is not affected
            calls["oom"] += 1
            raise torch.OutOfMemoryError("CUDA out of memory. Tried to allocate 2.00 GiB")
        return real_loss(self, batch, denom)

    monkeypatch.setattr(WorldModel, "loss", flaky)
    m = tiny_model()
    res = train_model(m, split, split, cfg, "", FAR, log=lambda *_: None)
    assert calls["oom"] >= 1 and res["micro_batches"] == 4 and res["epochs_done"] == 2
    assert res["history"][0]["batch_games"] == TINY["batch_games"]  # same batches
    for (k, a), b in zip(ref.state_dict().items(), m.state_dict().values(), strict=True):
        assert torch.allclose(a, b, atol=1e-4), k


def test_deadline_mid_epoch_resumes_exactly(split, tmp_path, monkeypatch):
    cfg = {**TINY, "dropout": 0.1, "epochs": 2}
    a = tiny_model()
    ra = train_model(a, split, split, cfg, "", FAR, log=lambda *_: None)
    path = str(tmp_path / "d.pt")
    real_loss = WorldModel.loss
    seen = {"n": 0}

    def counting(self, batch, denom=None):
        seen["n"] += int(self.training)
        return real_loss(self, batch, denom)

    real_time = time.time
    monkeypatch.setattr(WorldModel, "loss", counting)
    monkeypatch.setattr(train.time, "time", lambda: real_time() + (1e9 if seen["n"] >= 2 else 0))
    b = tiny_model()
    rb = train_model(b, split, split, cfg, path, FAR, log=lambda *_: None)  # stops in epoch 0
    assert rb["stopped"] == "deadline" and rb["epochs_done"] == 0
    ck = torch.load(path, weights_only=False)
    assert ck["batch_in_epoch"] == 2 and ck["step"] == 2
    monkeypatch.undo()
    c = tiny_model(seed=7)
    rc = train_model(c, split, split, cfg, path, FAR, log=lambda *_: None)
    assert rc["epochs_done"] == 2
    for (k, va), vc in zip(a.state_dict().items(), c.state_dict().values(), strict=True):
        assert torch.equal(va, vc), k
    assert [h["val_total"] for h in ra["history"]] == [h["val_total"] for h in rc["history"]]


In [ ]:
%%writefile /content/wm_code/test_records.py
"""Tests for records.py: a hand-made game checked field by field, guards, validation."""

from __future__ import annotations

import copy
import os

import numpy as np
import polars as pl
import pytest
import records as R
import schema as S

H, A = "HOM", "AWY"


def _row(order, play_type, posteam, clock, tot=(0, 0), to=(3, 3), half="Half1", qtr=2,
         down=None, ytg=None, yl=None, desc="", nfl="", rush=0, pas=0, two=0):  # fmt: skip
    return {
        "game_id": "2023_01_AWY_HOM", "play_id": float(order * 10), "season": 2023,
        "season_type": "REG", "home_team": H, "away_team": A, "posteam": posteam,
        "order_sequence": float(order), "desc": desc, "qtr": float(qtr), "game_half": half,
        "down": down, "ydstogo": ytg, "yardline_100": yl,
        "game_seconds_remaining": float(clock), "total_home_score": float(tot[0]),
        "total_away_score": float(tot[1]), "home_timeouts_remaining": float(to[0]),
        "away_timeouts_remaining": float(to[1]), "play_type": play_type,
        "play_type_nfl": nfl, "rush": float(rush), "pass": float(pas),
        "two_point_attempt": float(two), "home_score": 15.0, "away_score": 11.0,
    }  # fmt: skip


def synthetic_game() -> pl.DataFrame:
    rows = [
        _row(0, None, None, 1900, qtr=1, desc="GAME", nfl="GAME_START"),
        _row(1, "kickoff", H, 1900, yl=35, desc="X kicks 65 yards from AWY 35"),
        _row(2, "run", H, 1896, tot=(6, 0), down=1, ytg=10, yl=75, rush=1),
        _row(3, "extra_point", H, 1890, tot=(7, 0), yl=15),
        _row(4, "kickoff", A, 1890, tot=(7, 0), yl=35, desc="Y kicks 65 yards"),
        _row(
            5, "no_play", A, 1885, tot=(7, 0), to=(3, 2), desc="Timeout #1 by AWY", nfl="TIMEOUT"
        ),  # fmt: skip
        _row(6, "pass", A, 1885, tot=(7, 0), to=(3, 2), down=1, ytg=10, yl=70, pas=1),
        _row(7, "punt", A, 1850, tot=(7, 0), to=(3, 2), down=4, ytg=3, yl=63),
        _row(
            8,
            "no_play",
            H,
            1842,
            tot=(7, 0),
            to=(3, 2),
            down=1,
            ytg=10,
            yl=80,
            pas=1,
            desc="PENALTY on AWY, Holding - No Play.",
            nfl="PENALTY",
        ),  # fmt: skip
        _row(9, "pass", H, 1838, tot=(13, 0), to=(2, 2), down=1, ytg=15, yl=85, pas=1),
        _row(10, "pass", H, 1838, tot=(15, 0), to=(2, 2), yl=2, pas=1, two=1),
        _row(11, "kickoff", A, 1838, tot=(15, 0), to=(2, 2), yl=35, desc="kicks 65"),
        _row(12, "qb_kneel", A, 1832, tot=(15, 0), to=(2, 2), down=1, ytg=10, yl=75),
        _row(13, None, None, 1800, tot=(15, 0), to=(2, 2), desc="END QUARTER 2"),
        _row(14, "kickoff", H, 1800, tot=(15, 0), half="Half2", qtr=3, yl=35, desc="kicks"),
        _row(
            15, "run", H, 1795, tot=(15, 2), half="Half2", qtr=3, down=1, ytg=10, yl=99, rush=1
        ),  # fmt: skip
        _row(16, "kickoff", A, 1790, tot=(15, 2), half="Half2", qtr=3, yl=20, desc="kicks"),
        _row(
            17, "field_goal", A, 1760, tot=(15, 5), half="Half2", qtr=4, down=4, ytg=5, yl=20
        ),  # fmt: skip
        _row(
            18,
            "kickoff",
            H,
            1760,
            tot=(15, 5),
            half="Half2",
            qtr=4,
            yl=35,
            desc="Z kicks onside 12 yards",
        ),  # fmt: skip
        _row(
            19, "run", A, 600, tot=(15, 11), half="Overtime", qtr=5, down=1, ytg=10, yl=75, rush=1
        ),  # fmt: skip
    ]
    return pl.DataFrame(rows, infer_schema_length=None)


SCHED = pl.DataFrame({"game_id": ["2023_01_AWY_HOM"], "spread_line": [-3.0]})

# phase, poss, down, ytg, yl, clock, diff, to_h, to_a, action
EXPECTED = [
    ("KICKOFF", "HOME", 0, 0, 35, 1900, 0, 3, 3, "KICKOFF"),
    ("PLAY", "HOME", 1, 10, 75, 1896, 0, 3, 3, "RUN"),
    ("TRY", "HOME", 0, 0, 15, 1890, 6, 3, 3, "XP"),
    ("KICKOFF", "AWAY", 0, 0, 35, 1890, 7, 3, 3, "KICKOFF"),
    ("PLAY", "AWAY", 1, 10, 70, 1885, 7, 3, 2, "PASS"),
    ("PLAY", "AWAY", 4, 3, 63, 1850, 7, 3, 2, "PUNT"),
    ("PLAY", "HOME", 1, 10, 80, 1842, 7, 3, 2, "PASS"),
    ("PLAY", "HOME", 1, 15, 85, 1838, 7, 3, 2, "PASS"),
    ("TRY", "HOME", 0, 0, 2, 1838, 13, 2, 2, "TWO_PT"),
    ("KICKOFF", "AWAY", 0, 0, 35, 1838, 15, 2, 2, "KICKOFF"),
    ("PLAY", "AWAY", 1, 10, 75, 1832, 15, 2, 2, "KNEEL"),
    ("KICKOFF", "HOME", 0, 0, 35, 1800, 15, 3, 3, "KICKOFF"),
    ("PLAY", "HOME", 1, 10, 99, 1795, 15, 3, 3, "RUN"),
    ("KICKOFF", "AWAY", 0, 0, 20, 1790, 13, 3, 3, "KICKOFF"),
    ("PLAY", "AWAY", 4, 5, 20, 1760, 13, 3, 3, "FG"),
    ("KICKOFF", "HOME", 0, 0, 35, 1760, 10, 3, 3, "ONSIDE"),
]
# score_delta, elapsed, to_used_home, to_used_away (target of record i)
EXPECTED_T = [
    (0, 4, 0, 0), (6, 6, 0, 0), (1, 0, 0, 0), (0, 5, 0, 1), (0, 35, 0, 0), (0, 8, 0, 0),
    (0, 4, 0, 0), (6, 0, 1, 0), (2, 0, 0, 0), (0, 6, 0, 0), (0, 32, 0, 0), (0, 5, 0, 0),
    (-2, 5, 0, 0), (0, 30, 0, 0), (-3, 0, 0, 0),
]  # fmt: skip


@pytest.fixture(scope="module")
def built():
    return R.build_split(synthetic_game(), SCHED, [2023])


def test_states_by_hand(built):
    split, report = built
    assert len(split.game_ids) == 1 and report["rows_kept"] == len(EXPECTED)
    for i, e in enumerate(EXPECTED):
        d = R.decode_record(split, i)
        got = (d["phase"], d["poss"], d["down"], d["ydstogo"], d["yardline_100"],
               d["clock_seconds"], d["score_diff_home"], d["timeouts_home"],
               d["timeouts_away"], d["action"])  # fmt: skip
        assert got == e, (i, got, e)


def test_targets_by_hand(built):
    split, _ = built
    for i, (dlt, el, uh, ua) in enumerate(EXPECTED_T):
        t = R.decode_record(split, i)["target"]
        nxt = EXPECTED[i + 1]
        assert (t["phase"], t["poss"], t["down"], t["ydstogo"], t["yardline"], t["action"]) == (
            nxt[0],
            nxt[1],
            nxt[2],
            nxt[3],
            nxt[4],
            nxt[9],
        ), i
        assert (t["score_delta"], t["elapsed"], t["to_used_home"], t["to_used_away"]) == (
            dlt,
            el,
            uh,
            ua,
        ), i
    last = split.targets[-1]
    assert last[0] == S.PHASE["END"]
    # the final record (onside kick at 1760 s, diff 10) scores nothing; elapsed clips at 60
    assert last[S.TARGET_INDEX["score_delta"]] == S.SCORE_DELTA_CODE[0]
    assert last[S.TARGET_INDEX["elapsed"]] == S.elapsed_code(1760)
    assert not last[list(R.END_ZERO_FIELDS)].any()


def test_header_final_and_report(built):
    split, report = built
    assert split.header[0, 0] == S.spread_code(-3.0) == 19
    assert split.header[0, 1] == S.POSS_CODE["HOME"]
    assert split.final_diff[0] == 10  # regulation 15-5; the OT score 15-11 is cut
    assert report["dropped"]["timeout_only"] == 1
    assert report["dropped"]["overtime_or_null_qtr"] == 1
    assert report["soft"]["halftime_crossings"] == 1
    assert report["overtime_games_cut"] == 1
    np.testing.assert_array_equal(R.fourth_down_index(split), [5, 14])


def test_validate_passes_and_bookkeeping(built):
    split, _ = built
    v = R.validate_split(split, min_transition_match=1.0)
    assert v["info_mismatched_pairs"] == 0


def test_first_half_snap_at_1800_repaired():
    df = synthetic_game().with_columns(
        pl.when(pl.col("order_sequence") == 12).then(1800.0)
        .otherwise(pl.col("game_seconds_remaining")).alias("game_seconds_remaining")
    )  # fmt: skip
    split, report = R.build_split(df, SCHED, [2023])
    assert report["repairs"]["first_half_clock_1800_set_to_1801"] == 1
    assert split.raw[10, 0] == 1801
    assert R.validate_split(split, 1.0)["info_mismatched_pairs"] == 0


def test_points_on_final_record_are_in_end_target():
    # walk-off: HOME kicks a FG with 3 s left in regulation, then overtime rows follow
    walk = _row(
        18.5, "field_goal", H, 3, tot=(18, 5), half="Half2", qtr=4, down=4, ytg=2, yl=10
    )  # fmt: skip
    df = pl.concat([synthetic_game(), pl.DataFrame([walk], schema=synthetic_game().schema)])
    split, report = R.build_split(df, SCHED, [2023])
    assert split.final_diff[0] == 13 and report["soft"]["games_scoring_on_last_record"] == 1
    d = R.decode_record(split, len(split.states) - 1)
    assert (d["action"], d["score_diff_home"], d["clock_seconds"]) == ("FG", 10, 3)
    assert d["target"] == {"phase": "END", "score_delta": 3, "elapsed": 3}
    v = R.validate_split(split, 0.9)  # the 1760 s -> 3 s jump is one clock mismatch
    assert v["final_diff_ne_last_score_plus_end_delta"] == 0
    assert v["info_games_scoring_on_last_record"] == 1


def test_validate_catches_final_diff_mismatch(built):
    split = copy.deepcopy(built[0])
    split.final_diff[0] += 3  # END target no longer explains the final score
    with pytest.raises(AssertionError, match="final_diff_ne_last_score_plus_end_delta"):
        R.validate_split(split, 1.0)


def test_noplay_punt_and_fg_keep_their_action():
    wiped = {
        7: ("4-J.Doe punts 45 yards to HOM 20. PENALTY on HOM, Holding, enforced - No Play.", 0),
        17: ("1-K.Kik 38 yard field goal is GOOD, NULLIFIED by Penalty - No Play.", 0),
    }  # fmt: skip
    df = synthetic_game().with_columns(
        pl.when(pl.col("order_sequence").is_in([float(k) for k in wiped]))
        .then(pl.lit("no_play")).otherwise(pl.col("play_type")).alias("play_type"),
        pl.col("order_sequence").replace_strict(
            {float(k): v[0] for k, v in wiped.items()},
            default=pl.col("desc"),
            return_dtype=pl.String,
        ).alias("desc"),
    )  # fmt: skip
    split, _ = R.build_split(df, SCHED, [2023])
    acts = [R.decode_record(split, i)["action"] for i in (5, 14)]
    assert acts == ["PUNT", "FG"]
    assert R.decode_record(split, 6)["action"] == "PASS"  # penalty row with pass flag


@pytest.mark.parametrize("bad", [[2026], [2009], [2024, 2026], []])
def test_season_guard(bad):
    with pytest.raises(ValueError):
        R.load_pbp(bad)
    with pytest.raises(ValueError):
        R.load_schedules(bad)
    with pytest.raises(ValueError):
        R.build_split(synthetic_game(), SCHED, bad)


@pytest.mark.parametrize(
    "corrupt",
    [
        lambda s: s.states.__setitem__((3, S.STATE_INDEX["down"]), 2),  # down on a kickoff
        lambda s: s.states.__setitem__((1, S.STATE_INDEX["poss"]), 0),  # pad code
        lambda s: s.targets.__setitem__((-1, 0), S.PHASE["PLAY"]),  # no END
        lambda s: s.targets.__setitem__((4, S.TARGET_INDEX["action"]), 99),  # out of vocab
        lambda s: s.offsets.__setitem__(-1, 3),  # offsets do not cover all records
        lambda s: s.raw.__setitem__((5, 1), 40),  # score bookkeeping broken
    ],
)
def test_validate_catches_corruption(built, corrupt):
    split = copy.deepcopy(built[0])
    corrupt(split)
    with pytest.raises(AssertionError):
        R.validate_split(split, min_transition_match=0.99)


def test_disjoint():
    s, _ = R.build_split(synthetic_game(), SCHED, [2023])
    with pytest.raises(AssertionError):
        R.validate_disjoint({"train": s, "val": s})


@pytest.mark.skipif(os.environ.get("WM_REAL_DATA") != "1", reason="set WM_REAL_DATA=1")
def test_real_2023():
    split, report = R.build_split(R.load_pbp([2023]), R.load_schedules([2023]), [2023])
    v = R.validate_split(split)
    assert report["games"] > 250 and v["info_apply_transition_match_fraction"] >= 0.99


In [ ]:
%%writefile /content/wm_code/test_run.py
"""Tests for run.py with stub records/model/train/simulate modules (fast, CPU, no network)."""

from __future__ import annotations

import sys
import time
import types

import numpy as np
import pandas as pd
import pytest
import run as R
import schema as S


# ---------------------------------------------------------------- stubs
class _StubModel:
    def __init__(self, cfg=None):
        self.cfg = dict(cfg or {})

    def count_params(self):
        return 1234

    def eval(self):
        return self

    def to(self, device):
        return self


def _stubs(calls: dict, fail_train_once: bool = False, build_seed: int = 0,
           interrupt_wp_call: int | None = None) -> dict[str, types.ModuleType]:
    rec = types.ModuleType("records")

    def load_pbp(seasons):
        assert max(seasons) <= 2025
        calls.setdefault("load_pbp", []).append(list(seasons))
        return pd.DataFrame({"game_id": ["x"], "play_id": [1], "home_wp": [0.5],
                             "vegas_home_wp": [0.5]})

    def load_schedules(seasons):
        return pd.DataFrame({"game_id": ["x"]})

    def build_split(pbp, schedules, seasons):
        calls.setdefault("build_split", []).append(list(seasons))
        sp = R.synthetic_split(n_games=4, seed=int(seasons[0]) + build_seed)
        sp.game_ids = [f"{seasons[0]}_{g}" for g in sp.game_ids]
        return sp, {"games": 4}

    rec.load_pbp, rec.load_schedules, rec.build_split = load_pbp, load_schedules, build_split
    rec.validate_split = lambda split: {"ok": True}
    rec.fourth_down_index = lambda split: np.flatnonzero(
        (split.states[:, S.STATE_INDEX["phase"]] == S.PHASE["PLAY"])
        & (split.states[:, S.STATE_INDEX["down"]] == 4))

    model = types.ModuleType("model")
    model.WorldModel = _StubModel

    train = types.ModuleType("train")

    def train_model(m, tr, va, cfg, ckpt_path, deadline_ts, log):
        calls["train"] = calls.get("train", 0) + 1
        import torch

        resumed = R.Path(ckpt_path).exists()
        torch.save({"epoch": 1, "best_val": 1.1}, ckpt_path)
        if fail_train_once and calls["train"] == 1:
            raise RuntimeError("disconnect")
        log("epoch 0: train 1.0 val 1.1")
        return {"resumed": resumed, "history": [{"epoch": 0, "train_loss": 1.0,
                                                 "val_total": 1.1}],
                "train_records_per_s": 1e6}

    train.train_model = train_model
    train.load_checkpoint = lambda path, cfg_override=None, device=None: (_StubModel(), {})
    train.evaluate_nll = lambda m, split, cfg=None: {
        "total": 9.0, "fields": {n: 0.9 for n, _ in S.TARGET_FIELDS}}

    sim = types.ModuleType("simulate")

    def wp_by_simulation(m, split, idx, n, cfg, deadline_ts, rng):
        calls["wp_calls"] = calls.get("wp_calls", 0) + 1
        if interrupt_wp_call is not None and calls["wp_calls"] == interrupt_wp_call:
            raise KeyboardInterrupt
        calls["wp"] = calls.get("wp", 0) + len(idx)

        g = np.searchsorted(split.offsets, idx, side="right") - 1
        return pd.DataFrame({"idx": idx, "home_wp_sim": rng.uniform(0.1, 0.9, len(idx)),
                             "home_result": S.outcome(split.final_diff[g]),
                             "done": np.ones(len(idx), dtype=bool),
                             "mean_steps": np.full(len(idx), 10.0)})

    def plan_fourth_downs(m, split, idx, n, cfg, deadline_ts, rng):
        calls["plan"] = calls.get("plan", 0) + len(idx)
        k = len(idx)
        return pd.DataFrame({"idx": idx, "wp_go": np.full(k, 0.6), "wp_fg": np.full(k, 0.5),
                             "wp_punt": np.full(k, 0.55), "se_go": np.full(k, 0.01),
                             "se_fg": np.full(k, 0.01), "se_punt": np.full(k, 0.01),
                             "best": ["GO"] * k, "margin": np.full(k, 0.05),
                             "too_close": np.zeros(k, dtype=bool),
                             "real_action": ["PUNT"] * k, "real_choice": ["PUNT"] * k,
                             "done": np.ones(k, dtype=bool)})

    sim.wp_by_simulation, sim.plan_fourth_downs = wp_by_simulation, plan_fourth_downs
    sim.simulate_games = lambda m, split, n, cfg, d, rng: {"n_games": n, "sim": {"x": 1.0},
                                                             "real": {"x": 1.1}}
    return {"records": rec, "model": model, "train": train, "simulate": sim}


@pytest.fixture
def stubbed(monkeypatch):
    calls: dict = {}
    for name, mod in _stubs(calls).items():
        monkeypatch.setitem(sys.modules, name, mod)
    return calls


def _tiny_preset(**kw):
    p = R.get_preset("smoke")
    p["seasons"] = {"train": [2019], "val": [2020], "test": [2021]}
    p.update(kw)
    return p


# ---------------------------------------------------------------- tests
def test_pick_preset():
    assert R.pick_preset("Tesla T4", 15.0) == "t4"
    assert R.pick_preset("NVIDIA L4", 22.5) == "l4"
    assert R.pick_preset("NVIDIA A100-SXM4-40GB", 40.0) == "a100"
    assert R.pick_preset("NVIDIA H100 80GB HBM3", 80.0) == "a100"
    assert R.pick_preset("Some New GPU", 48.0) == "a100"
    assert R.pick_preset("Some Old GPU", 8.0) == "t4"


def test_pick_preset_cpu(monkeypatch):
    import torch

    monkeypatch.setattr(torch.cuda, "is_available", lambda: False)
    assert R.pick_preset() == "smoke"


def test_presets_use_full_seasons_and_fit_budget():
    for name in ("t4", "l4", "a100"):
        p = R.PRESETS[name]
        assert p["seasons"]["train"] == list(range(2010, 2022))
        assert p["seasons"]["val"] == [2022, 2023]
        assert p["seasons"]["test"] == [2024, 2025]
        assert p["budget_min"] <= 55
        assert p["reserve_package_s"] > 0
    for p in R.PRESETS.values():
        assert set(p["stage_min"]) == set(R.STAGES)
        for s in p["seasons"].values():
            assert max(s) <= R.MAX_SEASON


def test_2026_guard(tmp_path, stubbed):
    with pytest.raises(AssertionError):
        R.check_seasons([2025, 2026])
    p = _tiny_preset()
    p["seasons"]["test"] = [2026]
    run = R.Run(p, tmp_path, quiet=True)
    out = R.run_stage(run, "data")
    assert out["status"] == "error" and "2025" in out["error"]
    assert all(max(s) <= 2025 for s in stubbed.get("load_pbp", []))


def test_split_roundtrip(tmp_path):
    sp = R.synthetic_split(3, seed=2)
    R.save_split(tmp_path / "a.npz", sp, w=np.arange(R.n_records(sp), dtype=float))
    back, extra = R.load_split(tmp_path / "a.npz")
    assert back.game_ids == sp.game_ids
    assert np.array_equal(back.states, sp.states)
    assert len(extra["w"]) == R.n_records(sp)
    both = R.concat_splits([sp, back])
    assert both.offsets[-1] == 2 * R.n_records(sp)
    sub = R.subset_split(both, [4])
    assert np.array_equal(sub.states, sp.states[sp.offsets[1]:sp.offsets[2]])


def test_frequency_baseline_beats_uniform():
    sp = R.synthetic_split(6, seed=3)
    tables = R.fit_frequency_baseline(sp)
    nll = R.baseline_nll(tables, sp)
    uniform = sum(np.log(size) for _, size in S.TARGET_FIELDS)
    assert np.isfinite(nll["total_per_transition"])
    assert nll["total_per_transition"] < uniform


def test_hypothetical_edits_codes():
    sp = R.synthetic_split(4, seed=4)
    hyp, doc = R.build_hypothetical(sp)
    j = R.n_records(hyp) - 1
    st = hyp.states[j]
    assert st[S.STATE_INDEX["down"]] == 4 and st[S.STATE_INDEX["ydstogo"]] == 2
    assert st[S.STATE_INDEX["yardline"]] == 60
    assert hyp.raw[j, 0] == 240 and st[S.STATE_INDEX["clock"]] == S.clock_code(240)
    sign = 1 if st[S.STATE_INDEX["poss"]] == S.POSS_CODE["HOME"] else -1
    assert hyp.raw[j, 1] == 4 * sign
    assert st[S.STATE_INDEX["score"]] == S.score_code(4 * sign)
    assert doc["base_play_id"] in sp.play_ids


def test_bootstrap_interval_contains_mean():
    rng = np.random.default_rng(0)
    v = rng.normal(size=200)
    b = R.game_bootstrap(v, np.repeat(np.arange(20), 10), 300, rng)
    assert b["lo95"] <= b["mean"] <= b["hi95"]


def test_full_run_resume_and_test_once(tmp_path, stubbed):
    run = R.Run(_tiny_preset(), tmp_path, quiet=True)
    out = R.run_all(run)
    for s in ("data", "train", "eval", "test", "plan", "package"):
        assert out[s]["status"] == "done", (s, out[s])
    res = R.read_json(tmp_path / "results.json")
    assert res["stages"]["test"]["status"] == "done"
    assert res["prereg"]["config_hash"] == run.full_config_hash()
    assert (tmp_path / "world_model_results.zip").exists()
    assert "Preset smoke" in R.summary(run)
    # Re-running skips every finished stage.
    before = dict(stubbed)
    run2 = R.Run(_tiny_preset(), tmp_path, quiet=True)
    R.run_all(run2)
    assert stubbed["train"] == before["train"] and stubbed["wp"] == before["wp"]
    assert stubbed["build_split"] == before["build_split"]
    # Changing the test plan after scoring is refused without the flag ...
    p = _tiny_preset()
    p["test"]["wp_states"] = 10
    out3 = R.run_stage(R.Run(p, tmp_path, quiet=True), "test")
    assert out3["status"] == "error" and "refusing" in out3["error"]
    # ... and allowed, and recorded, with it.
    out4 = R.run_stage(R.Run(p, tmp_path, quiet=True, allow_rescore=True), "test")
    assert out4["status"] == "done"
    prereg = R.read_json(tmp_path / "prereg.json")
    assert prereg["attempts"][-1]["rescore"] is True


def test_prereg_written_before_test_data_is_read(tmp_path, stubbed, monkeypatch):
    run = R.Run(_tiny_preset(), tmp_path, quiet=True)
    for s in ("data", "train"):
        R.run_stage(run, s)
    seen = []
    orig = R.Run.split

    def spy(self, name):
        if name == "test":
            seen.append((tmp_path / "prereg.json").exists())
        return orig(self, name)

    monkeypatch.setattr(R.Run, "split", spy)
    R.run_stage(run, "test")
    assert seen and all(seen)


def test_training_resumes_after_disconnect(tmp_path, monkeypatch):
    calls: dict = {}
    for name, mod in _stubs(calls, fail_train_once=True).items():
        monkeypatch.setitem(sys.modules, name, mod)
    run = R.Run(_tiny_preset(), tmp_path, quiet=True)
    R.run_stage(run, "data")
    assert R.run_stage(run, "train")["status"] == "error"
    assert not run.is_done("train")
    out = R.run_stage(R.Run(_tiny_preset(), tmp_path, quiet=True), "train")
    assert out["status"] == "done" and out["history"]["resumed"] is True


def test_deadline_guard_still_writes_results(tmp_path, stubbed):
    p = _tiny_preset(budget_min=0.01, reserve_package_s=1)
    run = R.Run(p, tmp_path, quiet=True)
    time.sleep(0.05)
    out = R.run_all(run)
    assert out["data"]["status"] == "skipped" and "time budget" in out["data"]["reason"]
    assert out["train"]["status"] == "skipped"
    res = R.read_json(tmp_path / "results.json")
    assert res["stages"]["eval"]["status"] == "skipped"
    assert out["package"]["status"] == "done"


def test_ledger_shrinks_budget_on_resume(tmp_path):
    p = _tiny_preset()
    R.write_json(tmp_path / "ledger.json", {"consumed_s": 600.0})
    run = R.Run(p, tmp_path, quiet=True)
    assert run.remaining() < 60 * p["budget_min"] - 590


def test_resume_survives_source_change(tmp_path, monkeypatch):
    calls: dict = {}
    for name, mod in _stubs(calls).items():
        monkeypatch.setitem(sys.modules, name, mod)
    run = R.Run(_tiny_preset(), tmp_path, quiet=True)
    R.run_stage(run, "data")
    R.run_stage(run, "train")
    ckpt = run.ckpt_path()
    # Edited model/train/records code: the train stage stays done, same checkpoint.
    orig = R._source_hash
    monkeypatch.setattr(R, "_source_hash", lambda mods: "edited" + orig(mods))
    run2 = R.Run(_tiny_preset(), tmp_path, quiet=True)
    assert run2.is_done("train") and run2.ckpt_path() == ckpt
    assert not run2.is_done("data")  # records changed: rebuild in memory and compare
    out = R.run_stage(run2, "data")
    assert set(out["seasons_rebuilt"].values()) == {"identical, kept"}
    assert run2.is_done("train") and run2.ckpt_path() == ckpt
    # Code that really changes the arrays replaces them, and training starts again.
    for name, mod in _stubs(calls, build_seed=99).items():
        monkeypatch.setitem(sys.modules, name, mod)
    monkeypatch.setattr(R, "_source_hash", lambda mods: "edited again" + orig(mods))
    run3 = R.Run(_tiny_preset(), tmp_path, quiet=True)
    out = R.run_stage(run3, "data")
    assert "replaced" in out["seasons_rebuilt"].values()
    assert not run3.is_done("train") and run3.ckpt_path() != ckpt


def test_stop_after_train(tmp_path, stubbed):
    run = R.Run(_tiny_preset(), tmp_path, quiet=True)
    out = R.run_all(run, stop_after="train")
    assert "eval" not in out and out["package"]["status"] == "done"
    res = R.read_json(tmp_path / "results.json")
    assert res["stages"]["train"]["status"] == "done"
    assert res["stages"]["eval"]["status"] == "missing"
    assert "train    done" in R.status_table(run)
    with pytest.raises(ValueError):
        R._beyond("eval", "nonsense")


def test_interrupted_eval_resumes_from_saved_chunks(tmp_path, monkeypatch, capsys):
    calls: dict = {}
    for name, mod in _stubs(calls, interrupt_wp_call=2).items():
        monkeypatch.setitem(sys.modules, name, mod)
    p = _tiny_preset()
    p["eval"].update(wp_states=6, chunk_states=2, sim_games=0)
    run = R.Run(p, tmp_path, quiet=True)
    out = R.run_all(run, stop_after="eval")
    assert out["stopped_at"] == "eval"
    assert "Stopped safely at eval" in capsys.readouterr().out
    res = R.read_json(tmp_path / "results.json")
    assert res["stages"]["eval"]["status"] == "interrupted"
    assert calls["wp"] == 2  # one chunk finished and was saved
    calls2: dict = {}
    for name, mod in _stubs(calls2).items():
        monkeypatch.setitem(sys.modules, name, mod)
    out = R.run_stage(R.Run(p, tmp_path, quiet=True), "eval")
    assert out["status"] == "done"
    assert calls2["wp"] == 4  # only the two missing chunks were simulated
    assert out["wp"]["progress"]["reused_states"] == 2 and out["wp"]["n_states"] == 6


def test_keyboard_interrupt_mid_training_resumes_real_modules(tmp_path, monkeypatch, capsys):
    """Real model.py and train.py: stop after epoch 0, then resume at epoch 1."""
    pytest.importorskip("torch")
    import model as real_model
    import train as real_train

    calls: dict = {}
    stubs = _stubs(calls)
    monkeypatch.setitem(sys.modules, "records", stubs["records"])
    monkeypatch.setitem(sys.modules, "model", real_model)
    monkeypatch.setitem(sys.modules, "train", real_train)
    p = _tiny_preset()
    p["model"] = {"d_model": 16, "n_layers": 1, "n_heads": 2, "d_ff": 32, "dec_hidden": 16,
                  "max_len": 256, "dropout": 0.0}
    p["train"].update(epochs=2, patience=5, batch_games=4, warmup_steps=2)
    orig_log = R.Run.log

    def log_then_stop(self, *msg):
        orig_log(self, *msg)
        if any("epoch 0:" in str(m) for m in msg):
            raise KeyboardInterrupt

    monkeypatch.setattr(R.Run, "log", log_then_stop)
    run = R.Run(p, tmp_path, quiet=True)
    out = R.run_all(run, stop_after="train")
    assert out["stopped_at"] == "train"
    assert "Stopped safely at train" in capsys.readouterr().out
    assert R.read_json(tmp_path / "results.json")["stages"]["train"]["status"] == "interrupted"
    assert run.ckpt_path().exists() and not run.is_done("train")
    assert R.checkpoint_info(run)["epochs_done"] == 1
    monkeypatch.setattr(R.Run, "log", orig_log)
    run2 = R.Run(p, tmp_path, quiet=True)
    out = R.run_stage(run2, "train")
    assert out["status"] == "done"
    hist = out["history"]
    assert hist["epochs_done"] == 2
    assert [h["epoch"] for h in hist["history"]] == [0, 1]
    assert R.checkpoint_info(run2)["epochs_done"] == 2


def test_mismatched_checkpoint_starts_fresh(tmp_path):
    torch = pytest.importorskip("torch")
    import model as real_model

    m = real_model.WorldModel({"d_model": 16, "n_layers": 1, "n_heads": 2, "d_ff": 32,
                               "dec_hidden": 16})
    path = tmp_path / "ck.pt"
    torch.save({"current_model": {"not_a_param": torch.zeros(3)}}, path)
    assert "missing" in R._check_checkpoint(path, m)
    torch.save({"current_model": m.state_dict()}, path)
    assert R._check_checkpoint(path, m) is None


Run the unit tests (CPU, under a minute). The notebook stops here if any fail.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-x', '-p', 'no:cacheprovider',
                    '/content/wm_code'], cwd='/content/wm_code', capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
if r.returncode != 0:
    raise SystemExit('Unit tests failed: stop here and send the output above.')
print('Unit tests passed.')

Preflight: hardware, chosen preset and a short benchmark.

In [ ]:
import json, os, sys, time
sys.path.insert(0, '/content/wm_code')
os.chdir('/content/wm_code')
import run as R

info = R.system_info()
print(json.dumps(info, indent=1))
name = R.pick_preset()
print('Chosen preset:', name)
print('Time budget (minutes):', R.PRESETS[name]['budget_min'],
      ' stage ceilings:', R.PRESETS[name]['stage_min'])
if name == 'smoke':
    print('No GPU found: this runs the small CPU smoke preset (2021/2022/2023 seasons only).')
    print('For the real run: Runtime > Change runtime type > GPU, then Run all again.')
# About 30 seconds: measures training and rollout speed and scales the preset if needed.
# On a resume the saved preset is reused, so finished stages keep their hashes.
RUN = R.prepare(name, force=tuple(FORCE))
print('Work folder:', RUN.workdir)
print('Preset in use:', json.dumps({k: RUN.preset[k] for k in
      ('name', 'model', 'train', 'eval', 'test', 'plan')}, indent=1))
print(f'Minutes left in the budget: {RUN.remaining() / 60:.1f}')

**Stage 1: download play-by-play season by season, build the splits, validate them.**

In [ ]:
out = R.notebook_stage(RUN, 'data', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

**Stage 2: train the world model (checkpointed every epoch; resumes on re-run).**

In [ ]:
out = R.notebook_stage(RUN, 'train', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

**Stage 3: validation seasons only: NLL vs baseline, league stats, WP calibration.**

In [ ]:
out = R.notebook_stage(RUN, 'eval', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

**Stage 4: write prereg.json, then score the test seasons once.**

In [ ]:
out = R.notebook_stage(RUN, 'test', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

**Stage 5: fourth-down recommendations on test-season fourth downs.**

In [ ]:
out = R.notebook_stage(RUN, 'plan', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

**Stage 6: results.json, figures and the zip.**

In [ ]:
out = R.notebook_stage(RUN, 'package', STOP_AFTER)
print('status:', out['status'], out.get('reason') or out.get('error') or '')

Results in plain language.

In [ ]:
print(R.summary(RUN))
from IPython.display import Image, display
for f in sorted((RUN.workdir / 'figures').glob('*.png')):
    print(f.name)
    display(Image(filename=str(f)))

Download the results zip.

In [ ]:
# Download the results. Unzip into the repo under results/world_model/ on the branch
# claude/world-model-colab, then commit. The zip holds results.json, figures, stage outputs,
# the data reports, prereg.json and the run log (no data or checkpoints).
zip_path = RUN.workdir / 'world_model_results.zip'
print(zip_path, f'{zip_path.stat().st_size / 1e6:.1f} MB')
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Download it from the file browser instead:', e)